# Course Recommendation dengan Encoder–Decoder LSTM + Attention Layer

Notebook ini membangun sistem rekomendasi course karyawan berbasis riwayat course yang pernah diambil. Pipeline disusun dengan skenario evaluasi temporal, train-only transformation, attention layer, evaluasi ranking Top-N, analisis distribusi, sanity check berbasis popularitas, serta penyimpanan artifact eksperimen.

Fokus utama notebook ini adalah menghasilkan rekomendasi course berikutnya secara leakage-safe dan mudah direplikasi untuk kebutuhan analisis akademik maupun pengembangan sistem rekomendasi.

## 1. Import Library

**Penjelasan:** Bagian ini memanggil library utama yang dipakai dalam notebook, yaitu `pandas` dan `numpy` untuk pengolahan data, `MinMaxScaler` untuk normalisasi fitur numerik, serta TensorFlow/Keras untuk membangun model Encoder–Decoder LSTM. Pengaturan seed dan filter warning juga dibuat di awal agar eksperimen lebih stabil dan output notebook tetap bersih ketika dijalankan ulang.

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, RepeatVector, Concatenate, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
tf.get_logger().setLevel("ERROR")

np.random.seed(42)
tf.random.set_seed(42)


## 2. Load Dataset Fleksibel untuk Colab

**Penjelasan:** Bagian ini membaca empat file input utama: `interactions.csv`, `courses.csv`, `users.csv`, dan `course_alias_map.csv`. Fungsi `load_csv_flexible()` dibuat agar notebook tetap bisa membaca nama file standar maupun nama file hasil upload Colab seperti `interactions(1).csv` atau `courses(1).csv`. Dengan demikian, proses load dataset cukup dilakukan satu kali dan tidak perlu diduplikasi pada bagian lain.

In [ ]:
from pathlib import Path


DATA_DIR = Path(os.getenv("COURSE_RECOMMENDER_DATA_DIR", "data")).expanduser().resolve()
ALLOW_FLEXIBLE_CSV_MATCH = True


def load_csv_flexible(base_filename, data_dir=DATA_DIR):
    """Read a required CSV from a configured data directory.

    The flexible match supports exported files such as interactions(8).csv,
    but it never prompts for uploads. This makes the training notebook safe to
    run in automated jobs.
    """
    data_dir = Path(data_dir)
    base_path = data_dir / base_filename

    if base_path.exists():
        return pd.read_csv(base_path)

    if ALLOW_FLEXIBLE_CSV_MATCH:
        stem = Path(base_filename).stem
        candidates = sorted(
            [p for p in data_dir.glob(f"{stem}*.csv") if p.is_file()],
            key=lambda p: p.stat().st_mtime,
            reverse=True
        )

        if candidates:
            print(f"Using file for {base_filename}: {candidates[0].name}")
            return pd.read_csv(candidates[0])

    raise FileNotFoundError(
        f"Required file {base_filename} was not found in {data_dir}. "
        "Set COURSE_RECOMMENDER_DATA_DIR to the directory containing "
        "interactions.csv, courses.csv, users.csv, and course_alias_map.csv."
    )


interactions = load_csv_flexible("interactions.csv")
courses = load_csv_flexible("courses.csv")
users = load_csv_flexible("users.csv")
alias_map = load_csv_flexible("course_alias_map.csv")

print(f"Dataset loaded from: {DATA_DIR}")


## 3. Standarisasi Kolom dan Validasi Kolom Wajib

**Penjelasan:** Bagian ini membersihkan nama kolom, ID, dan teks agar format data menjadi konsisten. Kode juga memastikan kolom wajib tersedia pada setiap dataset, seperti `user_id`, `course_id`, `interaction_type`, `canonical_course_id`, dan mapping alias course.

**Yang perlu diperhatikan:** Tujuannya adalah mencegah error di tahap merge dan memastikan data yang masuk ke model sudah memiliki struktur minimal yang benar.

In [ ]:
def clean_column_name(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip().str.lower()
    return df


def clean_id(series):
    s = series.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.replace({"nan": np.nan, "none": np.nan, "None": np.nan, "": np.nan})
    return s


def clean_text(series):
    s = series.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.replace({"nan": np.nan, "none": np.nan, "None": np.nan, "": np.nan})
    return s


interactions = clean_column_name(interactions)
courses = clean_column_name(courses)
users = clean_column_name(users)
alias_map = clean_column_name(alias_map)


required_interactions = ["user_id", "course_id", "interaction_type"]
required_courses = ["canonical_course_id", "course_name"]
required_users = ["user_id"]
required_alias = ["legacy_course_id", "canonical_course_id"]

for col in required_interactions:
    if col not in interactions.columns:
        raise ValueError(f"Kolom {col} tidak ada di interactions.csv")

for col in required_courses:
    if col not in courses.columns:
        raise ValueError(f"Kolom {col} tidak ada di courses.csv")

for col in required_users:
    if col not in users.columns:
        raise ValueError(f"Kolom {col} tidak ada di users.csv")

for col in required_alias:
    if col not in alias_map.columns:
        raise ValueError(f"Kolom {col} tidak ada di course_alias_map.csv")


if "interaction_id" not in interactions.columns:
    interactions["interaction_id"] = np.arange(1, len(interactions) + 1)

interactions["interaction_id"] = clean_id(interactions["interaction_id"])
interactions["user_id"] = clean_id(interactions["user_id"])
interactions["course_id"] = clean_id(interactions["course_id"])
interactions["interaction_type"] = clean_text(interactions["interaction_type"]).str.lower()

if "source_system" in interactions.columns:
    interactions["source_system"] = clean_text(interactions["source_system"])
else:
    interactions["source_system"] = "unknown"


users["user_id"] = clean_id(users["user_id"])

if "full_name" in users.columns:
    users["full_name"] = clean_text(users["full_name"])
else:
    users["full_name"] = "Unknown"


courses["canonical_course_id"] = clean_id(courses["canonical_course_id"])
courses["course_name"] = clean_text(courses["course_name"])

if "description" in courses.columns:
    courses["description"] = clean_text(courses["description"])
else:
    courses["description"] = "Not Available"

if "level" in courses.columns:
    courses["level"] = clean_text(courses["level"])
else:
    courses["level"] = "Not Available"

if "kompetensi" in courses.columns:
    courses["kompetensi"] = clean_text(courses["kompetensi"])
elif "competency" in courses.columns:
    courses["kompetensi"] = clean_text(courses["competency"])
else:
    courses["kompetensi"] = "Not Available"


alias_map["legacy_course_id"] = clean_id(alias_map["legacy_course_id"])
alias_map["canonical_course_id"] = clean_id(alias_map["canonical_course_id"])


interactions = interactions.dropna(subset=["user_id", "course_id"])
courses = courses.dropna(subset=["canonical_course_id", "course_name"])
users = users.dropna(subset=["user_id"])
alias_map = alias_map.dropna(subset=["legacy_course_id", "canonical_course_id"])

interactions = interactions.drop_duplicates()
courses = courses.drop_duplicates(subset=["canonical_course_id"])
users = users.drop_duplicates(subset=["user_id"])
alias_map = alias_map.drop_duplicates(subset=["legacy_course_id"])

print("Interactions:", interactions.shape)
print("Courses:", courses.shape)
print("Users:", users.shape)
print("Alias map:", alias_map.shape)

## 4. Mapping Alias Course dan Timestamp Interaksi

**Penjelasan:** Bagian ini menambahkan urutan baris, mengubah timestamp ke format datetime, lalu memetakan `course_id` lama ke `canonical_course_id` menggunakan `course_alias_map.csv`. Jika suatu course tidak ada di alias map, kode tetap memakai `course_id` awal sebagai canonical ID.

**Yang perlu diperhatikan:** Output yang dicek adalah jumlah timestamp valid/tidak valid, jumlah course yang match dengan alias map, dan contoh data hasil mapping.

In [ ]:
interactions["_row_order"] = np.arange(len(interactions))

if "event_timestamp" in interactions.columns:
    interactions["event_timestamp_raw"] = interactions["event_timestamp"]
    interactions["event_timestamp"] = pd.to_datetime(interactions["event_timestamp"], errors="coerce")
else:
    interactions["event_timestamp_raw"] = np.nan
    interactions["event_timestamp"] = pd.NaT


alias_map_use = alias_map.rename(
    columns={"canonical_course_id": "mapped_canonical_course_id"}
)

interactions_mapped = interactions.merge(
    alias_map_use,
    left_on="course_id",
    right_on="legacy_course_id",
    how="left"
)

interactions_mapped["canonical_course_id"] = interactions_mapped["mapped_canonical_course_id"].fillna(
    interactions_mapped["course_id"]
)

print("Timestamp valid:", interactions_mapped["event_timestamp"].notna().sum())
print("Timestamp tidak valid:", interactions_mapped["event_timestamp"].isna().sum())

print("\nTotal interactions:", len(interactions_mapped))
print("Match alias map:", interactions_mapped["legacy_course_id"].notna().sum())
print("Tidak match alias map:", interactions_mapped["legacy_course_id"].isna().sum())
print("Persentase match alias map:", round(interactions_mapped["legacy_course_id"].notna().mean() * 100, 2), "%")

display(interactions_mapped.head())

## 5. Pengecekan Keterhubungan Data

**Penjelasan:** Bagian ini mengecek apakah data interaksi benar-benar terhubung dengan data user dan data course. Pengecekan dilakukan dengan merge ke tabel `users` dan `courses`, lalu menghitung jumlah data yang match dan tidak match.

**Yang perlu diperhatikan:** Tahap ini penting untuk membuktikan kualitas relasi antar dataset sebelum data dipakai untuk training model rekomendasi.

In [ ]:
check_user = interactions_mapped.merge(
    users[["user_id"]].assign(user_connected=1),
    on="user_id",
    how="left"
)

check_course = interactions_mapped.merge(
    courses[["canonical_course_id"]].assign(course_connected=1),
    on="canonical_course_id",
    how="left"
)

print("=== INTERACTIONS KE USERS ===")
print("Total:", len(check_user))
print("Match users:", check_user["user_connected"].notna().sum())
print("Tidak match users:", check_user["user_connected"].isna().sum())
print("Persentase match users:", round(check_user["user_connected"].notna().mean() * 100, 2), "%")

print("\n=== INTERACTIONS KE COURSES ===")
print("Total:", len(check_course))
print("Match courses:", check_course["course_connected"].notna().sum())
print("Tidak match courses:", check_course["course_connected"].isna().sum())
print("Persentase match courses:", round(check_course["course_connected"].notna().mean() * 100, 2), "%")

print("\nContoh course tidak match:")
display(
    check_course[check_course["course_connected"].isna()][
        ["course_id", "canonical_course_id", "interaction_type"]
    ].drop_duplicates().head(10)
)

## 6. Identifikasi Interaction Type

**Penjelasan:** Bagian ini melihat distribusi `interaction_type`, lalu membuat dua indikator: `is_certificate` dan `is_enrolled`. Indikator tersebut dibuat menggunakan daftar keyword seperti `certificate`, `sertifikat`, `enroll`, dan `lulus`.

**Yang perlu diperhatikan:** Hasilnya dipakai untuk membedakan interaksi yang menunjukkan enrollment dan interaksi yang menunjukkan sertifikat.

In [ ]:
print("=== DISTRIBUSI INTERACTION TYPE ===")
display(interactions_mapped["interaction_type"].value_counts(dropna=False))

certificate_keywords = [
    "cert", "certificate", "certified", "sertifikat",
    "bersertifikat", "passed", "pass", "lulus"
]

enroll_keywords = [
    "enroll", "enrolled", "enrollment", "ambil", "mengambil"
]


def contains_keyword(text, keywords):
    text = str(text).lower()
    return any(k in text for k in keywords)


interactions_mapped["is_certificate"] = interactions_mapped["interaction_type"].apply(
    lambda x: int(contains_keyword(x, certificate_keywords))
)

interactions_mapped["is_enrolled"] = interactions_mapped["interaction_type"].apply(
    lambda x: int(contains_keyword(x, enroll_keywords))
)

print("Jumlah interaksi certificate:", interactions_mapped["is_certificate"].sum())
print("Jumlah interaksi enrollment:", interactions_mapped["is_enrolled"].sum())

## 7. Course Noise Cleaning

**Penjelasan:** Bagian ini membersihkan course yang bersifat dummy, uji coba, test, sandbox, UAT, atau noise internal sebelum data masuk ke pembentukan sequence rekomendasi. Pembersihan dilakukan pada level katalog course setelah `canonical_course_id` terbentuk dan sebelum merge ke `data_raw`, sehingga course noise tidak ikut masuk ke `user_course_base`, train, validation, test, vocabulary, maupun output rekomendasi.

**Yang perlu diperhatikan:** Filtering dibuat konservatif. Kata seperti `test`, `stress`, `trial`, atau `development` tidak otomatis dibuang jika konteksnya merupakan istilah materi valid, misalnya `Non Distructive Test`, `Stress Management`, `Trial Balance`, atau `Development`. Output utama bagian ini adalah `course_noise_audit_df` dan `course_noise_summary_df` untuk menunjukkan course apa saja yang dibuang, alasan pembersihan, dan jumlah interaksi yang terdampak.

**Keputusan course ambigu:** Course dengan nama yang ambigu, seperti `Course senin`, diperlakukan secara eksplisit melalui variabel `COURSE_SENIN_DOMAIN_DECISION`. Nilai default `keep` dipakai secara konservatif agar course tidak dihapus hanya berdasarkan nama yang ambigu. Jika domain expert memastikan course tersebut dummy/noise, nilai dapat diubah menjadi `remove` sehingga course masuk ke `MANUAL_NOISE_COURSE_NAMES` dan dikeluarkan dari katalog modeling.

In [ ]:
# ============================================================
# COURSE NOISE CLEANING
# ============================================================

import re


def normalize_noise_text(value):
    """Normalisasi teks agar pencarian pola dummy/test/noise lebih stabil."""
    if pd.isna(value):
        return ""

    text = str(value).lower().strip()
    text = text.replace("“", '"').replace("”", '"').replace("’", "'")
    text = re.sub(r"[_\-/]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# Pola high-confidence untuk course noise.
# Pola ini sengaja diarahkan ke nama course yang memang terlihat sebagai dummy/test/internal,
# bukan semua course yang memuat kata teknis seperti test, stress, trial, atau development.
COURSE_NOISE_NAME_PATTERNS = {
    "uji coba / coba course": [
        r"\buji\s*coba\b",
        r"^coba$",
        r"\bcoba\s+course\b",
        r"\bcourse\s+coba\b",
    ],
    "generic test course/material": [
        r"^test(?:\s+\d+[a-z]?|\s+scorm|\s+ssl|\s+course|\s+for\s+remedial|\s+doc\s+uat|\s+uat|\s*[-:]|$)",
        r"^materi\s+test\b",
        r"^materi\s+course\s+test\b",
        r"^mcl\s+test\b",
        r"\btest\s*&\s*pilot\s+course\b",
        r"\bdoc\s+uat\b",
        r"\buat\b",
    ],
    "dummy/sandbox/internal testing": [
        r"\bdummy\b",
        r"\bsandbox\b",
        r"\binternal\s+test\b",
        r"\btesting\s+course\b",
    ],
}

# Pola pengecualian untuk mencegah false positive pada istilah teknis valid.
COURSE_NOISE_KEEP_PATTERNS = [
    r"\bnon\s+dis?tructive\s+test\b",
    r"\bstress\s+management\b",
    r"\btrial\s+balance\b",
    r"\bparameter\s+uji\b",
    r"\bpengujian\b",
    r"\bdevelopment\b",
]

COURSE_NOISE_DESCRIPTION_PATTERNS = {
    "description states testing only": [
        r"\bhanya\s+uji\s*coba\b",
        r"\bstress\s*test\b",
        r"\bfor\s+testing\s+only\b",
    ]
}


def detect_course_noise_reason(row):
    """Mengembalikan alasan noise jika course perlu dibuang, atau string kosong jika valid."""
    course_name = normalize_noise_text(row.get("course_name", ""))
    description = normalize_noise_text(row.get("description", ""))
    combined_text = f"{course_name} {description}".strip()

    # Jika nama course masuk pola pengecualian, jangan hapus hanya karena mengandung istilah teknis.
    for keep_pattern in COURSE_NOISE_KEEP_PATTERNS:
        if re.search(keep_pattern, course_name):
            return ""

    reasons = []

    for reason, patterns in COURSE_NOISE_NAME_PATTERNS.items():
        if any(re.search(pattern, course_name) for pattern in patterns):
            reasons.append(reason)

    # Pola deskripsi hanya dipakai sebagai sinyal tambahan yang cukup jelas.
    # Ini membantu menangkap course yang deskripsinya eksplisit menyatakan uji coba/testing.
    for reason, patterns in COURSE_NOISE_DESCRIPTION_PATTERNS.items():
        if any(re.search(pattern, combined_text) for pattern in patterns):
            reasons.append(reason)

    # Hilangkan duplikasi alasan, pertahankan urutan.
    reasons = list(dict.fromkeys(reasons))
    return "; ".join(reasons)


courses_before_noise_cleaning = courses.copy()

course_interaction_count_df = (
    interactions_mapped
    .groupby("canonical_course_id")
    .size()
    .reset_index(name="interaction_count")
)

course_noise_audit_df = courses_before_noise_cleaning.copy()
course_noise_audit_df["noise_reason"] = course_noise_audit_df.apply(
    detect_course_noise_reason,
    axis=1
)
course_noise_audit_df["is_noise_course"] = course_noise_audit_df["noise_reason"].ne("")

# Daftar ini default kosong agar filtering tetap konservatif.
# Jika domain expert memastikan course tertentu sebagai dummy/noise, isi nama course di sini.
MANUAL_NOISE_COURSE_NAMES = []

# Keputusan eksplisit untuk course ambigu seperti "Course senin".
# Nilai default "keep" dipilih secara konservatif karena belum ada bukti domain
# bahwa course tersebut pasti dummy/noise. Ubah menjadi "remove" hanya jika
# domain expert memastikan bahwa course tersebut bukan materi pembelajaran valid.
COURSE_SENIN_DOMAIN_DECISION = "keep"  # Pilihan valid: "keep" atau "remove"
COURSE_SENIN_CANDIDATE_NAMES = ["Course senin"]

if COURSE_SENIN_DOMAIN_DECISION not in {"keep", "remove"}:
    raise ValueError("COURSE_SENIN_DOMAIN_DECISION harus bernilai 'keep' atau 'remove'.")

if COURSE_SENIN_DOMAIN_DECISION == "remove":
    MANUAL_NOISE_COURSE_NAMES = list(
        dict.fromkeys(MANUAL_NOISE_COURSE_NAMES + COURSE_SENIN_CANDIDATE_NAMES)
    )

manual_noise_names_normalized = {
    normalize_noise_text(name)
    for name in MANUAL_NOISE_COURSE_NAMES
    if str(name).strip()
}

if manual_noise_names_normalized:
    manual_noise_mask = course_noise_audit_df["course_name"].apply(normalize_noise_text).isin(
        manual_noise_names_normalized
    )

    def append_manual_noise_reason(existing_reason):
        if existing_reason:
            return f"{existing_reason}; manual domain-confirmed noise"
        return "manual domain-confirmed noise"

    course_noise_audit_df.loc[manual_noise_mask, "noise_reason"] = (
        course_noise_audit_df.loc[manual_noise_mask, "noise_reason"].apply(append_manual_noise_reason)
    )
    course_noise_audit_df["is_noise_course"] = course_noise_audit_df["noise_reason"].ne("")

course_noise_audit_df = course_noise_audit_df.merge(
    course_interaction_count_df,
    on="canonical_course_id",
    how="left"
)
course_noise_audit_df["interaction_count"] = (
    course_noise_audit_df["interaction_count"].fillna(0).astype(int)
)

# Kandidat manual review untuk nama course yang ambigu.
# Tabel ini tidak otomatis menghapus course; fungsinya sebagai audit agar keputusan tetap berbasis domain knowledge.
MANUAL_REVIEW_COURSE_PATTERNS = [
    r"\bcourse\s+senin\b",
]

manual_review_mask = course_noise_audit_df["course_name"].apply(
    lambda value: any(
        re.search(pattern, normalize_noise_text(value))
        for pattern in MANUAL_REVIEW_COURSE_PATTERNS
    )
)

course_manual_review_df = (
    course_noise_audit_df[manual_review_mask]
    .sort_values(["interaction_count", "course_name"], ascending=[False, True])
    .reset_index(drop=True)
)


course_senin_mask = course_noise_audit_df["course_name"].apply(
    lambda value: normalize_noise_text(value) in {
        normalize_noise_text(name)
        for name in COURSE_SENIN_CANDIDATE_NAMES
    }
)

course_senin_decision_source_df = (
    course_noise_audit_df[course_senin_mask]
    .sort_values(["interaction_count", "course_name"], ascending=[False, True])
    .reset_index(drop=True)
)

if len(course_senin_decision_source_df) > 0:
    course_senin_decision_df = course_senin_decision_source_df[
        [
            "canonical_course_id",
            "course_name",
            "description",
            "level",
            "kompetensi",
            "interaction_count",
            "is_noise_course",
            "noise_reason"
        ]
    ].copy()

    course_senin_decision_df["domain_decision"] = COURSE_SENIN_DOMAIN_DECISION
    course_senin_decision_df["pipeline_action"] = np.where(
        course_senin_decision_df["is_noise_course"],
        "removed from modeling catalog",
        "kept in modeling catalog"
    )
    course_senin_decision_df["decision_rationale"] = np.where(
        course_senin_decision_df["is_noise_course"],
        "Removed only because it was explicitly confirmed as noise or matched high-confidence noise rules.",
        "Kept by default because the name is ambiguous and there is no domain-confirmed evidence that it is dummy/noise."
    )
else:
    course_senin_decision_df = pd.DataFrame([{
        "canonical_course_id": np.nan,
        "course_name": "Course senin",
        "description": np.nan,
        "level": np.nan,
        "kompetensi": np.nan,
        "interaction_count": 0,
        "is_noise_course": False,
        "noise_reason": "",
        "domain_decision": COURSE_SENIN_DOMAIN_DECISION,
        "pipeline_action": "not found in course catalog",
        "decision_rationale": "No exact Course senin record was found in the available course catalog."
    }])

removed_noise_course_ids = set(
    course_noise_audit_df.loc[
        course_noise_audit_df["is_noise_course"],
        "canonical_course_id"
    ].astype(str)
)

removed_noise_interaction_count = int(
    interactions_mapped["canonical_course_id"].astype(str).isin(removed_noise_course_ids).sum()
)

course_noise_removed_df = (
    course_noise_audit_df[course_noise_audit_df["is_noise_course"]]
    .sort_values(["interaction_count", "course_name"], ascending=[False, True])
    .reset_index(drop=True)
)

course_noise_summary_df = pd.DataFrame({
    "metric": [
        "Course catalog sebelum noise cleaning",
        "Course noise terdeteksi dan dihapus",
        "Course catalog setelah noise cleaning",
        "Interaksi yang terdampak noise cleaning",
        "Persentase interaksi terdampak (%)",
        "Course kandidat manual review",
        "Keputusan Course senin"
    ],
    "value": [
        len(courses_before_noise_cleaning),
        len(course_noise_removed_df),
        len(courses_before_noise_cleaning) - len(course_noise_removed_df),
        removed_noise_interaction_count,
        round(removed_noise_interaction_count / max(len(interactions_mapped), 1) * 100, 4),
        len(course_manual_review_df),
        COURSE_SENIN_DOMAIN_DECISION
    ]
})

print("=== COURSE NOISE CLEANING ===")
display(course_noise_summary_df)

print("\nDaftar course dummy/test/noise yang dihapus:")
display(
    course_noise_removed_df[
        [
            "canonical_course_id",
            "course_name",
            "description",
            "level",
            "kompetensi",
            "noise_reason",
            "interaction_count"
        ]
    ].head(50)
)

print("\nKandidat manual review course ambigu:")
if len(course_manual_review_df) > 0:
    display(
        course_manual_review_df[
            [
                "canonical_course_id",
                "course_name",
                "description",
                "level",
                "kompetensi",
                "is_noise_course",
                "noise_reason",
                "interaction_count"
            ]
        ]
    )
    print(
        "Catatan: course pada tabel manual review tidak otomatis dihapus kecuali sudah "
        "dimasukkan ke MANUAL_NOISE_COURSE_NAMES berdasarkan konfirmasi domain expert."
    )
else:
    print("Tidak ada course ambigu yang cocok dengan pola manual review saat ini.")

print("\nKeputusan eksplisit untuk Course senin:")
display(course_senin_decision_df)

# Update catalog course yang akan dipakai pipeline berikutnya.
# Interaksi tidak langsung dihapus di sini; setelah merge, course noise tidak lagi match ke catalog
# dan akan terfilter pada basic filtering karena course_name/kompetensi menjadi Not Available.
# Dengan begitu audit relasi tetap transparan dan pipeline tetap aman untuk dijalankan ulang.
courses = (
    course_noise_audit_df[~course_noise_audit_df["is_noise_course"]]
    .drop(columns=["noise_reason", "is_noise_course", "interaction_count"], errors="ignore")
    .reset_index(drop=True)
)

print("\nCourses setelah noise cleaning:", courses.shape)


## 8. Merge Interactions, Courses, dan Users

**Penjelasan:** Bagian ini menggabungkan data interaksi dengan data course dan data user. Setelah merge, nilai kosong pada nama course, deskripsi, level, kompetensi, dan nama user diisi dengan nilai default agar tidak mengganggu proses berikutnya.

**Yang perlu diperhatikan:** Output yang ditampilkan adalah contoh data gabungan dan ukuran dataset setelah proses merge.

In [ ]:
data_raw = interactions_mapped.merge(
    courses,
    on="canonical_course_id",
    how="left"
)

data_raw = data_raw.merge(
    users[["user_id", "full_name"]],
    on="user_id",
    how="left"
)

data_raw["course_name"] = data_raw["course_name"].fillna("Not Available")
data_raw["description"] = data_raw["description"].fillna("Not Available")
data_raw["level"] = data_raw["level"].fillna("Not Available")
data_raw["kompetensi"] = data_raw["kompetensi"].fillna("Not Available")
data_raw["full_name"] = data_raw["full_name"].fillna("Unknown")

display(data_raw.head())
print(data_raw.shape)

## 9. Pembentukan User-Course Base

**Penjelasan:** Bagian ini membentuk data dasar per kombinasi `user_id` dan `canonical_course_id`. Jika user memiliki beberapa interaksi pada course yang sama, data digabung menjadi satu baris dengan mengambil informasi course pertama serta timestamp/interaksi paling awal.

**Yang perlu diperhatikan:** Hasil tahap ini adalah riwayat course unik per user yang sudah memiliki urutan waktu untuk dipakai sebagai sequence rekomendasi.

In [ ]:
def safe_min_datetime(x):
    x = x.dropna()
    return x.min() if len(x) > 0 else pd.NaT


def safe_min_numeric_like(x):
    x_num = pd.to_numeric(x, errors="coerce").dropna()
    return x_num.min() if len(x_num) > 0 else np.nan


user_course_base = data_raw.groupby(
    ["user_id", "canonical_course_id"],
    as_index=False
).agg(
    course_name=("course_name", "first"),
    description=("description", "first"),
    level=("level", "first"),
    kompetensi=("kompetensi", "first"),
    first_event_timestamp=("event_timestamp", safe_min_datetime),
    first_interaction_id=("interaction_id", safe_min_numeric_like),
    first_row_order=("_row_order", "min"),
    enrolled_available_raw=("is_enrolled", "max"),
    certificate_available_raw=("is_certificate", "max")
)

user_course_base["sort_timestamp"] = user_course_base["first_event_timestamp"]
user_course_base["sort_interaction_id"] = user_course_base["first_interaction_id"].fillna(999999999)
user_course_base["sort_row_order"] = user_course_base["first_row_order"]

user_course_base = user_course_base.sort_values(
    ["user_id", "sort_timestamp", "sort_interaction_id", "sort_row_order"]
).reset_index(drop=True)

display(user_course_base.head())
print("User-course base:", user_course_base.shape)

## 10. Basic Filtering Data

**Penjelasan:** Bagian ini membersihkan data awal dengan menghapus course yang informasinya tidak tersedia dan memilih user yang memiliki riwayat minimal `MIN_USER_HISTORY_FOR_SPLIT = 4` course. Batas minimal ini diperlukan karena pipeline membagi riwayat user menjadi train, validation target, dan test target.

**Yang perlu diperhatikan:** Output yang dicek adalah jumlah data, jumlah user, dan jumlah course setelah filtering dasar.

In [ ]:
MIN_USER_HISTORY_FOR_SPLIT = 4

data_base = user_course_base.copy()

data_base = data_base[data_base["course_name"] != "Not Available"].copy()
data_base = data_base[data_base["kompetensi"] != "Not Available"].copy()

user_counts = data_base.groupby("user_id")["canonical_course_id"].count()
valid_users = user_counts[user_counts >= MIN_USER_HISTORY_FOR_SPLIT].index

data_base = data_base[data_base["user_id"].isin(valid_users)].copy()

data_base = data_base.sort_values(
    ["user_id", "sort_timestamp", "sort_interaction_id", "sort_row_order"]
).reset_index(drop=True)

print("Data base setelah basic filtering:", data_base.shape)
print("Jumlah user:", data_base["user_id"].nunique())
print("Jumlah course:", data_base["canonical_course_id"].nunique())

display(data_base.head())

## 11. Temporal Split Per User

**Penjelasan:** Bagian ini membagi riwayat course setiap user secara temporal. Course terakhir dijadikan test target, course kedua terakhir dijadikan validation target, data sebelum test dijadikan pretest history, dan data sebelum validation dijadikan train core.

**Yang perlu diperhatikan:** Split seperti ini membuat evaluasi lebih realistis karena model dilatih dari riwayat masa lalu dan diuji untuk memprediksi course berikutnya.

In [ ]:
train_core_rows = []
val_target_rows = []
pretest_history_rows = []
test_target_rows = []

for user_id, group in data_base.groupby("user_id"):
    group = group.sort_values(
        ["sort_timestamp", "sort_interaction_id", "sort_row_order"]
    )

    if len(group) >= 4:
        # Course terakhir = test target
        test_target_rows.append(group.iloc[-1])

        # Semua sebelum test = history yang tersedia sebelum test
        pretest_history_rows.append(group.iloc[:-1])

        # Course kedua terakhir = validation target
        val_target_rows.append(group.iloc[-2])

        # Semua sebelum validation = train core
        train_core_rows.append(group.iloc[:-2])


train_core_raw = pd.concat(train_core_rows).reset_index(drop=True)
val_target_raw = pd.DataFrame(val_target_rows).reset_index(drop=True)
pretest_history_raw = pd.concat(pretest_history_rows).reset_index(drop=True)
test_target_raw = pd.DataFrame(test_target_rows).reset_index(drop=True)

print("=== TEMPORAL SPLIT ===")
print("Train core raw:", train_core_raw.shape)
print("Validation target raw:", val_target_raw.shape)
print("Pretest history raw:", pretest_history_raw.shape)
print("Test target raw:", test_target_raw.shape)

print("User train core:", train_core_raw["user_id"].nunique())
print("User validation:", val_target_raw["user_id"].nunique())
print("User test:", test_target_raw["user_id"].nunique())

## 12. Filtering Course Berdasarkan Train Core

**Penjelasan:** Bagian ini memilih course yang cukup sering muncul pada `train_core_raw` dengan batas minimal `MIN_COURSE_COUNT_TRAIN = 20`. Filtering dilakukan berdasarkan training saja agar validation dan test tidak ikut menentukan vocabulary course.

**Justifikasi awal:** Nilai 20 digunakan sebagai kompromi antara stabilitas sinyal pembelajaran dan keluasan katalog rekomendasi. Course dengan frekuensi sangat rendah cenderung sulit dipelajari oleh model sequence karena jumlah contoh historisnya terbatas, sedangkan threshold yang terlalu tinggi dapat menghapus terlalu banyak kandidat course dan target evaluasi. Oleh karena itu, nilai ini tidak hanya ditetapkan sebagai konstanta, tetapi juga diuji kembali pada bagian sensitivitas threshold.

**Yang perlu diperhatikan:** Kode juga memastikan user validation dan test masih memiliki history yang cukup setelah filtering sehingga evaluasi tetap valid.

In [ ]:
MIN_COURSE_COUNT_TRAIN = 20

course_counts_train = train_core_raw["canonical_course_id"].value_counts()
valid_train_courses = course_counts_train[course_counts_train >= MIN_COURSE_COUNT_TRAIN].index

print("Jumlah course di train core sebelum filtering:", train_core_raw["canonical_course_id"].nunique())
print("Jumlah valid course berdasarkan train core:", len(valid_train_courses))

train_core = train_core_raw[
    train_core_raw["canonical_course_id"].isin(valid_train_courses)
].copy()

val_target = val_target_raw[
    val_target_raw["canonical_course_id"].isin(valid_train_courses)
].copy()

pretest_history = pretest_history_raw[
    pretest_history_raw["canonical_course_id"].isin(valid_train_courses)
].copy()

test_target = test_target_raw[
    test_target_raw["canonical_course_id"].isin(valid_train_courses)
].copy()


# Pastikan validation punya history dari train_core
train_core_user_counts = train_core.groupby("user_id")["canonical_course_id"].count()
valid_val_users = train_core_user_counts[train_core_user_counts >= 2].index
val_target = val_target[val_target["user_id"].isin(valid_val_users)].copy()

# Pastikan test punya history dari pretest_history
pretest_user_counts = pretest_history.groupby("user_id")["canonical_course_id"].count()
valid_test_users = pretest_user_counts[pretest_user_counts >= 2].index
test_target = test_target[test_target["user_id"].isin(valid_test_users)].copy()

print("\n=== SETELAH FILTER COURSE DARI TRAIN SAJA ===")
print("Train core:", train_core.shape)
print("Validation target:", val_target.shape)
print("Pretest history:", pretest_history.shape)
print("Test target:", test_target.shape)

if len(train_core) == 0 or len(val_target) == 0 or len(test_target) == 0:
    raise ValueError(
        "Data terlalu sedikit setelah filtering. Coba turunkan MIN_COURSE_COUNT_TRAIN menjadi 10."
    )

## 13. Fitur Sertifikat Berbasis Train Saja

**Penjelasan:** Bagian ini menghitung statistik sertifikat per course hanya dari `train_core`. Statistik utama yang dibuat adalah `certificate_count_train` dan `course_has_certificate_train`.

**Definisi `certificate_count_train`:**

`certificate_count_train` adalah jumlah kemunculan/interaksi sebuah course pada data training (`train_core`) yang memiliki nilai `certificate_available_raw = 1`.

Secara sederhana:

`certificate_count_train(course) = jumlah baris di train_core untuk course tersebut dengan certificate_available_raw = 1`

**Hal yang perlu ditegaskan:**

1. `certificate_count_train` **bukan** jumlah sertifikat yang dimiliki oleh seorang user/karyawan.
2. `certificate_count_train` **bukan** target yang diprediksi oleh model.
3. `certificate_count_train` **bukan** statistik dari seluruh dataset, karena tidak menggunakan validation dan test.
4. Nilai ini adalah fitur tambahan berbasis course yang dihitung dari data training saja.
5. Jika sebuah course tidak memiliki informasi sertifikat pada data training, nilainya diisi `0`.

**Turunan fitur:**

- `course_has_certificate_train = 1` jika `certificate_count_train > 0`.
- `course_has_certificate_train = 0` jika `certificate_count_train = 0`.

**Alasan leakage-safe:** pendekatan ini mengurangi risiko data leakage karena statistik sertifikat yang digunakan sebagai fitur hanya dihitung dari `train_core`, bukan dari target validation atau test.

In [ ]:
course_cert_stats_train = train_core.groupby("canonical_course_id", as_index=False).agg(
    certificate_count_train=("certificate_available_raw", "sum")
)

course_cert_stats_train["course_has_certificate_train"] = np.where(
    course_cert_stats_train["certificate_count_train"] > 0,
    1,
    0
)


def add_train_only_certificate_features(df):
    df = df.copy()

    df = df.drop(
        columns=[
            "certificate_count",
            "course_has_certificate",
            "certificate_count_train",
            "course_has_certificate_train"
        ],
        errors="ignore"
    )

    df = df.merge(
        course_cert_stats_train,
        on="canonical_course_id",
        how="left"
    )

    df["certificate_count_train"] = df["certificate_count_train"].fillna(0)
    df["course_has_certificate_train"] = df["course_has_certificate_train"].fillna(0).astype(int)

    return df


train_core = add_train_only_certificate_features(train_core)
val_target = add_train_only_certificate_features(val_target)
pretest_history = add_train_only_certificate_features(pretest_history)
test_target = add_train_only_certificate_features(test_target)

print("Certificate stats dihitung dari train_core saja untuk menjaga evaluasi tetap leakage-safe.")
display(course_cert_stats_train.head())

### Interpretasi Fitur `certificate_count_train`

Bagian ini ditambahkan untuk memperjelas makna `certificate_count_train` agar tidak tertukar dengan jumlah sertifikat milik user.

| Komponen | Interpretasi yang benar |
|---|---|
| `certificate_count_train` | Frekuensi kemunculan course pada data training yang memiliki `certificate_available_raw = 1`. |
| `course_has_certificate_train` | Indikator biner apakah course tersebut pernah muncul sebagai course bersertifikat pada data training. |
| Sumber data | Hanya `train_core`. |
| Fungsi dalam model | Fitur tambahan untuk membantu model memahami karakteristik course. |
| Bukan berarti | Bukan jumlah sertifikat user, bukan target prediksi, dan bukan statistik dari validation/test. |

Dengan interpretasi ini, `certificate_count_train` dapat dijelaskan dalam paper sebagai **train-only course-level certificate frequency feature**.

In [ ]:
certificate_feature_explanation = pd.DataFrame({
    "feature": [
        "certificate_count_train",
        "course_has_certificate_train",
        "certificate_count_scaled"
    ],
    "definition": [
        "Jumlah baris pada train_core untuk course tertentu dengan certificate_available_raw = 1.",
        "Indikator 1/0 yang menunjukkan apakah certificate_count_train lebih besar dari 0.",
        "Versi MinMax-scaled dari certificate_count_train yang digunakan sebagai input numerik model."
    ],
    "data_source": [
        "train_core only",
        "derived from certificate_count_train",
        "fit scaler on train_core only"
    ],
    "not_interpreted_as": [
        "Bukan jumlah sertifikat milik user dan bukan target prediksi.",
        "Bukan label target rekomendasi.",
        "Bukan statistik dari validation/test."
    ]
})

display(certificate_feature_explanation)

certificate_stats_summary = course_cert_stats_train["certificate_count_train"].describe().to_frame().T
certificate_stats_summary.index = ["certificate_count_train"]
display(certificate_stats_summary)

course_name_lookup = train_core[["canonical_course_id", "course_name"]].drop_duplicates("canonical_course_id")
certificate_stats_preview = (
    course_cert_stats_train
    .merge(course_name_lookup, on="canonical_course_id", how="left")
    .sort_values("certificate_count_train", ascending=False)
    .head(10)[[
        "canonical_course_id",
        "course_name",
        "certificate_count_train",
        "course_has_certificate_train"
    ]]
)

display(certificate_stats_preview)


## 14. Encoding dan Scaling Train-Only

**Penjelasan:** Bagian ini membuat encoder manual untuk course, kompetensi, dan level berdasarkan `train_core` saja. Kompetensi dan level diberi token `__UNKNOWN__` untuk menangani nilai yang tidak muncul saat training.

**Peran `certificate_count_train`:** Nilai `certificate_count_train` tidak langsung dipakai dalam bentuk mentah, tetapi dinormalisasi menjadi `certificate_count_scaled` menggunakan `MinMaxScaler` yang di-fit hanya pada data training. Dengan demikian, model menerima fitur numerik dalam skala yang lebih stabil.

**Yang perlu diperhatikan:** Encoder kategori dan scaler numerik sama-sama dibangun dari `train_core` saja. Validation, pretest history, dan test hanya ditransformasi menggunakan encoder/scaler yang sudah dibuat dari training.

In [ ]:
UNKNOWN_TOKEN = "__UNKNOWN__"

# Course encoder manual dari train_core saja
train_course_ids = sorted(train_core["canonical_course_id"].astype(str).unique().tolist())

course_to_idx = {
    course_id: idx + 1
    for idx, course_id in enumerate(train_course_ids)
}

idx_to_course = {
    idx: course_id
    for course_id, idx in course_to_idx.items()
}


# Competency encoder dari train_core saja + unknown token
train_competencies = sorted(train_core["kompetensi"].astype(str).unique().tolist())
if UNKNOWN_TOKEN not in train_competencies:
    train_competencies.append(UNKNOWN_TOKEN)

competency_to_idx = {
    val: idx + 1
    for idx, val in enumerate(train_competencies)
}


# Level encoder dari train_core saja + unknown token
train_levels = sorted(train_core["level"].astype(str).unique().tolist())
if UNKNOWN_TOKEN not in train_levels:
    train_levels.append(UNKNOWN_TOKEN)

level_to_idx = {
    val: idx + 1
    for idx, val in enumerate(train_levels)
}


# Scaler fit dari train_core saja
scaler = MinMaxScaler()
scaler.fit(train_core[["certificate_count_train"]])


def encode_and_scale(df):
    df = df.copy()

    df["canonical_course_id"] = df["canonical_course_id"].astype(str)
    df["kompetensi"] = df["kompetensi"].astype(str)
    df["level"] = df["level"].astype(str)

    df["course_encoded"] = df["canonical_course_id"].map(course_to_idx)

    df["competency_clean"] = df["kompetensi"].where(
        df["kompetensi"].isin(competency_to_idx.keys()),
        UNKNOWN_TOKEN
    )

    df["level_clean"] = df["level"].where(
        df["level"].isin(level_to_idx.keys()),
        UNKNOWN_TOKEN
    )

    df["competency_encoded"] = df["competency_clean"].map(competency_to_idx)
    df["level_encoded"] = df["level_clean"].map(level_to_idx)

    df = df.dropna(subset=["course_encoded", "competency_encoded", "level_encoded"])

    df["course_encoded"] = df["course_encoded"].astype(int)
    df["competency_encoded"] = df["competency_encoded"].astype(int)
    df["level_encoded"] = df["level_encoded"].astype(int)

    df[["certificate_count_scaled"]] = scaler.transform(
        df[["certificate_count_train"]]
    )

    return df


train_core_enc = encode_and_scale(train_core)
val_target_enc = encode_and_scale(val_target)
pretest_history_enc = encode_and_scale(pretest_history)
test_target_enc = encode_and_scale(test_target)

num_courses = len(course_to_idx) + 1
num_competencies = len(competency_to_idx) + 1
num_levels = len(level_to_idx) + 1

print("Jumlah course dari train_core:", num_courses - 1)
print("Jumlah kompetensi dari train_core:", num_competencies - 1)
print("Jumlah level dari train_core:", num_levels - 1)

print("\nTrain core encoded:", train_core_enc.shape)
print("Validation target encoded:", val_target_enc.shape)
print("Pretest history encoded:", pretest_history_enc.shape)
print("Test target encoded:", test_target_enc.shape)

## 15. Course Catalog

**Penjelasan:** Bagian ini membuat katalog course dari data training yang sudah di-encode. Catalog menyimpan `course_encoded`, `canonical_course_id`, nama course, deskripsi, kompetensi, level, dan fitur sertifikat.

**Yang perlu diperhatikan:** Catalog ini dipakai untuk menerjemahkan output model dari ID hasil encoding menjadi informasi course yang bisa dibaca pengguna.

In [ ]:
course_catalog = train_core_enc.drop_duplicates("course_encoded")[[
    "course_encoded",
    "canonical_course_id",
    "course_name",
    "description",
    "kompetensi",
    "level",
    "course_has_certificate_train",
    "certificate_count_train"
]].reset_index(drop=True)

display(course_catalog.head())

## 16. Membuat Sequence Training

**Penjelasan:** Bagian ini mengubah riwayat course setiap user menjadi sequence input dan target training. Contohnya, jika riwayat user adalah A, B, C, D, maka training sample dapat dibentuk menjadi A → B, A-B → C, dan A-B-C → D.

**Yang perlu diperhatikan:** Setiap sequence dipadding sampai `max_sequence_length = 10`, sehingga semua input memiliki panjang yang sama saat masuk ke model LSTM.

In [ ]:
max_sequence_length = 10

train_sequences = train_core_enc.groupby("user_id").agg({
    "course_encoded": list,
    "competency_encoded": list,
    "level_encoded": list,
    "course_has_certificate_train": list,
    "certificate_count_scaled": list
}).reset_index()


X_course_train = []
X_competency_train = []
X_level_train = []
X_course_has_certificate_train = []
X_certificate_count_train = []
y_train = []

for _, row in train_sequences.iterrows():
    course_seq = row["course_encoded"]
    competency_seq = row["competency_encoded"]
    level_seq = row["level_encoded"]
    course_cert_seq = row["course_has_certificate_train"]
    cert_count_seq = row["certificate_count_scaled"]

    if len(course_seq) > 1:
        for i in range(1, len(course_seq)):
            X_course_train.append(course_seq[:i])
            X_competency_train.append(competency_seq[:i])
            X_level_train.append(level_seq[:i])
            X_course_has_certificate_train.append(course_cert_seq[:i])
            X_certificate_count_train.append(cert_count_seq[:i])
            y_train.append(course_seq[i])


X_course_train = pad_sequences(X_course_train, maxlen=max_sequence_length, padding="pre")
X_competency_train = pad_sequences(X_competency_train, maxlen=max_sequence_length, padding="pre")
X_level_train = pad_sequences(X_level_train, maxlen=max_sequence_length, padding="pre")

X_course_has_certificate_train = pad_sequences(
    X_course_has_certificate_train,
    maxlen=max_sequence_length,
    padding="pre",
    dtype="float32"
)

X_certificate_count_train = pad_sequences(
    X_certificate_count_train,
    maxlen=max_sequence_length,
    padding="pre",
    dtype="float32"
)

X_course_has_certificate_train = np.expand_dims(X_course_has_certificate_train, axis=-1)
X_certificate_count_train = np.expand_dims(X_certificate_count_train, axis=-1)

y_train = np.array(y_train)

print("X_course_train:", X_course_train.shape)
print("X_competency_train:", X_competency_train.shape)
print("X_level_train:", X_level_train.shape)
print("X_course_has_certificate_train:", X_course_has_certificate_train.shape)
print("X_certificate_count_train:", X_certificate_count_train.shape)
print("y_train:", y_train.shape)

## 17. Membuat Dataset Validation dan Test Holdout

**Penjelasan:** Bagian ini mendefinisikan fungsi `make_temporal_holdout_dataset()` untuk membuat input evaluasi dari history dan target temporal. Fungsi ini menggabungkan history user dengan target course yang benar, lalu menyiapkan input model dan metadata ground truth.

**Yang perlu diperhatikan:** Fungsi ini dipakai agar validation dan test memiliki format input yang sama dengan training.

In [ ]:
def make_temporal_holdout_dataset(history_df, target_df, label_name="test"):
    history_sequences = history_df.groupby("user_id").agg({
        "course_encoded": list,
        "competency_encoded": list,
        "level_encoded": list,
        "course_has_certificate_train": list,
        "certificate_count_scaled": list
    }).reset_index()

    merged = target_df.merge(
        history_sequences,
        on="user_id",
        how="inner",
        suffixes=("_target", "_history")
    )

    X_course = []
    X_competency = []
    X_level = []
    X_course_has_certificate = []
    X_certificate_count = []
    y = []
    metadata = []

    for _, row in merged.iterrows():
        history_course = row["course_encoded_history"]

        if len(history_course) < 2:
            continue

        X_course.append(history_course)
        X_competency.append(row["competency_encoded_history"])
        X_level.append(row["level_encoded_history"])
        X_course_has_certificate.append(row["course_has_certificate_train_history"])
        X_certificate_count.append(row["certificate_count_scaled_history"])

        y.append(row["course_encoded_target"])

        metadata.append({
            "user_id": row["user_id"],
            "history_encoded": history_course,
            "ground_truth_encoded": row["course_encoded_target"],
            "ground_truth_course": row["course_name"],
            "sequence_length": len(history_course),
            "dataset": label_name
        })

    X_course = pad_sequences(X_course, maxlen=max_sequence_length, padding="pre")
    X_competency = pad_sequences(X_competency, maxlen=max_sequence_length, padding="pre")
    X_level = pad_sequences(X_level, maxlen=max_sequence_length, padding="pre")

    X_course_has_certificate = pad_sequences(
        X_course_has_certificate,
        maxlen=max_sequence_length,
        padding="pre",
        dtype="float32"
    )

    X_certificate_count = pad_sequences(
        X_certificate_count,
        maxlen=max_sequence_length,
        padding="pre",
        dtype="float32"
    )

    X_course_has_certificate = np.expand_dims(X_course_has_certificate, axis=-1)
    X_certificate_count = np.expand_dims(X_certificate_count, axis=-1)

    y = np.array(y).astype(int)
    metadata = pd.DataFrame(metadata)

    return (
        X_course,
        X_competency,
        X_level,
        X_course_has_certificate,
        X_certificate_count,
        y,
        metadata
    )

## 18. Menyiapkan Data Validation dan Test

**Penjelasan:** Bagian ini memanggil fungsi holdout untuk membentuk dataset validation dan dataset test. Validation memakai `train_core_enc` sebagai history dan `val_target_enc` sebagai target, sedangkan test memakai `pretest_history_enc` sebagai history dan `test_target_enc` sebagai target.

**Yang perlu diperhatikan:** Output yang dicek adalah jumlah sample validation dan test yang berhasil dibentuk.

In [ ]:
(
    X_course_val,
    X_competency_val,
    X_level_val,
    X_course_has_certificate_val,
    X_certificate_count_val,
    y_val,
    val_metadata
) = make_temporal_holdout_dataset(
    history_df=train_core_enc,
    target_df=val_target_enc,
    label_name="validation"
)


(
    X_course_test,
    X_competency_test,
    X_level_test,
    X_course_has_certificate_test,
    X_certificate_count_test,
    y_test,
    test_metadata
) = make_temporal_holdout_dataset(
    history_df=pretest_history_enc,
    target_df=test_target_enc,
    label_name="test"
)


print("=== VALIDATION TEMPORAL ===")
print("X_course_val:", X_course_val.shape)
print("y_val:", y_val.shape)
print("val_metadata:", val_metadata.shape)

print("\n=== TEST ISOLATED ===")
print("X_course_test:", X_course_test.shape)
print("y_test:", y_test.shape)
print("test_metadata:", test_metadata.shape)

if len(y_val) == 0 or len(y_test) == 0:
    raise ValueError("Validation atau test kosong. Turunkan MIN_COURSE_COUNT_TRAIN.")

## 19. Leakage-Safe Evaluation Audit

**Penjelasan:** Bagian ini melakukan audit kebocoran data (*data leakage*) pada skenario rekomendasi course berbasis urutan waktu. Audit dilakukan untuk memastikan bahwa target validation dan target test tidak ikut masuk ke input history yang digunakan untuk memprediksi target tersebut.

**Prinsip leakage-safe yang digunakan:**

1. **Training model hanya menggunakan `train_core_enc`.** Validation target dan test target tidak digunakan untuk membentuk sequence training.
2. **Vocabulary course, encoder kompetensi, encoder level, fitur sertifikat, dan scaler numerik dibangun dari data training saja.** Validation dan test hanya ditransformasi menggunakan objek transformasi dari training.
3. **Validation mengevaluasi course kedua terakhir.** Input validation hanya berasal dari `train_core_enc`, yaitu riwayat sebelum validation target.
4. **Test mengevaluasi course terakhir.** Input test berasal dari `pretest_history_enc`, yaitu seluruh riwayat sebelum test target. Dengan demikian, course validation boleh menjadi bagian dari history test karena secara temporal memang terjadi sebelum test target.
5. **Course target tidak boleh muncul di history evaluasi untuk user yang sama.** Jika target sudah ada di history, model mengevaluasi course yang sebenarnya sudah diketahui sebelumnya sehingga hasil menjadi tidak valid.

**Yang perlu diperhatikan:** Jika seluruh indikator leakage bernilai `0` dan status audit bernilai `PASS`, maka pipeline evaluasi dapat dijelaskan sebagai *temporal, train-only transformed, and leakage-safe* untuk skenario *next-course recommendation*.

In [ ]:
def get_user_course_pairs(df):
    """Membuat pasangan unik user-course untuk audit overlap."""
    return df[["user_id", "canonical_course_id"]].astype(str).drop_duplicates()


# Audit 1: validation target tidak boleh masuk train_core user yang sama
val_leak_records = []

for _, row in val_metadata.iterrows():
    user_id = str(row["user_id"])
    gt = int(row["ground_truth_encoded"])

    user_train_courses = train_core_enc[
        train_core_enc["user_id"].astype(str) == user_id
    ]["course_encoded"].astype(int).tolist()

    if gt in user_train_courses:
        val_leak_records.append({
            "user_id": user_id,
            "ground_truth_encoded": gt
        })

val_leak_df = pd.DataFrame(val_leak_records)


# Audit 2: test target tidak boleh masuk pretest history user yang sama
# Catatan: pretest_history boleh memuat validation target karena posisinya memang sebelum test target.
test_leak_records = []

for _, row in test_metadata.iterrows():
    user_id = str(row["user_id"])
    gt = int(row["ground_truth_encoded"])

    user_history_courses = pretest_history_enc[
        pretest_history_enc["user_id"].astype(str) == user_id
    ]["course_encoded"].astype(int).tolist()

    if gt in user_history_courses:
        test_leak_records.append({
            "user_id": user_id,
            "ground_truth_encoded": gt
        })


test_leak_df = pd.DataFrame(test_leak_records)


# Audit 3: pasangan user-course test tidak overlap dengan train_core
train_pairs = get_user_course_pairs(train_core_enc)
test_pairs = get_user_course_pairs(test_target_enc)
val_pairs = get_user_course_pairs(val_target_enc)

overlap_train_test = train_pairs.merge(
    test_pairs,
    on=["user_id", "canonical_course_id"],
    how="inner"
)


# Audit 4: pasangan user-course validation tidak overlap dengan train_core
overlap_train_val = train_pairs.merge(
    val_pairs,
    on=["user_id", "canonical_course_id"],
    how="inner"
)


leakage_audit_df = pd.DataFrame({
    "audit_item": [
        "Validation target appears in train history for the same user",
        "Test target appears in pretest history for the same user",
        "User-course overlap between train_core and test_target",
        "User-course overlap between train_core and val_target"
    ],
    "count": [
        len(val_leak_df),
        len(test_leak_df),
        len(overlap_train_test),
        len(overlap_train_val)
    ]
})

leakage_audit_df["status"] = np.where(
    leakage_audit_df["count"] == 0,
    "PASS",
    "CHECK"
)


train_only_transform_audit = pd.DataFrame({
    "component": [
        "Course vocabulary",
        "Competency encoder",
        "Level encoder",
        "Certificate feature statistics",
        "Certificate count scaler",
        "Course catalog"
    ],
    "source_used_for_fit_or_build": [
        "train_core only",
        "train_core only",
        "train_core only",
        "train_core only",
        "train_core only",
        "train_core_enc only"
    ],
    "leakage_safe_reason": [
        "Course kandidat valid ditentukan dari kemunculan pada training, bukan dari validation/test.",
        "Kategori kompetensi validation/test hanya dipetakan memakai encoder dari training.",
        "Kategori level validation/test hanya dipetakan memakai encoder dari training.",
        "certificate_count_train dihitung dari train_core, bukan seluruh dataset.",
        "MinMaxScaler di-fit pada certificate_count_train milik train_core.",
        "Metadata rekomendasi akhir berasal dari course yang dikenal pada training."
    ]
})


print("=== LEAKAGE AUDIT ===")
display(leakage_audit_df)

print("\n=== TRAIN-ONLY TRANSFORMATION AUDIT ===")
display(train_only_transform_audit)

if (leakage_audit_df["count"] == 0).all():
    print("\nKESIMPULAN: PASS. Pipeline sudah temporal, train-only transformed, dan leakage-safe untuk skenario next-course recommendation.")
else:
    print("\nKESIMPULAN: CHECK. Masih ada indikasi leakage/overlap yang perlu diperiksa.")

    if len(val_leak_df) > 0:
        print("\nContoh validation leakage:")
        display(val_leak_df.head())

    if len(test_leak_df) > 0:
        print("\nContoh test leakage:")
        display(test_leak_df.head())

    if len(overlap_train_test) > 0:
        print("\nContoh overlap train-test:")
        display(overlap_train_test.head())

    if len(overlap_train_val) > 0:
        print("\nContoh overlap train-validation:")
        display(overlap_train_val.head())

### Protokol Pencegahan Data Leakage

Pipeline evaluasi pada penelitian ini dirancang menggunakan skema pemisahan temporal per user untuk mengurangi risiko *data leakage*. Pada setiap user, course terakhir digunakan sebagai target test, course kedua terakhir digunakan sebagai target validation, sedangkan riwayat sebelum target validation digunakan sebagai data training utama (`train_core`). Dengan cara ini, model dilatih hanya menggunakan interaksi yang terjadi sebelum target validation dan test.

Seluruh proses pembentukan fitur juga dilakukan secara *train-only*. Vocabulary course, encoder kompetensi, encoder level, statistik sertifikat (`certificate_count_train`), serta scaler numerik dibangun hanya dari `train_core`. Data validation dan test tidak digunakan untuk membangun encoder, scaler, maupun statistik fitur; keduanya hanya ditransformasi menggunakan objek yang sudah dibentuk dari training. Strategi ini mencegah informasi dari data evaluasi masuk ke tahap pelatihan atau preprocessing.

Pada evaluasi validation, input history berasal dari `train_core`, sedangkan targetnya adalah course kedua terakhir. Pada evaluasi test, input history berasal dari `pretest_history`, yaitu seluruh riwayat sebelum course terakhir. Penggunaan `pretest_history` pada test masih valid untuk skenario *next-course recommendation* karena sistem diasumsikan merekomendasikan course berikutnya berdasarkan semua course yang sudah diketahui sebelum target test. Dengan demikian, course validation dapat muncul sebagai bagian dari history test karena posisinya memang terjadi sebelum test target, bukan setelahnya.

Selain itu, audit leakage dilakukan untuk memastikan target validation tidak muncul dalam history training user yang sama, target test tidak muncul dalam pretest history user yang sama, serta tidak terdapat overlap pasangan user-course antara training dan target evaluasi. Apabila seluruh hasil audit bernilai nol, maka pipeline dapat dinyatakan *temporal, train-only transformed, and leakage-safe* untuk evaluasi rekomendasi Top-N.

## 20. Distribusi Target Test

**Penjelasan:** Bagian ini menganalisis distribusi course yang menjadi target pada data test. Tujuannya adalah memastikan evaluasi tidak hanya dilaporkan melalui metrik ranking, tetapi juga dijelaskan karakteristik target yang sedang diprediksi.

Analisis ini penting karena nilai Hit@K yang tinggi bisa dipengaruhi oleh dominasi beberapa course populer. Jika sebagian besar target test berasal dari sedikit course, maka model berpotensi terlihat baik hanya karena sering merekomendasikan course populer. Dengan menampilkan distribusi target test, penelitian menjadi lebih transparan dan lebih siap untuk dilaporkan dalam paper.

**Output yang ditampilkan:**

1. jumlah total sample test;
2. jumlah unique course target pada test;
3. proporsi target yang masuk Top-1, Top-5, dan Top-10 course paling sering muncul;
4. tabel Top-20 target test;
5. bucket frekuensi target untuk melihat apakah distribusi bersifat long-tail;
6. grafik Top-15 course target test.

In [ ]:
import matplotlib.pyplot as plt

# Distribusi course yang menjadi ground truth pada test set
# Sumber utama: test_metadata yang sudah dibentuk secara temporal dan leakage-safe.
test_target_distribution = (
    test_metadata
    .groupby(["ground_truth_encoded", "ground_truth_course"])
    .size()
    .reset_index(name="test_target_count")
    .sort_values("test_target_count", ascending=False)
    .reset_index(drop=True)
)

total_test_targets = int(test_target_distribution["test_target_count"].sum())
unique_test_targets = int(test_target_distribution["ground_truth_encoded"].nunique())

test_target_distribution["target_percentage"] = (
    test_target_distribution["test_target_count"] / total_test_targets * 100
)

test_target_distribution["cumulative_percentage"] = (
    test_target_distribution["target_percentage"].cumsum()
)

test_target_distribution["rank"] = np.arange(1, len(test_target_distribution) + 1)

test_target_distribution = test_target_distribution[
    [
        "rank",
        "ground_truth_encoded",
        "ground_truth_course",
        "test_target_count",
        "target_percentage",
        "cumulative_percentage"
    ]
]

# Tambahan konteks: popularitas course target tersebut pada data training.
# Ini membantu membaca apakah target test didominasi course populer dari training.
train_course_popularity = (
    train_core_enc
    .groupby("course_encoded")
    .size()
    .reset_index(name="train_core_count")
)

test_target_distribution = test_target_distribution.merge(
    train_course_popularity,
    left_on="ground_truth_encoded",
    right_on="course_encoded",
    how="left"
).drop(columns=["course_encoded"])

test_target_distribution["train_core_count"] = (
    test_target_distribution["train_core_count"].fillna(0).astype(int)
)

# Ringkasan konsentrasi distribusi target test
top_1_share = float(test_target_distribution.head(1)["target_percentage"].sum())
top_5_share = float(test_target_distribution.head(5)["target_percentage"].sum())
top_10_share = float(test_target_distribution.head(10)["target_percentage"].sum())
top_20_share = float(test_target_distribution.head(20)["target_percentage"].sum())

median_target_count = float(test_target_distribution["test_target_count"].median())
max_target_count = int(test_target_distribution["test_target_count"].max())
min_target_count = int(test_target_distribution["test_target_count"].min())

target_distribution_summary_df = pd.DataFrame({
    "metric": [
        "Total test samples",
        "Unique target courses in test",
        "Top-1 target share (%)",
        "Top-5 target share (%)",
        "Top-10 target share (%)",
        "Top-20 target share (%)",
        "Max target count for one course",
        "Median target count per course",
        "Min target count for one course"
    ],
    "value": [
        total_test_targets,
        unique_test_targets,
        round(top_1_share, 2),
        round(top_5_share, 2),
        round(top_10_share, 2),
        round(top_20_share, 2),
        max_target_count,
        round(median_target_count, 2),
        min_target_count
    ]
})

print("=== RINGKASAN DISTRIBUSI TARGET TEST ===")
display(target_distribution_summary_df)

print("\n=== TOP-20 COURSE TARGET TEST ===")
display(
    test_target_distribution.head(20).assign(
        target_percentage=lambda df: df["target_percentage"].round(2),
        cumulative_percentage=lambda df: df["cumulative_percentage"].round(2)
    )
)

# Bucket frekuensi target untuk melihat apakah distribusi target test long-tail.
def target_frequency_bucket(count):
    if count == 1:
        return "1 sample"
    elif count <= 5:
        return "2-5 samples"
    elif count <= 10:
        return "6-10 samples"
    elif count <= 20:
        return "11-20 samples"
    else:
        return ">20 samples"


target_frequency_bucket_df = (
    test_target_distribution
    .assign(frequency_bucket=lambda df: df["test_target_count"].apply(target_frequency_bucket))
    .groupby("frequency_bucket")
    .agg(
        unique_course_count=("ground_truth_encoded", "nunique"),
        total_test_samples=("test_target_count", "sum")
    )
    .reset_index()
)

bucket_order = ["1 sample", "2-5 samples", "6-10 samples", "11-20 samples", ">20 samples"]
target_frequency_bucket_df["frequency_bucket"] = pd.Categorical(
    target_frequency_bucket_df["frequency_bucket"],
    categories=bucket_order,
    ordered=True
)

target_frequency_bucket_df = target_frequency_bucket_df.sort_values("frequency_bucket")
target_frequency_bucket_df["sample_percentage"] = (
    target_frequency_bucket_df["total_test_samples"] / total_test_targets * 100
).round(2)

print("\n=== BUCKET FREKUENSI TARGET TEST ===")
display(target_frequency_bucket_df)

# Visualisasi Top-15 target test
plot_top_n = 15
plot_df = test_target_distribution.head(plot_top_n).sort_values("test_target_count", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df["ground_truth_course"], plot_df["test_target_count"])
plt.xlabel("Jumlah kemunculan sebagai target test")
plt.ylabel("Course target")
plt.title(f"Top-{plot_top_n} Distribusi Course Target pada Test Set")
plt.tight_layout()
plt.show()

print("\nINTERPRETASI:")
print(
    f"Data test memiliki {total_test_targets} target dari {unique_test_targets} unique course. "
    f"Top-5 course target mencakup {top_5_share:.2f}% dari seluruh target test, "
    f"sedangkan Top-10 mencakup {top_10_share:.2f}%. "
    "Semakin kecil proporsi Top-5/Top-10, semakin tersebar target evaluasi; "
    "semakin besar proporsinya, semakin kuat indikasi bahwa evaluasi dipengaruhi oleh course populer."
)

### Analisis Distribusi Target pada Data Uji

Distribusi target test dianalisis untuk memahami karakteristik course yang diprediksi pada tahap evaluasi akhir. Analisis ini dilakukan dengan menghitung frekuensi setiap course yang muncul sebagai ground truth pada test set, jumlah unique target course, serta proporsi target yang terkonsentrasi pada Top-1, Top-5, Top-10, dan Top-20 course paling sering muncul.

Analisis ini diperlukan karena performa sistem rekomendasi tidak hanya dipengaruhi oleh arsitektur model, tetapi juga oleh distribusi item pada data evaluasi. Jika target test sangat terkonsentrasi pada sejumlah kecil course populer, maka nilai Hit@K yang tinggi dapat sebagian dipengaruhi oleh bias popularitas. Sebaliknya, apabila target tersebar pada banyak course, maka evaluasi menunjukkan bahwa model diuji pada variasi target yang lebih luas.

Dengan demikian, distribusi target test digunakan sebagai informasi pendukung untuk menafsirkan hasil Hit@K, MRR@10, dan NDCG@10 secara lebih transparan. Bagian ini juga memperkuat validitas evaluasi karena pembaca dapat melihat apakah test set didominasi oleh course populer atau memiliki pola long-tail yang lebih beragam.

## 21. Model Encoder–Decoder LSTM dengan Attention Layer

**Penjelasan:** Bagian ini membangun arsitektur model Encoder–Decoder LSTM yang dilengkapi dengan **temporal attention layer**. Input model terdiri dari riwayat course, kompetensi, level, indikator sertifikat, dan jumlah sertifikat yang sudah dinormalisasi.

**Peran attention layer:** Attention layer memberi bobot kepentingan yang berbeda pada setiap course dalam riwayat user. Dengan demikian, model tidak memperlakukan semua course sebelumnya sebagai informasi yang sama penting, tetapi dapat memfokuskan representasi pada course yang lebih relevan untuk memprediksi course berikutnya.

**Yang perlu diperhatikan:** Output sequence dari encoder LSTM diproses oleh attention layer untuk menghasilkan `context_vector`. Padding sequence dengan `course_encoded = 0` di-mask agar tidak ikut memengaruhi bobot attention. Context vector kemudian digabungkan dengan hidden state encoder, lalu diproses oleh decoder LSTM sebelum masuk ke layer softmax untuk memprediksi probabilitas course berikutnya.

In [ ]:
course_embedding_dim = 96
competency_embedding_dim = 32
level_embedding_dim = 8
latent_dim = 192


class TemporalAttention(tf.keras.layers.Layer):
    """
    Temporal attention layer untuk output sequence LSTM.

    Output:
    1. context_vector: representasi gabungan dari encoder output berdasarkan bobot attention.
    2. attention_weights: bobot perhatian untuk setiap timestep dalam sequence.

    Padding dengan course_encoded = 0 di-mask agar tidak ikut diberi bobot attention.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = Dense(1, activation="tanh", name="attention_score_dense")

    def call(self, inputs):
        sequence_output, course_sequence = inputs

        # sequence_output shape: (batch_size, sequence_length, latent_dim)
        # course_sequence shape: (batch_size, sequence_length)
        attention_score = self.score_dense(sequence_output)

        # Mask padding. Course encoded = 0 adalah PAD.
        mask = tf.cast(tf.not_equal(course_sequence, 0), tf.float32)
        mask = tf.expand_dims(mask, axis=-1)

        very_negative_value = tf.ones_like(attention_score) * -1e9
        masked_attention_score = tf.where(
            tf.equal(mask, 1.0),
            attention_score,
            very_negative_value
        )

        attention_weights = tf.nn.softmax(masked_attention_score, axis=1)

        context_vector = tf.reduce_sum(
            sequence_output * attention_weights,
            axis=1
        )

        attention_weights = tf.squeeze(attention_weights, axis=-1)

        return context_vector, attention_weights


course_input = Input(shape=(max_sequence_length,), name="course_input")

course_embedding = Embedding(
    input_dim=num_courses,
    output_dim=course_embedding_dim,
    name="course_embedding"
)(course_input)


competency_input = Input(shape=(max_sequence_length,), name="competency_input")

competency_embedding = Embedding(
    input_dim=num_competencies,
    output_dim=competency_embedding_dim,
    name="competency_embedding"
)(competency_input)


level_input = Input(shape=(max_sequence_length,), name="level_input")

level_embedding = Embedding(
    input_dim=num_levels,
    output_dim=level_embedding_dim,
    name="level_embedding"
)(level_input)


course_has_certificate_input = Input(
    shape=(max_sequence_length, 1),
    name="course_has_certificate_input"
)

certificate_count_input = Input(
    shape=(max_sequence_length, 1),
    name="certificate_count_input"
)


combined_input = Concatenate(name="combined_encoder_input")([
    course_embedding,
    competency_embedding,
    level_embedding,
    course_has_certificate_input,
    certificate_count_input
])


encoder_sequence_output, state_h, state_c = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="encoder_lstm"
)(combined_input)


attention_context, attention_weights = TemporalAttention(
    name="temporal_attention"
)([
    encoder_sequence_output,
    course_input
])


attention_enhanced_state = Concatenate(
    name="attention_state_concatenate"
)([
    state_h,
    attention_context
])


attention_enhanced_state = Dense(
    latent_dim,
    activation="tanh",
    name="attention_enhanced_state"
)(attention_enhanced_state)


decoder_input = RepeatVector(
    1,
    name="decoder_repeat"
)(attention_enhanced_state)


decoder_output = LSTM(
    latent_dim,
    return_sequences=False,
    name="decoder_lstm"
)(
    decoder_input,
    initial_state=[attention_enhanced_state, state_c]
)


decoder_representation = Dropout(
    0.3,
    name="decoder_representation"
)(decoder_output)


course_prediction = Dense(
    num_courses,
    activation="softmax",
    name="course_prediction"
)(decoder_representation)


model = Model(
    inputs=[
        course_input,
        competency_input,
        level_input,
        course_has_certificate_input,
        certificate_count_input
    ],
    outputs=course_prediction
)


attention_weight_model = Model(
    inputs=[
        course_input,
        competency_input,
        level_input,
        course_has_certificate_input,
        certificate_count_input
    ],
    outputs=attention_weights,
    name="attention_weight_model"
)


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy"
)

model.summary()

## 22. Training Model

**Penjelasan:** Bagian ini melatih model menggunakan data sequence training dan validation temporal. `EarlyStopping` dipakai untuk menghentikan training ketika `val_loss` tidak membaik, sedangkan `ReduceLROnPlateau` menurunkan learning rate jika performa validasi stagnan.

**Yang perlu diperhatikan:** Pada recommender system, performa utama tidak dinilai dengan metrik klasifikasi biasa. Model tetap dilatih dengan `sparse_categorical_crossentropy`, tetapi kualitas rekomendasi dievaluasi pada tahap berikutnya menggunakan metrik ranking Top-K seperti Hit@K / Recall@K, MRR@10, dan NDCG@10.

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-5
)

history = model.fit(
    [
        X_course_train,
        X_competency_train,
        X_level_train,
        X_course_has_certificate_train,
        X_certificate_count_train
    ],
    y_train,
    validation_data=(
        [
            X_course_val,
            X_competency_val,
            X_level_val,
            X_course_has_certificate_val,
            X_certificate_count_val
        ],
        y_val
    ),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    shuffle=True
)

### 22.1 Interpretasi Attention Weight

**Penjelasan:** Bagian ini menampilkan bobot attention untuk satu user contoh. Nilai `attention_weight` menunjukkan seberapa besar perhatian model terhadap setiap course dalam riwayat input ketika membentuk prediksi course berikutnya.

**Cara membaca:** Course dengan `attention_weight` lebih tinggi berarti course tersebut dianggap lebih penting oleh model dalam membentuk representasi user. Nilai ini bukan skor rekomendasi akhir, tetapi alat bantu interpretasi untuk melihat bagian riwayat mana yang lebih diperhatikan model.

In [ ]:
def get_attention_course_name(encoded_id):
    """Mengambil nama course dari course_encoded untuk tabel attention."""
    encoded_id = int(encoded_id)

    if encoded_id == 0:
        return "PAD"

    info = course_catalog[course_catalog["course_encoded"] == encoded_id]

    if not info.empty:
        return info.iloc[0]["course_name"]

    canonical_id = idx_to_course.get(encoded_id, "Unknown Course")
    return str(canonical_id)


def show_attention_for_user(user_id, source_df=pretest_history_enc):
    """
    Menampilkan bobot attention untuk riwayat course user tertentu.

    Semakin besar attention_weight, semakin besar kontribusi course tersebut
    dalam membentuk prediksi course berikutnya.
    """
    user_id = str(user_id).strip().replace(".0", "")
    user_history = source_df[source_df["user_id"].astype(str) == user_id].copy()

    if user_history.empty:
        print("User ID tidak ditemukan.")
        return None

    sort_columns = [
        col for col in [
            "sort_timestamp",
            "sort_interaction_id",
            "sort_row_order",
            "first_event_timestamp"
        ]
        if col in user_history.columns
    ]

    if sort_columns:
        user_history = user_history.sort_values(sort_columns)

    course_seq = user_history["course_encoded"].astype(int).tolist()
    competency_seq = user_history["competency_encoded"].astype(int).tolist()
    level_seq = user_history["level_encoded"].astype(int).tolist()
    cert_has_seq = user_history["course_has_certificate_train"].astype(float).tolist()
    cert_count_seq = user_history["certificate_count_scaled"].astype(float).tolist()

    X_course_single = pad_sequences(
        [course_seq],
        maxlen=max_sequence_length,
        padding="pre"
    )

    X_competency_single = pad_sequences(
        [competency_seq],
        maxlen=max_sequence_length,
        padding="pre"
    )

    X_level_single = pad_sequences(
        [level_seq],
        maxlen=max_sequence_length,
        padding="pre"
    )

    X_course_has_certificate_single = pad_sequences(
        [cert_has_seq],
        maxlen=max_sequence_length,
        padding="pre",
        dtype="float32"
    )

    X_certificate_count_single = pad_sequences(
        [cert_count_seq],
        maxlen=max_sequence_length,
        padding="pre",
        dtype="float32"
    )

    X_course_has_certificate_single = np.expand_dims(
        X_course_has_certificate_single,
        axis=-1
    )

    X_certificate_count_single = np.expand_dims(
        X_certificate_count_single,
        axis=-1
    )

    attention_values = attention_weight_model.predict(
        [
            X_course_single,
            X_competency_single,
            X_level_single,
            X_course_has_certificate_single,
            X_certificate_count_single
        ],
        verbose=0
    )[0]

    padded_course_sequence = X_course_single[0]

    attention_result = pd.DataFrame({
        "input_position": np.arange(1, max_sequence_length + 1),
        "course_encoded": padded_course_sequence,
        "attention_weight": attention_values
    })

    attention_result = attention_result[attention_result["course_encoded"] != 0].copy()
    attention_result["course_name"] = attention_result["course_encoded"].apply(get_attention_course_name)

    attention_result = attention_result[[
        "input_position",
        "course_encoded",
        "course_name",
        "attention_weight"
    ]]

    attention_result["attention_weight"] = attention_result["attention_weight"].round(4)

    attention_result = attention_result.sort_values(
        "attention_weight",
        ascending=False
    ).reset_index(drop=True)

    return attention_result


sample_user_id = test_metadata["user_id"].iloc[0]
attention_example_df = show_attention_for_user(sample_user_id)

print("Contoh bobot attention untuk user:", sample_user_id)
display(attention_example_df)

### Interpretasi Mekanisme Attention

Model rekomendasi pada penelitian ini dikembangkan menggunakan arsitektur Encoder–Decoder LSTM yang dilengkapi dengan temporal attention layer. Attention layer digunakan untuk memberikan bobot kepentingan yang berbeda pada setiap course dalam riwayat pembelajaran karyawan. Dengan pendekatan ini, model tidak memperlakukan seluruh course sebelumnya sebagai informasi yang sama penting, tetapi dapat memfokuskan representasi pada course yang lebih relevan terhadap prediksi course berikutnya.

Secara teknis, output sequence dari encoder LSTM diproses oleh temporal attention layer untuk menghasilkan bobot attention pada setiap timestep. Padding sequence diberi masking sehingga tidak ikut memengaruhi pembentukan context vector. Context vector hasil attention kemudian digabungkan dengan hidden state encoder untuk membentuk representasi yang lebih informatif sebelum diproses oleh decoder LSTM.

Penambahan attention layer juga meningkatkan aspek interpretabilitas model, karena bobot attention dapat dianalisis untuk melihat course mana dalam riwayat user yang paling berkontribusi terhadap rekomendasi. Dengan demikian, model tidak hanya menghasilkan daftar rekomendasi Top-N, tetapi juga menyediakan informasi pendukung mengenai pola perhatian model terhadap riwayat pembelajaran user.

## 23. Fungsi Masking dan Evaluasi Ranking Top-K

**Penjelasan:** Bagian ini membuat fungsi untuk memfilter course yang sudah pernah diambil user agar tidak direkomendasikan ulang. Setelah masking, skor model dievaluasi menggunakan metrik ranking rekomendasi: **Hit@1, Hit@5, Hit@10, Recall@1, Recall@5, Recall@10, MRR@10, dan NDCG@10**.

**Yang perlu diperhatikan:** Pada skenario next-course recommendation, setiap sampel test hanya memiliki satu ground-truth course. Karena itu, **Hit@K bernilai sama dengan Recall@K**: nilainya 1 jika course target muncul dalam Top-K rekomendasi, dan 0 jika tidak muncul. MRR@10 menilai posisi ranking target, sedangkan NDCG@10 memberi penalti jika target muncul di ranking yang lebih rendah.

In [ ]:
def mask_seen_courses(scores, X_history):
    """Mask course yang sudah pernah diambil user agar tidak direkomendasikan ulang."""
    scores = scores.copy()

    for i in range(scores.shape[0]):
        seen_courses = [int(x) for x in X_history[i] if int(x) != 0]

        # Index 0 adalah padding, bukan course valid.
        scores[i, 0] = -np.inf

        for c in seen_courses:
            if c < scores.shape[1]:
                scores[i, c] = -np.inf

    return scores


def get_topk_indices_masked(scores, X_history, top_k=10):
    """Mengambil indeks course Top-K setelah course yang sudah pernah diambil di-mask."""
    scores_masked = mask_seen_courses(scores, X_history)
    actual_k = min(top_k, scores_masked.shape[1] - 1)

    top_indices = np.argsort(scores_masked, axis=1)[:, -actual_k:][:, ::-1]
    return top_indices


def evaluate_topk_ranking_masked(scores, X_history, y_true, k_values=(1, 5, 10)):
    """
    Evaluasi ranking Top-K untuk next-course recommendation.

    Metrik:
    - Hit@K / Recall@K: 1 jika ground-truth course muncul di Top-K, selain itu 0.
    - MRR@10: rata-rata reciprocal rank ground-truth pada Top-10.
    - NDCG@10: rata-rata discounted gain ground-truth pada Top-10.

    Catatan:
    Karena tiap sample hanya memiliki satu target course, Hit@K = Recall@K.
    """
    y_true = np.asarray(y_true).astype(int).reshape(-1)

    max_k = max(max(k_values), 10)
    top_indices = get_topk_indices_masked(scores, X_history, top_k=max_k)

    metrics = {}

    for k in k_values:
        actual_k = min(k, top_indices.shape[1])
        hits = np.array([
            y_true[i] in top_indices[i, :actual_k]
            for i in range(len(y_true))
        ], dtype=float)

        metrics[f"Hit@{k}"] = hits.mean()
        metrics[f"Recall@{k}"] = hits.mean()

    reciprocal_ranks = []
    ndcg_scores = []

    for i in range(len(y_true)):
        recommendations = top_indices[i, :10].tolist()
        ground_truth = int(y_true[i])

        if ground_truth in recommendations:
            rank = recommendations.index(ground_truth) + 1
            reciprocal_ranks.append(1 / rank)
            ndcg_scores.append(1 / np.log2(rank + 1))
        else:
            reciprocal_ranks.append(0)
            ndcg_scores.append(0)

    metrics["MRR@10"] = np.mean(reciprocal_ranks)
    metrics["NDCG@10"] = np.mean(ndcg_scores)

    metrics_df = pd.DataFrame({
        "metric": list(metrics.keys()),
        "score": list(metrics.values())
    })
    metrics_df["percentage"] = metrics_df["score"] * 100

    top10_indices = top_indices[:, :10]

    return metrics, metrics_df, top10_indices


## 24. Evaluasi Softmax Ranking Top-K pada Data Test

**Penjelasan:** Bagian ini mengevaluasi model pada data test menggunakan output softmax. Model menghasilkan skor probabilitas untuk semua course, kemudian course yang sudah pernah diambil user di-mask agar tidak muncul lagi pada rekomendasi.

**Yang perlu diperhatikan:** Evaluasi recommender system lebih tepat menggunakan metrik ranking, bukan metrik klasifikasi umum. Output utama bagian ini adalah **Hit@1, Hit@5, Hit@10, Recall@1, Recall@5, Recall@10, MRR@10, dan NDCG@10**.

In [ ]:
test_loss = model.evaluate(
    [
        X_course_test,
        X_competency_test,
        X_level_test,
        X_course_has_certificate_test,
        X_certificate_count_test
    ],
    y_test,
    verbose=0
)

softmax_scores_test = model.predict(
    [
        X_course_test,
        X_competency_test,
        X_level_test,
        X_course_has_certificate_test,
        X_certificate_count_test
    ],
    verbose=0
)

softmax_metrics, softmax_metrics_df, softmax_top10_indices = evaluate_topk_ranking_masked(
    softmax_scores_test,
    X_course_test,
    y_test,
    k_values=(1, 5, 10)
)

print("Test Loss:", test_loss)
print()
print("=== Masked Softmax Top-K Ranking Metrics ===")
display(
    softmax_metrics_df.assign(
        score=softmax_metrics_df["score"].round(4),
        percentage=softmax_metrics_df["percentage"].round(2)
    )
)


## 25. Evaluasi Similarity Ranking Top-K pada Data Test

**Penjelasan:** Bagian ini mengevaluasi rekomendasi berbasis similarity antara representasi user dan vektor course. Sama seperti evaluasi softmax, course yang sudah pernah diambil user di-mask terlebih dahulu sebelum dihitung metrik ranking Top-K.

**Yang perlu diperhatikan:** Evaluasi similarity juga dilaporkan dengan **Hit@1, Hit@5, Hit@10, Recall@1, Recall@5, Recall@10, MRR@10, dan NDCG@10**, bukan dengan metrik klasifikasi umum.

In [ ]:
decoder_vector_model = Model(
    inputs=model.inputs,
    outputs=model.get_layer("decoder_representation").output
)

course_output_weights = model.get_layer("course_prediction").get_weights()[0]
course_vectors = course_output_weights.T


def normalize_vector(matrix):
    norm = np.linalg.norm(matrix, axis=1, keepdims=True)
    norm = np.where(norm == 0, 1e-8, norm)
    return matrix / norm


def calculate_similarity(user_vectors, course_vectors):
    user_norm = normalize_vector(user_vectors)
    course_norm = normalize_vector(course_vectors)

    cosine = np.dot(user_norm, course_norm.T)

    similarity_0_1 = (cosine + 1) / 2
    similarity_0_1[:, 0] = -np.inf

    return similarity_0_1


user_vectors_test = decoder_vector_model.predict(
    [
        X_course_test,
        X_competency_test,
        X_level_test,
        X_course_has_certificate_test,
        X_certificate_count_test
    ],
    verbose=0
)

similarity_scores_test = calculate_similarity(user_vectors_test, course_vectors)

similarity_metrics, similarity_metrics_df, similarity_top10_indices = evaluate_topk_ranking_masked(
    similarity_scores_test,
    X_course_test,
    y_test,
    k_values=(1, 5, 10)
)

print("=== Masked Similarity Top-K Ranking Metrics ===")
display(
    similarity_metrics_df.assign(
        score=similarity_metrics_df["score"].round(4),
        percentage=similarity_metrics_df["percentage"].round(2)
    )
)


## 26. Top-K Performance Curve

**Penjelasan:**  
Bagian ini menambahkan **Top-K Performance Curve** untuk melihat perubahan performa model ketika nilai K dinaikkan secara bertahap. Jika evaluasi utama hanya menampilkan Hit@1, Hit@5, Hit@10, MRR@10, dan NDCG@10, kurva Top-K membantu memperlihatkan tren performa dari K=1 sampai K=10.

Kurva ini penting untuk paper karena dapat menunjukkan apakah performa rekomendasi meningkat secara stabil ketika daftar rekomendasi diperluas. Dalam sistem rekomendasi course, nilai K merepresentasikan jumlah course yang ditampilkan kepada karyawan. Karena target utama sistem ini adalah rekomendasi Top-10, kurva dibuat dari **K=1 sampai K=10**.

Metrik yang divisualisasikan:
- **Hit@K / Recall@K**: apakah course ground truth muncul dalam Top-K rekomendasi.
- **MRR@K**: memperhatikan seberapa awal posisi course ground truth dalam Top-K.
- **NDCG@K**: memberi bobot lebih tinggi jika ground truth muncul pada ranking yang lebih atas.

Catatan:
Karena setiap sampel test hanya memiliki satu ground-truth next course, maka **Hit@K = Recall@K**.

In [ ]:
# ============================================================
# TOP-K PERFORMANCE CURVE
# ============================================================

import matplotlib.pyplot as plt


def calculate_topk_performance_curve(
    scores,
    X_history,
    y_true,
    k_values=range(1, 11),
    ranking_method="Model"
):
    """
    Menghitung performa ranking untuk beberapa nilai K.

    Output utama:
    - Hit@K / Recall@K
    - MRR@K
    - NDCG@K

    Catatan:
    Karena setiap sample hanya memiliki satu target course,
    Hit@K dan Recall@K bernilai sama.
    """
    y_true = np.asarray(y_true).astype(int).reshape(-1)

    candidate_count = max(int(scores.shape[1]) - 1, 1)
    clean_k_values = sorted({
        min(int(k), candidate_count)
        for k in k_values
        if int(k) >= 1
    })

    if len(clean_k_values) == 0:
        return pd.DataFrame(columns=[
            "ranking_method",
            "K",
            "Hit@K",
            "Recall@K",
            "MRR@K",
            "NDCG@K",
            "Hit@K (%)",
            "Recall@K (%)",
            "MRR@K (%)",
            "NDCG@K (%)"
        ])

    max_k = max(clean_k_values)
    top_indices = get_topk_indices_masked(scores, X_history, top_k=max_k)

    curve_rows = []

    for k in clean_k_values:
        actual_k = min(k, top_indices.shape[1])

        hit_scores = []
        reciprocal_ranks = []
        ndcg_scores = []

        for i in range(len(y_true)):
            ground_truth = int(y_true[i])
            recommendations = top_indices[i, :actual_k].astype(int).tolist()

            if ground_truth in recommendations:
                rank = recommendations.index(ground_truth) + 1

                hit_scores.append(1.0)
                reciprocal_ranks.append(1.0 / rank)
                ndcg_scores.append(1.0 / np.log2(rank + 1))
            else:
                hit_scores.append(0.0)
                reciprocal_ranks.append(0.0)
                ndcg_scores.append(0.0)

        hit_at_k = float(np.mean(hit_scores)) if len(hit_scores) > 0 else 0.0
        mrr_at_k = float(np.mean(reciprocal_ranks)) if len(reciprocal_ranks) > 0 else 0.0
        ndcg_at_k = float(np.mean(ndcg_scores)) if len(ndcg_scores) > 0 else 0.0

        curve_rows.append({
            "ranking_method": ranking_method,
            "K": actual_k,
            "Hit@K": hit_at_k,
            "Recall@K": hit_at_k,
            "MRR@K": mrr_at_k,
            "NDCG@K": ndcg_at_k,
            "Hit@K (%)": hit_at_k * 100,
            "Recall@K (%)": hit_at_k * 100,
            "MRR@K (%)": mrr_at_k * 100,
            "NDCG@K (%)": ndcg_at_k * 100
        })

    return pd.DataFrame(curve_rows)


TOP_K_CURVE_VALUES = list(range(1, 11))

softmax_topk_curve_df = calculate_topk_performance_curve(
    softmax_scores_test,
    X_course_test,
    y_test,
    k_values=TOP_K_CURVE_VALUES,
    ranking_method="Softmax Ranking"
)

similarity_topk_curve_df = calculate_topk_performance_curve(
    similarity_scores_test,
    X_course_test,
    y_test,
    k_values=TOP_K_CURVE_VALUES,
    ranking_method="Similarity Ranking"
)

topk_performance_curve_df = pd.concat(
    [softmax_topk_curve_df, similarity_topk_curve_df],
    ignore_index=True
)

topk_curve_display = topk_performance_curve_df.copy()
percentage_columns = ["Hit@K (%)", "Recall@K (%)", "MRR@K (%)", "NDCG@K (%)"]
score_columns = ["Hit@K", "Recall@K", "MRR@K", "NDCG@K"]

topk_curve_display[percentage_columns] = topk_curve_display[percentage_columns].round(2)
topk_curve_display[score_columns] = topk_curve_display[score_columns].round(4)

print("=== Top-K Performance Curve Table ===")
display(topk_curve_display)


# ------------------------------------------------------------
# Visualisasi 1: Hit@K / Recall@K Curve
# ------------------------------------------------------------
plt.figure(figsize=(9, 5))

for method_name, group_df in topk_performance_curve_df.groupby("ranking_method"):
    group_df = group_df.sort_values("K")
    plt.plot(
        group_df["K"],
        group_df["Hit@K"],
        marker="o",
        label=method_name
    )

plt.title("Top-K Performance Curve - Hit@K / Recall@K")
plt.xlabel("K")
plt.ylabel("Hit@K / Recall@K")
plt.xticks(TOP_K_CURVE_VALUES)
plt.ylim(0, 1)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Visualisasi 2: MRR@K Curve
# ------------------------------------------------------------
plt.figure(figsize=(9, 5))

for method_name, group_df in topk_performance_curve_df.groupby("ranking_method"):
    group_df = group_df.sort_values("K")
    plt.plot(
        group_df["K"],
        group_df["MRR@K"],
        marker="o",
        label=method_name
    )

plt.title("Top-K Performance Curve - MRR@K")
plt.xlabel("K")
plt.ylabel("MRR@K")
plt.xticks(TOP_K_CURVE_VALUES)
plt.ylim(0, 1)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Visualisasi 3: NDCG@K Curve
# ------------------------------------------------------------
plt.figure(figsize=(9, 5))

for method_name, group_df in topk_performance_curve_df.groupby("ranking_method"):
    group_df = group_df.sort_values("K")
    plt.plot(
        group_df["K"],
        group_df["NDCG@K"],
        marker="o",
        label=method_name
    )

plt.title("Top-K Performance Curve - NDCG@K")
plt.xlabel("K")
plt.ylabel("NDCG@K")
plt.xticks(TOP_K_CURVE_VALUES)
plt.ylim(0, 1)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


### Interpretasi Kurva Performa Top-K

Top-K Performance Curve digunakan untuk menganalisis perubahan performa sistem rekomendasi ketika jumlah rekomendasi yang ditampilkan kepada pengguna dinaikkan dari K=1 sampai K=10. Pada konteks next-course recommendation, kurva ini membantu menunjukkan apakah model mampu menempatkan course target pada posisi ranking atas atau baru berhasil menemukan target ketika daftar rekomendasi diperluas.

Nilai **Hit@K/Recall@K** yang meningkat ketika K bertambah menunjukkan bahwa peluang ground-truth course muncul dalam daftar rekomendasi menjadi lebih besar. Namun, interpretasi tidak boleh hanya bergantung pada Hit@K, karena Hit@K tidak membedakan apakah course target muncul di ranking pertama atau ranking kesepuluh. Oleh karena itu, **MRR@K** dan **NDCG@K** juga digunakan untuk mengevaluasi kualitas posisi ranking. Semakin tinggi MRR@K dan NDCG@K, semakin baik kemampuan model dalam menempatkan course yang relevan pada urutan rekomendasi teratas.

Dalam penelitian ini, kurva Top-K juga digunakan untuk membandingkan dua pendekatan ranking, yaitu **Softmax Ranking** dan **Similarity Ranking**. Metode yang lebih stabil dan memiliki nilai Hit@K, MRR@K, serta NDCG@K lebih tinggi pada rentang K=1 sampai K=10 dapat dipertimbangkan sebagai pendekatan ranking utama untuk sistem rekomendasi course.

## 27. Fungsi Decode Course

**Penjelasan:** Bagian ini membuat fungsi untuk mengubah `course_encoded` menjadi nama course asli. Fungsi `decode_course_list()` juga disediakan agar daftar rekomendasi hasil encoding bisa ditampilkan dalam bentuk nama course.

**Yang perlu diperhatikan:** Tahap ini penting karena output model masih berupa ID numerik, sementara pengguna membutuhkan nama course yang mudah dibaca.

In [ ]:
def decode_course(encoded_id):
    encoded_id = int(encoded_id)

    if encoded_id == 0:
        return "PAD"

    canonical_id = idx_to_course.get(encoded_id)

    if canonical_id is None:
        return "Unknown Course"

    info = course_catalog[course_catalog["course_encoded"] == encoded_id]

    if not info.empty:
        return info.iloc[0]["course_name"]

    return str(canonical_id)


def decode_course_list(encoded_list):
    return [
        decode_course(x)
        for x in encoded_list
        if int(x) != 0
    ]

## 28. Coverage@10

**Penjelasan:** Bagian ini menambahkan metrik **Coverage@10** untuk mengevaluasi seberapa luas katalog course yang benar-benar muncul dalam daftar Top-10 rekomendasi. Metrik ini melengkapi Hit@10/Recall@10, MRR@10, NDCG@10, dan distribusi target test.

**Formula:**

`Coverage@10 = jumlah unique course yang muncul dalam Top-10 rekomendasi / jumlah total course kandidat dalam katalog training`

**Mengapa penting:** Hit@10 yang tinggi belum tentu berarti sistem rekomendasi sudah beragam. Jika rekomendasi hanya berputar pada beberapa course populer, Hit@10 bisa tetap tinggi tetapi coverage rendah. Karena itu, Coverage@10 digunakan untuk melihat apakah model mampu mengekspos banyak course berbeda atau terlalu terkonsentrasi pada course tertentu.

**Interpretasi:**

- Coverage@10 tinggi berarti rekomendasi menyebar ke banyak course dalam katalog.
- Coverage@10 rendah berarti rekomendasi cenderung terkonsentrasi pada sedikit course populer.
- Coverage@10 perlu dibaca bersama Hit@10 dan distribusi target test, bukan sebagai pengganti metrik ranking.

In [ ]:
def calculate_coverage_at_k(topk_indices, catalog_df, k=10, model_name="Model"):
    """
    Menghitung catalog Coverage@K untuk recommender system.

    Coverage@K = jumlah unique course valid yang muncul pada Top-K rekomendasi
                 dibagi jumlah total course kandidat valid pada catalog training.

    Catatan:
    - Index 0 tidak dihitung karena merupakan PAD, bukan course valid.
    - Catalog yang digunakan adalah course_catalog berbasis train_core, sehingga tetap train-only.
    """
    topk_array = np.asarray(topk_indices).astype(int)[:, :k]

    catalog_course_ids = set(
        catalog_df["course_encoded"]
        .dropna()
        .astype(int)
        .tolist()
    )
    catalog_course_ids.discard(0)

    recommended_courses = [
        int(course_id)
        for course_id in topk_array.flatten().tolist()
        if int(course_id) != 0
    ]

    unique_recommended_courses = set(recommended_courses)
    unique_recommended_valid = unique_recommended_courses.intersection(catalog_course_ids)
    uncovered_courses = catalog_course_ids.difference(unique_recommended_valid)

    total_candidate_courses = len(catalog_course_ids)
    unique_recommended_count = len(unique_recommended_valid)
    coverage_score = (
        unique_recommended_count / total_candidate_courses
        if total_candidate_courses > 0
        else 0
    )

    recommendation_counts = (
        pd.Series(recommended_courses, name="course_encoded")
        .value_counts()
        .rename_axis("course_encoded")
        .reset_index(name="recommendation_count_at_k")
    )

    recommendation_distribution = recommendation_counts.merge(
        catalog_df[[
            "course_encoded",
            "canonical_course_id",
            "course_name",
            "kompetensi",
            "level",
            "course_has_certificate_train",
            "certificate_count_train"
        ]].drop_duplicates("course_encoded"),
        on="course_encoded",
        how="left"
    )

    total_recommendation_slots = int(topk_array.size)
    valid_recommendation_slots = int(len(recommended_courses))

    if valid_recommendation_slots > 0:
        recommendation_distribution["recommendation_percentage_at_k"] = (
            recommendation_distribution["recommendation_count_at_k"] / valid_recommendation_slots * 100
        )
        top_1_recommendation_share = recommendation_distribution.head(1)["recommendation_count_at_k"].sum() / valid_recommendation_slots
        top_5_recommendation_share = recommendation_distribution.head(5)["recommendation_count_at_k"].sum() / valid_recommendation_slots
        top_10_recommendation_share = recommendation_distribution.head(10)["recommendation_count_at_k"].sum() / valid_recommendation_slots
    else:
        recommendation_distribution["recommendation_percentage_at_k"] = 0
        top_1_recommendation_share = 0
        top_5_recommendation_share = 0
        top_10_recommendation_share = 0

    recommendation_distribution["model"] = model_name
    recommendation_distribution["k"] = k

    uncovered_df = pd.DataFrame({"course_encoded": sorted(list(uncovered_courses))})
    if not uncovered_df.empty:
        uncovered_df = uncovered_df.merge(
            catalog_df[[
                "course_encoded",
                "canonical_course_id",
                "course_name",
                "kompetensi",
                "level"
            ]].drop_duplicates("course_encoded"),
            on="course_encoded",
            how="left"
        )
    else:
        uncovered_df = pd.DataFrame(columns=[
            "course_encoded",
            "canonical_course_id",
            "course_name",
            "kompetensi",
            "level"
        ])

    summary = {
        "model": model_name,
        "k": k,
        "total_candidate_courses": total_candidate_courses,
        "unique_recommended_courses": unique_recommended_count,
        "uncovered_courses": len(uncovered_courses),
        "coverage_at_k": coverage_score,
        "coverage_at_k_percent": coverage_score * 100,
        "total_recommendation_slots": total_recommendation_slots,
        "valid_recommendation_slots": valid_recommendation_slots,
        "top_1_recommendation_share_percent": top_1_recommendation_share * 100,
        "top_5_recommendation_share_percent": top_5_recommendation_share * 100,
        "top_10_recommendation_share_percent": top_10_recommendation_share * 100
    }

    ordered_distribution_columns = [
        "model",
        "k",
        "course_encoded",
        "canonical_course_id",
        "course_name",
        "kompetensi",
        "level",
        "course_has_certificate_train",
        "certificate_count_train",
        "recommendation_count_at_k",
        "recommendation_percentage_at_k"
    ]

    recommendation_distribution = recommendation_distribution[ordered_distribution_columns]

    return summary, recommendation_distribution, uncovered_df


def plot_top_recommended_courses(distribution_df, title, top_n=15):
    """Menampilkan grafik course yang paling sering muncul pada Top-K rekomendasi."""
    if distribution_df.empty:
        print("Distribusi rekomendasi kosong, grafik tidak dapat ditampilkan.")
        return

    plot_df = distribution_df.head(top_n).copy()
    plot_df = plot_df.sort_values("recommendation_count_at_k", ascending=True)

    plt.figure(figsize=(10, 6))
    plt.barh(plot_df["course_name"], plot_df["recommendation_count_at_k"])
    plt.xlabel("Jumlah Kemunculan dalam Top-10 Rekomendasi")
    plt.ylabel("Course")
    plt.title(title)
    plt.tight_layout()
    plt.show()


softmax_coverage_summary, softmax_recommendation_distribution, softmax_uncovered_courses = calculate_coverage_at_k(
    softmax_top10_indices,
    course_catalog,
    k=10,
    model_name="Softmax Ranking"
)

similarity_coverage_summary, similarity_recommendation_distribution, similarity_uncovered_courses = calculate_coverage_at_k(
    similarity_top10_indices,
    course_catalog,
    k=10,
    model_name="Similarity Ranking"
)

coverage_summary_df = pd.DataFrame([
    softmax_coverage_summary,
    similarity_coverage_summary
])

coverage_summary_display = coverage_summary_df.copy()
percentage_columns = [
    "coverage_at_k_percent",
    "top_1_recommendation_share_percent",
    "top_5_recommendation_share_percent",
    "top_10_recommendation_share_percent"
]
coverage_summary_display[percentage_columns] = coverage_summary_display[percentage_columns].round(2)
coverage_summary_display["coverage_at_k"] = coverage_summary_display["coverage_at_k"].round(4)

print("=== Coverage@10 Summary ===")
display(coverage_summary_display)

print("\n=== Top-15 Course yang Paling Sering Direkomendasikan — Softmax Ranking ===")
display(
    softmax_recommendation_distribution.head(15).assign(
        recommendation_percentage_at_k=lambda df: df["recommendation_percentage_at_k"].round(2)
    )
)

print("\n=== Top-15 Course yang Paling Sering Direkomendasikan — Similarity Ranking ===")
display(
    similarity_recommendation_distribution.head(15).assign(
        recommendation_percentage_at_k=lambda df: df["recommendation_percentage_at_k"].round(2)
    )
)

plot_top_recommended_courses(
    softmax_recommendation_distribution,
    "Top-15 Course yang Paling Sering Muncul dalam Top-10 Rekomendasi — Softmax Ranking",
    top_n=15
)

plot_top_recommended_courses(
    similarity_recommendation_distribution,
    "Top-15 Course yang Paling Sering Muncul dalam Top-10 Rekomendasi — Similarity Ranking",
    top_n=15
)


### Analisis Coverage@10

Coverage@10 digunakan untuk mengukur proporsi course dalam katalog kandidat yang berhasil terekspos setidaknya satu kali pada daftar 10 rekomendasi teratas. Berbeda dari Hit@10/Recall@10 yang mengukur apakah target aktual muncul dalam rekomendasi, Coverage@10 mengevaluasi keluasan item yang direkomendasikan oleh sistem.

Metrik ini penting karena sistem rekomendasi dapat memperoleh Hit@10 tinggi apabila target test banyak didominasi oleh course populer. Namun, tanpa coverage, belum dapat diketahui apakah sistem hanya merekomendasikan sedikit course populer atau mampu menyebarkan rekomendasi ke berbagai course yang tersedia. Oleh karena itu, Coverage@10 dalam penelitian ini digunakan sebagai metrik pelengkap untuk menilai keberagaman katalog rekomendasi.

Secara metodologis, Coverage@10 dihitung menggunakan `course_catalog` yang dibentuk dari data training (`train_core`). Dengan demikian, perhitungan coverage tetap konsisten dengan prinsip train-only transformation dan tidak mengambil informasi dari target validation maupun target test. Nilai Coverage@10 yang lebih tinggi menunjukkan bahwa model mengekspos lebih banyak course berbeda, sedangkan nilai yang rendah menunjukkan adanya kecenderungan rekomendasi yang terkonsentrasi pada course tertentu.

## 29. Ablation Study Fitur

**Penjelasan:**  
Bagian ini menambahkan **ablation study** untuk menguji kontribusi setiap kelompok fitur terhadap performa sistem rekomendasi course. Ablation study dilakukan dengan melatih beberapa variasi model menggunakan kombinasi fitur yang berbeda, mulai dari model dasar yang hanya memakai riwayat course sampai model penuh yang memakai seluruh fitur.

**Tujuan ablation study:**

1. Mengetahui apakah fitur tambahan seperti `kompetensi`, `level`, `course_has_certificate_train`, dan `certificate_count_scaled` benar-benar membantu performa model.
2. Membandingkan model dasar dengan model penuh secara lebih objektif.
3. Memberikan bukti eksperimen tambahan agar penelitian lebih kuat untuk kebutuhan paper.
4. Menjaga evaluasi tetap leakage-safe karena seluruh fitur tetap berasal dari pipeline train-only encoding dan train-only scaling.

**Catatan penting:**  
Seluruh konfigurasi ablation menggunakan Encoder–Decoder LSTM dengan attention layer. Dengan demikian, ablation study tetap fokus membandingkan kontribusi fitur, sedangkan arsitektur model dijaga konsisten dengan model utama.

In [ ]:
# ============================================================
# ABLATION STUDY FITUR
# ============================================================

def make_safe_layer_name(text):
    """Membuat nama layer aman untuk Keras."""
    safe = "".join(ch.lower() if ch.isalnum() else "_" for ch in text)
    safe = "_".join([part for part in safe.split("_") if part])
    return safe


ablation_experiments = [
    {
        "experiment": "A1 - Course History Only",
        "use_competency": False,
        "use_level": False,
        "use_certificate_flag": False,
        "use_certificate_count": False
    },
    {
        "experiment": "A2 - Course + Competency",
        "use_competency": True,
        "use_level": False,
        "use_certificate_flag": False,
        "use_certificate_count": False
    },
    {
        "experiment": "A3 - Course + Competency + Level",
        "use_competency": True,
        "use_level": True,
        "use_certificate_flag": False,
        "use_certificate_count": False
    },
    {
        "experiment": "A4 - Course + Competency + Level + Certificate Flag",
        "use_competency": True,
        "use_level": True,
        "use_certificate_flag": True,
        "use_certificate_count": False
    },
    {
        "experiment": "A5 - Full Feature Model",
        "use_competency": True,
        "use_level": True,
        "use_certificate_flag": True,
        "use_certificate_count": True
    }
]


def build_ablation_lstm_model(config):
    """
    Membuat model Encoder–Decoder LSTM dengan attention untuk ablation study.

    Course history selalu digunakan sebagai fitur utama.
    Fitur lain ditambahkan sesuai konfigurasi eksperimen.
    Arsitektur tetap menggunakan attention agar perbandingan antar eksperimen
    hanya dipengaruhi oleh perbedaan fitur.
    """
    prefix = make_safe_layer_name(config["experiment"])

    input_layers = []
    feature_layers = []

    # 1. Course input wajib digunakan
    course_input_ablation = Input(
        shape=(max_sequence_length,),
        name=f"{prefix}_course_input"
    )

    course_embedding_ablation = Embedding(
        input_dim=num_courses,
        output_dim=course_embedding_dim,
        name=f"{prefix}_course_embedding"
    )(course_input_ablation)

    input_layers.append(course_input_ablation)
    feature_layers.append(course_embedding_ablation)

    # 2. Competency input
    if config["use_competency"]:
        competency_input_ablation = Input(
            shape=(max_sequence_length,),
            name=f"{prefix}_competency_input"
        )

        competency_embedding_ablation = Embedding(
            input_dim=num_competencies,
            output_dim=competency_embedding_dim,
            name=f"{prefix}_competency_embedding"
        )(competency_input_ablation)

        input_layers.append(competency_input_ablation)
        feature_layers.append(competency_embedding_ablation)

    # 3. Level input
    if config["use_level"]:
        level_input_ablation = Input(
            shape=(max_sequence_length,),
            name=f"{prefix}_level_input"
        )

        level_embedding_ablation = Embedding(
            input_dim=num_levels,
            output_dim=level_embedding_dim,
            name=f"{prefix}_level_embedding"
        )(level_input_ablation)

        input_layers.append(level_input_ablation)
        feature_layers.append(level_embedding_ablation)

    # 4. Certificate flag input
    if config["use_certificate_flag"]:
        course_has_certificate_input_ablation = Input(
            shape=(max_sequence_length, 1),
            name=f"{prefix}_course_has_certificate_input"
        )

        input_layers.append(course_has_certificate_input_ablation)
        feature_layers.append(course_has_certificate_input_ablation)

    # 5. Certificate count input
    if config["use_certificate_count"]:
        certificate_count_input_ablation = Input(
            shape=(max_sequence_length, 1),
            name=f"{prefix}_certificate_count_input"
        )

        input_layers.append(certificate_count_input_ablation)
        feature_layers.append(certificate_count_input_ablation)

    # Gabungkan fitur
    if len(feature_layers) > 1:
        combined_input_ablation = Concatenate(
            name=f"{prefix}_combined_encoder_input"
        )(feature_layers)
    else:
        combined_input_ablation = feature_layers[0]

    # Encoder LSTM dengan return_sequences=True agar bisa diproses oleh attention layer
    encoder_sequence_output_ablation, state_h_ablation, state_c_ablation = LSTM(
        latent_dim,
        return_sequences=True,
        return_state=True,
        name=f"{prefix}_encoder_lstm"
    )(combined_input_ablation)

    attention_context_ablation, _ = TemporalAttention(
        name=f"{prefix}_temporal_attention"
    )([
        encoder_sequence_output_ablation,
        course_input_ablation
    ])

    attention_enhanced_state_ablation = Concatenate(
        name=f"{prefix}_attention_state_concatenate"
    )([
        state_h_ablation,
        attention_context_ablation
    ])

    attention_enhanced_state_ablation = Dense(
        latent_dim,
        activation="tanh",
        name=f"{prefix}_attention_enhanced_state"
    )(attention_enhanced_state_ablation)

    # Decoder LSTM
    decoder_input_ablation = RepeatVector(
        1,
        name=f"{prefix}_decoder_repeat"
    )(attention_enhanced_state_ablation)

    decoder_output_ablation = LSTM(
        latent_dim,
        return_sequences=False,
        name=f"{prefix}_decoder_lstm"
    )(
        decoder_input_ablation,
        initial_state=[attention_enhanced_state_ablation, state_c_ablation]
    )

    decoder_representation_ablation = Dropout(
        0.3,
        name=f"{prefix}_decoder_representation"
    )(decoder_output_ablation)

    course_prediction_ablation = Dense(
        num_courses,
        activation="softmax",
        name=f"{prefix}_course_prediction"
    )(decoder_representation_ablation)

    ablation_model = Model(
        inputs=input_layers,
        outputs=course_prediction_ablation,
        name=f"{prefix}_model"
    )

    ablation_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy"
    )

    return ablation_model, prefix


def get_ablation_inputs(config, split_name):
    """
    Mengambil input array sesuai fitur yang digunakan pada eksperimen ablation.

    split_name:
    - "train"
    - "val"
    - "test"
    """
    if split_name == "train":
        data = {
            "course": X_course_train,
            "competency": X_competency_train,
            "level": X_level_train,
            "certificate_flag": X_course_has_certificate_train,
            "certificate_count": X_certificate_count_train
        }
    elif split_name == "val":
        data = {
            "course": X_course_val,
            "competency": X_competency_val,
            "level": X_level_val,
            "certificate_flag": X_course_has_certificate_val,
            "certificate_count": X_certificate_count_val
        }
    elif split_name == "test":
        data = {
            "course": X_course_test,
            "competency": X_competency_test,
            "level": X_level_test,
            "certificate_flag": X_course_has_certificate_test,
            "certificate_count": X_certificate_count_test
        }
    else:
        raise ValueError("split_name harus bernilai 'train', 'val', atau 'test'.")

    inputs = [data["course"]]

    if config["use_competency"]:
        inputs.append(data["competency"])

    if config["use_level"]:
        inputs.append(data["level"])

    if config["use_certificate_flag"]:
        inputs.append(data["certificate_flag"])

    if config["use_certificate_count"]:
        inputs.append(data["certificate_count"])

    return inputs


def evaluate_ablation_model(ablation_model, prefix, config):
    """
    Mengevaluasi model ablation menggunakan:
    - Softmax ranking
    - Similarity ranking
    - Hit@1, Hit@5, Hit@10
    - Recall@1, Recall@5, Recall@10
    - MRR@10
    - NDCG@10
    - Coverage@10
    """
    test_inputs = get_ablation_inputs(config, split_name="test")

    # Evaluasi softmax ranking
    softmax_scores_ablation = ablation_model.predict(
        test_inputs,
        verbose=0
    )

    softmax_metrics_ablation, _, softmax_top10_ablation = evaluate_topk_ranking_masked(
        softmax_scores_ablation,
        X_course_test,
        y_test,
        k_values=(1, 5, 10)
    )

    softmax_coverage_ablation, _, _ = calculate_coverage_at_k(
        softmax_top10_ablation,
        course_catalog,
        k=10,
        model_name=f"{config['experiment']} - Softmax Ranking"
    )

    # Evaluasi similarity ranking
    decoder_vector_model_ablation = Model(
        inputs=ablation_model.inputs,
        outputs=ablation_model.get_layer(
            f"{prefix}_decoder_representation"
        ).output
    )

    course_output_weights_ablation = ablation_model.get_layer(
        f"{prefix}_course_prediction"
    ).get_weights()[0]

    course_vectors_ablation = course_output_weights_ablation.T

    user_vectors_test_ablation = decoder_vector_model_ablation.predict(
        test_inputs,
        verbose=0
    )

    similarity_scores_ablation = calculate_similarity(
        user_vectors_test_ablation,
        course_vectors_ablation
    )

    similarity_metrics_ablation, _, similarity_top10_ablation = evaluate_topk_ranking_masked(
        similarity_scores_ablation,
        X_course_test,
        y_test,
        k_values=(1, 5, 10)
    )

    similarity_coverage_ablation, _, _ = calculate_coverage_at_k(
        similarity_top10_ablation,
        course_catalog,
        k=10,
        model_name=f"{config['experiment']} - Similarity Ranking"
    )

    rows = []

    rows.append({
        "experiment": config["experiment"],
        "ranking_method": "Softmax Ranking",
        "architecture": "Encoder-Decoder LSTM + Attention",
        "feature_course_history": "Yes",
        "feature_competency": "Yes" if config["use_competency"] else "No",
        "feature_level": "Yes" if config["use_level"] else "No",
        "feature_certificate_flag": "Yes" if config["use_certificate_flag"] else "No",
        "feature_certificate_count": "Yes" if config["use_certificate_count"] else "No",
        "Hit@1": softmax_metrics_ablation["Hit@1"],
        "Hit@5": softmax_metrics_ablation["Hit@5"],
        "Hit@10": softmax_metrics_ablation["Hit@10"],
        "Recall@1": softmax_metrics_ablation["Recall@1"],
        "Recall@5": softmax_metrics_ablation["Recall@5"],
        "Recall@10": softmax_metrics_ablation["Recall@10"],
        "MRR@10": softmax_metrics_ablation["MRR@10"],
        "NDCG@10": softmax_metrics_ablation["NDCG@10"],
        "Coverage@10": softmax_coverage_ablation["coverage_at_k"]
    })

    rows.append({
        "experiment": config["experiment"],
        "ranking_method": "Similarity Ranking",
        "architecture": "Encoder-Decoder LSTM + Attention",
        "feature_course_history": "Yes",
        "feature_competency": "Yes" if config["use_competency"] else "No",
        "feature_level": "Yes" if config["use_level"] else "No",
        "feature_certificate_flag": "Yes" if config["use_certificate_flag"] else "No",
        "feature_certificate_count": "Yes" if config["use_certificate_count"] else "No",
        "Hit@1": similarity_metrics_ablation["Hit@1"],
        "Hit@5": similarity_metrics_ablation["Hit@5"],
        "Hit@10": similarity_metrics_ablation["Hit@10"],
        "Recall@1": similarity_metrics_ablation["Recall@1"],
        "Recall@5": similarity_metrics_ablation["Recall@5"],
        "Recall@10": similarity_metrics_ablation["Recall@10"],
        "MRR@10": similarity_metrics_ablation["MRR@10"],
        "NDCG@10": similarity_metrics_ablation["NDCG@10"],
        "Coverage@10": similarity_coverage_ablation["coverage_at_k"]
    })

    return rows


# ============================================================
# MENJALANKAN ABLATION STUDY
# ============================================================

ablation_results = []
ablation_histories = {}

# Mengikuti konfigurasi training model utama.
# Early stopping tetap aktif, sehingga training dapat berhenti sebelum 50 epoch.
ABLATION_EPOCHS = 50
ABLATION_BATCH_SIZE = 64

for experiment_id, config in enumerate(ablation_experiments, start=1):
    print("=" * 80)
    print("Menjalankan ablation study:", config["experiment"])
    print("=" * 80)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    ablation_model, prefix = build_ablation_lstm_model(config)

    train_inputs = get_ablation_inputs(config, split_name="train")
    val_inputs = get_ablation_inputs(config, split_name="val")

    ablation_early_stop = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=5,
        restore_best_weights=True
    )

    ablation_reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5
    )

    ablation_history = ablation_model.fit(
        train_inputs,
        y_train,
        validation_data=(val_inputs, y_val),
        epochs=ABLATION_EPOCHS,
        batch_size=ABLATION_BATCH_SIZE,
        callbacks=[ablation_early_stop, ablation_reduce_lr],
        shuffle=True,
        verbose=1
    )

    ablation_histories[config["experiment"]] = ablation_history.history

    experiment_rows = evaluate_ablation_model(
        ablation_model=ablation_model,
        prefix=prefix,
        config=config
    )

    ablation_results.extend(experiment_rows)


ablation_summary_df = pd.DataFrame(ablation_results)

metric_columns = [
    "Hit@1",
    "Hit@5",
    "Hit@10",
    "Recall@1",
    "Recall@5",
    "Recall@10",
    "MRR@10",
    "NDCG@10",
    "Coverage@10"
]

ablation_summary_display = ablation_summary_df.copy()

for col in metric_columns:
    ablation_summary_display[col] = (ablation_summary_display[col] * 100).round(2)

ablation_summary_display = ablation_summary_display.rename(columns={
    "Hit@1": "Hit@1 (%)",
    "Hit@5": "Hit@5 (%)",
    "Hit@10": "Hit@10 (%)",
    "Recall@1": "Recall@1 (%)",
    "Recall@5": "Recall@5 (%)",
    "Recall@10": "Recall@10 (%)",
    "MRR@10": "MRR@10 (%)",
    "NDCG@10": "NDCG@10 (%)",
    "Coverage@10": "Coverage@10 (%)"
})

print("=== Ablation Study Summary ===")
display(ablation_summary_display)

In [ ]:
# ============================================================
# ANALISIS DELTA TERHADAP BASELINE COURSE-ONLY
# ============================================================

PRIMARY_RANKING_METHOD = "Similarity Ranking"

primary_ablation_df = ablation_summary_df[
    ablation_summary_df["ranking_method"] == PRIMARY_RANKING_METHOD
].copy()

baseline_experiment = "A1 - Course History Only"

baseline_row = primary_ablation_df[
    primary_ablation_df["experiment"] == baseline_experiment
]

if baseline_row.empty:
    print("Baseline tidak ditemukan. Pastikan eksperimen A1 sudah berjalan.")
else:
    baseline_row = baseline_row.iloc[0]

    delta_df = primary_ablation_df.copy()

    for col in metric_columns:
        delta_df[f"Delta {col}"] = delta_df[col] - baseline_row[col]

    delta_display_columns = [
        "experiment",
        "ranking_method",
        "Hit@10",
        "MRR@10",
        "NDCG@10",
        "Coverage@10",
        "Delta Hit@10",
        "Delta MRR@10",
        "Delta NDCG@10",
        "Delta Coverage@10"
    ]

    ablation_delta_display = delta_df[delta_display_columns].copy()

    percentage_cols = [
        "Hit@10",
        "MRR@10",
        "NDCG@10",
        "Coverage@10",
        "Delta Hit@10",
        "Delta MRR@10",
        "Delta NDCG@10",
        "Delta Coverage@10"
    ]

    for col in percentage_cols:
        ablation_delta_display[col] = (ablation_delta_display[col] * 100).round(2)

    ablation_delta_display = ablation_delta_display.rename(columns={
        "Hit@10": "Hit@10 (%)",
        "MRR@10": "MRR@10 (%)",
        "NDCG@10": "NDCG@10 (%)",
        "Coverage@10": "Coverage@10 (%)",
        "Delta Hit@10": "Delta Hit@10 (%)",
        "Delta MRR@10": "Delta MRR@10 (%)",
        "Delta NDCG@10": "Delta NDCG@10 (%)",
        "Delta Coverage@10": "Delta Coverage@10 (%)"
    })

    print(f"=== Delta Ablation Study terhadap Baseline: {baseline_experiment} ===")
    display(ablation_delta_display)


In [ ]:
# ============================================================
# VISUALISASI ABLATION STUDY
# ============================================================

if "ablation_summary_df" not in globals():
    print("Ablation study belum dijalankan.")
else:
    plot_ablation_df = ablation_summary_df[
        ablation_summary_df["ranking_method"] == PRIMARY_RANKING_METHOD
    ].copy()

    plot_ablation_df = plot_ablation_df.set_index("experiment")

    plot_metrics = [
        "Hit@10",
        "MRR@10",
        "NDCG@10",
        "Coverage@10"
    ]

    plot_ablation_df[plot_metrics].plot(
        kind="bar",
        figsize=(12, 6)
    )

    plt.title(f"Ablation Study Feature Comparison - {PRIMARY_RANKING_METHOD}")
    plt.ylabel("Score")
    plt.xlabel("Experiment")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()


### 29.1 Interpretasi Ablation Study

Ablation study dilakukan untuk mengevaluasi kontribusi setiap kelompok fitur terhadap performa sistem rekomendasi course. Eksperimen dimulai dari model dasar yang hanya menggunakan riwayat course, kemudian fitur tambahan seperti kompetensi, level, indikator sertifikat, dan frekuensi sertifikat berbasis data training ditambahkan secara bertahap.

Seluruh konfigurasi ablation menggunakan arsitektur Encoder–Decoder LSTM dengan attention layer yang sama dengan model utama. Dengan demikian, perbandingan antar konfigurasi lebih adil karena perubahan performa terutama berasal dari perbedaan fitur input, bukan dari perbedaan arsitektur model.

Hasil ablation study digunakan untuk melihat apakah penambahan fitur metadata course mampu meningkatkan kualitas rekomendasi berdasarkan metrik Hit@K / Recall@K, MRR@10, NDCG@10, dan Coverage@10. Jika model dengan fitur lengkap menghasilkan nilai Hit@10, MRR@10, atau NDCG@10 yang lebih tinggi dibanding model dasar, maka fitur tambahan dapat dianggap memberikan kontribusi positif terhadap kemampuan model dalam memprediksi course berikutnya.

Seluruh eksperimen ablation tetap menggunakan skenario evaluasi temporal dan train-only transformation yang sama dengan model utama. Dengan demikian, hasil perbandingan antar konfigurasi fitur tetap konsisten dan leakage-safe.

## 30. Tabel Ground Truth Top-10

**Penjelasan:** Bagian ini membuat tabel evaluasi yang membandingkan ground truth course dengan Top-10 rekomendasi. Tabel menampilkan riwayat course terakhir user, course target yang benar, daftar rekomendasi, status **Hit@10**, dan ranking ground truth jika berhasil ditemukan.

**Yang perlu diperhatikan:** Output ini memudahkan analisis manual untuk melihat apakah rekomendasi model sudah menempatkan course target ke dalam Top-10 rekomendasi.

In [ ]:
def build_ground_truth_result(top_indices, metadata, y_true, X_history):
    results = []

    for i in range(len(y_true)):
        top_encoded = [int(x) for x in top_indices[i].tolist()]
        gt = int(y_true[i])

        hit_at_10 = gt in top_encoded

        if hit_at_10:
            rank_ground_truth = top_encoded.index(gt) + 1
        else:
            rank_ground_truth = None

        results.append({
            "user_id": metadata.iloc[i]["user_id"],
            "history_course": " | ".join(decode_course_list(metadata.iloc[i]["history_encoded"])[-5:]),
            "ground_truth_course": decode_course(gt),
            "top_10_recommendations": " | ".join(decode_course_list(top_encoded)),
            "hit_at_10": int(hit_at_10),
            "rank_ground_truth": rank_ground_truth
        })

    return pd.DataFrame(results)


softmax_ground_truth_result = build_ground_truth_result(
    softmax_top10_indices,
    test_metadata,
    y_test,
    X_course_test
)

similarity_ground_truth_result = build_ground_truth_result(
    similarity_top10_indices,
    test_metadata,
    y_test,
    X_course_test
)

print("=== SOFTMAX GROUND TRUTH RESULT ===")
display(softmax_ground_truth_result.head(10))

print()
print("=== SIMILARITY GROUND TRUTH RESULT ===")
display(similarity_ground_truth_result.head(10))


## 31. Error Analysis Rekomendasi Top-10

**Penjelasan:**  
Bagian ini menambahkan **Error Analysis** untuk melihat pola keberhasilan dan kegagalan model pada data test. Analisis ini memakai hasil Top-10 dari evaluasi Softmax Ranking dan Similarity Ranking.

Jika Top-K Performance Curve menjawab *seberapa baik performa model pada berbagai nilai K*, maka bagian ini menjawab *mengapa sebagian prediksi masih gagal atau berada pada ranking rendah*.

Analisis yang ditambahkan meliputi:

1. status **Hit@10** dan **Miss@10** per user;
2. ranking ground-truth course jika berhasil masuk Top-10;
3. kategori error: `Hit at Rank 1`, `Hit at Rank 2-5`, `Hit at Rank 6-10`, dan `Miss Top-10`;
4. pengaruh panjang riwayat user terhadap keberhasilan rekomendasi;
5. pengaruh frekuensi target course pada data training;
6. contoh kasus error untuk dianalisis secara kualitatif;
7. perbandingan kasus antara Softmax Ranking dan Similarity Ranking.

**Yang perlu diperhatikan:** Error analysis tidak mengganti metrik utama seperti Hit@K, MRR@K, NDCG@K, dan Coverage@10. Bagian ini berfungsi sebagai analisis pendukung agar paper tidak hanya melaporkan angka performa, tetapi juga menjelaskan pola kelemahan sistem rekomendasi.

In [ ]:
# ============================================================
# ERROR ANALYSIS REKOMENDASI TOP-10
# ============================================================


def categorize_history_length(length):
    """Mengelompokkan panjang riwayat user agar pola error lebih mudah dibaca."""
    length = int(length)

    if length <= 2:
        return "Short history (<=2)"
    elif length <= 5:
        return "Medium history (3-5)"
    else:
        return "Long history (>5)"


def add_target_frequency_bucket(error_df):
    """Membuat bucket frekuensi target berdasarkan distribusi target_train_frequency."""
    error_df = error_df.copy()

    if error_df.empty or "target_train_frequency" not in error_df.columns:
        error_df["target_frequency_bucket"] = "Unknown"
        return error_df

    freq_values = error_df["target_train_frequency"].fillna(0)

    if freq_values.nunique() <= 1:
        error_df["target_frequency_bucket"] = "Single-frequency target"
        return error_df

    q25 = freq_values.quantile(0.25)
    q75 = freq_values.quantile(0.75)

    def bucket(freq):
        if freq <= q25:
            return "Low-frequency target"
        elif freq <= q75:
            return "Medium-frequency target"
        else:
            return "High-frequency target"

    error_df["target_frequency_bucket"] = freq_values.apply(bucket)
    return error_df


def build_error_analysis_result(top_indices, metadata, y_true, X_history, model_name="Model", k=10):
    """
    Membuat tabel error analysis per sample test.

    Output utama:
    - apakah ground-truth course masuk Top-K;
    - ranking ground truth jika ditemukan;
    - kategori error/ranking;
    - panjang history user;
    - frekuensi target course di training;
    - rekomendasi Top-K dalam bentuk nama course.
    """
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    topk_array = np.asarray(top_indices).astype(int)[:, :k]

    train_frequency_by_encoded = (
        train_core_enc["course_encoded"]
        .astype(int)
        .value_counts()
        .to_dict()
        if "train_core_enc" in globals()
        else {}
    )

    test_frequency_by_encoded = pd.Series(y_true).value_counts().to_dict()

    rows = []

    for i in range(len(y_true)):
        ground_truth = int(y_true[i])
        recommendations = topk_array[i].astype(int).tolist()
        recommendations = [course_id for course_id in recommendations if course_id != 0]

        if ground_truth in recommendations:
            rank_ground_truth = recommendations.index(ground_truth) + 1
            hit_at_k = 1
        else:
            rank_ground_truth = np.nan
            hit_at_k = 0

        if hit_at_k == 0:
            error_type = f"Miss Top-{k}"
        elif rank_ground_truth == 1:
            error_type = "Hit at Rank 1"
        elif rank_ground_truth <= 5:
            error_type = "Hit at Rank 2-5"
        else:
            error_type = f"Hit at Rank 6-{k}"

        history_encoded = metadata.iloc[i]["history_encoded"]
        history_names = decode_course_list(history_encoded)

        rows.append({
            "ranking_method": model_name,
            "user_id": metadata.iloc[i]["user_id"],
            "history_length": int(metadata.iloc[i]["sequence_length"]),
            "history_length_bucket": categorize_history_length(metadata.iloc[i]["sequence_length"]),
            "history_last_5_courses": " | ".join(history_names[-5:]),
            "ground_truth_encoded": ground_truth,
            "ground_truth_course": decode_course(ground_truth),
            "target_train_frequency": int(train_frequency_by_encoded.get(ground_truth, 0)),
            "target_test_frequency": int(test_frequency_by_encoded.get(ground_truth, 0)),
            "top_1_recommendation": decode_course(recommendations[0]) if len(recommendations) > 0 else None,
            "top_k_recommendations": " | ".join(decode_course_list(recommendations)),
            "hit_at_k": hit_at_k,
            "rank_ground_truth": rank_ground_truth,
            "error_type": error_type
        })

    result_df = pd.DataFrame(rows)
    result_df = add_target_frequency_bucket(result_df)

    ordered_columns = [
        "ranking_method",
        "user_id",
        "history_length",
        "history_length_bucket",
        "ground_truth_course",
        "target_train_frequency",
        "target_frequency_bucket",
        "target_test_frequency",
        "top_1_recommendation",
        "hit_at_k",
        "rank_ground_truth",
        "error_type",
        "history_last_5_courses",
        "top_k_recommendations"
    ]

    return result_df[ordered_columns]


def summarize_error_analysis(error_df, k=10):
    """Membuat ringkasan global error analysis untuk tiap ranking method."""
    rows = []

    for method_name, group_df in error_df.groupby("ranking_method"):
        total_samples = len(group_df)
        hit_count = int(group_df["hit_at_k"].sum())
        miss_count = int(total_samples - hit_count)

        hit_df = group_df[group_df["hit_at_k"] == 1]
        miss_df = group_df[group_df["hit_at_k"] == 0]

        rows.append({
            "ranking_method": method_name,
            "total_samples": total_samples,
            f"Hit@{k}_count": hit_count,
            f"Miss@{k}_count": miss_count,
            f"Hit@{k}_percent": (hit_count / total_samples * 100) if total_samples > 0 else 0,
            f"Miss@{k}_percent": (miss_count / total_samples * 100) if total_samples > 0 else 0,
            "avg_rank_ground_truth_when_hit": hit_df["rank_ground_truth"].mean() if not hit_df.empty else np.nan,
            "median_rank_ground_truth_when_hit": hit_df["rank_ground_truth"].median() if not hit_df.empty else np.nan,
            "avg_history_length_hit": hit_df["history_length"].mean() if not hit_df.empty else np.nan,
            "avg_history_length_miss": miss_df["history_length"].mean() if not miss_df.empty else np.nan,
            "avg_target_train_frequency_hit": hit_df["target_train_frequency"].mean() if not hit_df.empty else np.nan,
            "avg_target_train_frequency_miss": miss_df["target_train_frequency"].mean() if not miss_df.empty else np.nan
        })

    return pd.DataFrame(rows)


def build_grouped_error_summary(error_df, group_column, k=10):
    """Membuat ringkasan Hit@K dan Miss@K berdasarkan kelompok tertentu."""
    grouped = (
        error_df
        .groupby(["ranking_method", group_column], dropna=False)
        .agg(
            total_samples=("hit_at_k", "count"),
            hit_count=("hit_at_k", "sum"),
            avg_rank_ground_truth_when_hit=("rank_ground_truth", "mean"),
            avg_target_train_frequency=("target_train_frequency", "mean"),
            avg_history_length=("history_length", "mean")
        )
        .reset_index()
    )

    grouped["miss_count"] = grouped["total_samples"] - grouped["hit_count"]
    grouped[f"Hit@{k}_percent"] = grouped["hit_count"] / grouped["total_samples"] * 100
    grouped[f"Miss@{k}_percent"] = grouped["miss_count"] / grouped["total_samples"] * 100

    return grouped


def plot_error_type_distribution(error_type_summary_df):
    """Menampilkan distribusi kategori error/hit untuk tiap metode ranking."""
    if error_type_summary_df.empty:
        print("Error type summary kosong, grafik tidak dapat ditampilkan.")
        return

    plot_df = error_type_summary_df.pivot(
        index="error_type",
        columns="ranking_method",
        values="sample_count"
    ).fillna(0)

    plot_df.plot(kind="bar", figsize=(10, 5))
    plt.title("Distribusi Error Type pada Top-10 Recommendation")
    plt.xlabel("Error Type")
    plt.ylabel("Jumlah Sample")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


def plot_hit_rate_by_group(grouped_df, group_column, title, k=10):
    """Menampilkan Hit@K berdasarkan bucket tertentu."""
    if grouped_df.empty:
        print("Grouped error summary kosong, grafik tidak dapat ditampilkan.")
        return

    plot_df = grouped_df.pivot(
        index=group_column,
        columns="ranking_method",
        values=f"Hit@{k}_percent"
    ).fillna(0)

    plot_df.plot(kind="bar", figsize=(10, 5))
    plt.title(title)
    plt.xlabel(group_column)
    plt.ylabel(f"Hit@{k} (%)")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 100)
    plt.tight_layout()
    plt.show()


ERROR_ANALYSIS_K = 10

softmax_error_analysis_df = build_error_analysis_result(
    softmax_top10_indices,
    test_metadata,
    y_test,
    X_course_test,
    model_name="Softmax Ranking",
    k=ERROR_ANALYSIS_K
)

similarity_error_analysis_df = build_error_analysis_result(
    similarity_top10_indices,
    test_metadata,
    y_test,
    X_course_test,
    model_name="Similarity Ranking",
    k=ERROR_ANALYSIS_K
)

error_analysis_detail_df = pd.concat(
    [softmax_error_analysis_df, similarity_error_analysis_df],
    ignore_index=True
)

error_analysis_summary_df = summarize_error_analysis(
    error_analysis_detail_df,
    k=ERROR_ANALYSIS_K
)

error_analysis_summary_display = error_analysis_summary_df.copy()
summary_percentage_columns = [
    f"Hit@{ERROR_ANALYSIS_K}_percent",
    f"Miss@{ERROR_ANALYSIS_K}_percent"
]
summary_numeric_columns = [
    "avg_rank_ground_truth_when_hit",
    "median_rank_ground_truth_when_hit",
    "avg_history_length_hit",
    "avg_history_length_miss",
    "avg_target_train_frequency_hit",
    "avg_target_train_frequency_miss"
]

for col in summary_percentage_columns + summary_numeric_columns:
    if col in error_analysis_summary_display.columns:
        error_analysis_summary_display[col] = error_analysis_summary_display[col].round(2)

print("=== Error Analysis Summary ===")
display(error_analysis_summary_display)


# ------------------------------------------------------------
# Distribusi kategori error
# ------------------------------------------------------------
error_type_summary_df = (
    error_analysis_detail_df
    .groupby(["ranking_method", "error_type"])
    .size()
    .reset_index(name="sample_count")
)

error_type_summary_df["percentage"] = (
    error_type_summary_df["sample_count"]
    / error_type_summary_df.groupby("ranking_method")["sample_count"].transform("sum")
    * 100
)

error_type_summary_display = error_type_summary_df.copy()
error_type_summary_display["percentage"] = error_type_summary_display["percentage"].round(2)

print("\n=== Distribusi Error Type ===")
display(error_type_summary_display)

plot_error_type_distribution(error_type_summary_df)


# ------------------------------------------------------------
# Error berdasarkan panjang history user
# ------------------------------------------------------------
history_bucket_error_summary_df = build_grouped_error_summary(
    error_analysis_detail_df,
    group_column="history_length_bucket",
    k=ERROR_ANALYSIS_K
)

history_bucket_error_summary_display = history_bucket_error_summary_df.copy()
for col in [
    f"Hit@{ERROR_ANALYSIS_K}_percent",
    f"Miss@{ERROR_ANALYSIS_K}_percent",
    "avg_rank_ground_truth_when_hit",
    "avg_target_train_frequency",
    "avg_history_length"
]:
    if col in history_bucket_error_summary_display.columns:
        history_bucket_error_summary_display[col] = history_bucket_error_summary_display[col].round(2)

print("\n=== Error Analysis berdasarkan Panjang History User ===")
display(history_bucket_error_summary_display)

plot_hit_rate_by_group(
    history_bucket_error_summary_df,
    group_column="history_length_bucket",
    title="Hit@10 berdasarkan Panjang History User",
    k=ERROR_ANALYSIS_K
)


# ------------------------------------------------------------
# Error berdasarkan frekuensi target course di train
# ------------------------------------------------------------
frequency_bucket_error_summary_df = build_grouped_error_summary(
    error_analysis_detail_df,
    group_column="target_frequency_bucket",
    k=ERROR_ANALYSIS_K
)

frequency_bucket_error_summary_display = frequency_bucket_error_summary_df.copy()
for col in [
    f"Hit@{ERROR_ANALYSIS_K}_percent",
    f"Miss@{ERROR_ANALYSIS_K}_percent",
    "avg_rank_ground_truth_when_hit",
    "avg_target_train_frequency",
    "avg_history_length"
]:
    if col in frequency_bucket_error_summary_display.columns:
        frequency_bucket_error_summary_display[col] = frequency_bucket_error_summary_display[col].round(2)

print("\n=== Error Analysis berdasarkan Frekuensi Target Course di Train ===")
display(frequency_bucket_error_summary_display)

plot_hit_rate_by_group(
    frequency_bucket_error_summary_df,
    group_column="target_frequency_bucket",
    title="Hit@10 berdasarkan Frekuensi Target Course di Train",
    k=ERROR_ANALYSIS_K
)


# ------------------------------------------------------------
# Contoh kasus miss dan ranking rendah
# ------------------------------------------------------------
miss_case_examples_df = (
    error_analysis_detail_df[error_analysis_detail_df["hit_at_k"] == 0]
    .sort_values(["ranking_method", "target_train_frequency", "history_length"], ascending=[True, True, True])
    .head(10)
)

low_rank_hit_examples_df = (
    error_analysis_detail_df[
        (error_analysis_detail_df["hit_at_k"] == 1)
        & (error_analysis_detail_df["rank_ground_truth"] > 5)
    ]
    .sort_values(["ranking_method", "rank_ground_truth", "target_train_frequency"], ascending=[True, False, True])
    .head(10)
)

print("\n=== Contoh Kasus Miss Top-10 ===")
if miss_case_examples_df.empty:
    print("Tidak ada kasus Miss Top-10 pada hasil evaluasi ini.")
else:
    display(miss_case_examples_df)

print("\n=== Contoh Kasus Hit tetapi Ranking Rendah (Rank > 5) ===")
if low_rank_hit_examples_df.empty:
    print("Tidak ada kasus Hit dengan ranking ground truth > 5 pada hasil evaluasi ini.")
else:
    display(low_rank_hit_examples_df)


# ------------------------------------------------------------
# Perbandingan kasus Softmax vs Similarity
# ------------------------------------------------------------
softmax_case_status = softmax_error_analysis_df[[
    "user_id",
    "ground_truth_course",
    "hit_at_k",
    "rank_ground_truth",
    "error_type"
]].rename(columns={
    "hit_at_k": "softmax_hit_at_10",
    "rank_ground_truth": "softmax_rank_ground_truth",
    "error_type": "softmax_error_type"
})

similarity_case_status = similarity_error_analysis_df[[
    "user_id",
    "ground_truth_course",
    "hit_at_k",
    "rank_ground_truth",
    "error_type"
]].rename(columns={
    "hit_at_k": "similarity_hit_at_10",
    "rank_ground_truth": "similarity_rank_ground_truth",
    "error_type": "similarity_error_type"
})

ranking_case_comparison_df = softmax_case_status.merge(
    similarity_case_status,
    on=["user_id", "ground_truth_course"],
    how="inner"
)

conditions = [
    (ranking_case_comparison_df["softmax_hit_at_10"] == 1) & (ranking_case_comparison_df["similarity_hit_at_10"] == 1),
    (ranking_case_comparison_df["softmax_hit_at_10"] == 1) & (ranking_case_comparison_df["similarity_hit_at_10"] == 0),
    (ranking_case_comparison_df["softmax_hit_at_10"] == 0) & (ranking_case_comparison_df["similarity_hit_at_10"] == 1),
    (ranking_case_comparison_df["softmax_hit_at_10"] == 0) & (ranking_case_comparison_df["similarity_hit_at_10"] == 0)
]
choices = [
    "Both Hit",
    "Only Softmax Hit",
    "Only Similarity Hit",
    "Both Miss"
]

ranking_case_comparison_df["comparison_status"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

ranking_case_comparison_summary_df = (
    ranking_case_comparison_df["comparison_status"]
    .value_counts()
    .rename_axis("comparison_status")
    .reset_index(name="sample_count")
)

ranking_case_comparison_summary_df["percentage"] = (
    ranking_case_comparison_summary_df["sample_count"]
    / ranking_case_comparison_summary_df["sample_count"].sum()
    * 100
)

ranking_case_comparison_summary_display = ranking_case_comparison_summary_df.copy()
ranking_case_comparison_summary_display["percentage"] = ranking_case_comparison_summary_display["percentage"].round(2)

print("\n=== Perbandingan Kasus Softmax vs Similarity ===")
display(ranking_case_comparison_summary_display)

print("\n=== Contoh Detail Perbandingan Softmax vs Similarity ===")
display(ranking_case_comparison_df.head(10))


### Interpretasi Error Analysis

Error analysis dilakukan untuk memahami pola kegagalan sistem rekomendasi course pada data test. Analisis ini tidak hanya melihat apakah ground-truth course berhasil masuk ke Top-10, tetapi juga memperhatikan posisi ranking ground truth, panjang riwayat user, serta frekuensi target course pada data training.

Kategori `Hit at Rank 1` menunjukkan bahwa model berhasil menempatkan course target sebagai rekomendasi paling atas. Kategori `Hit at Rank 2-5` dan `Hit at Rank 6-10` menunjukkan bahwa target course sudah berhasil ditemukan, tetapi belum selalu ditempatkan pada ranking terbaik. Sementara itu, `Miss Top-10` menunjukkan bahwa course target tidak muncul dalam sepuluh rekomendasi teratas.

Apabila banyak kasus berada pada kategori `Hit at Rank 6-10`, maka model sudah mampu menangkap relevansi course secara umum, tetapi kualitas pengurutan ranking masih perlu ditingkatkan. Sebaliknya, apabila proporsi `Miss Top-10` tinggi, maka model masih kesulitan mengenali pola transisi course pada sebagian user atau pada course target tertentu.

Analisis berdasarkan panjang riwayat user digunakan untuk melihat apakah model lebih stabil pada user dengan history yang lebih panjang. Dalam recommender system berbasis sequence, user dengan history pendek biasanya memberikan sinyal preferensi yang lebih terbatas, sehingga rekomendasi cenderung lebih sulit. Analisis berdasarkan frekuensi target course di data training digunakan untuk melihat apakah model lebih mudah memprediksi course yang sering muncul dibandingkan course yang relatif jarang. Jika error lebih banyak terjadi pada target dengan frekuensi rendah, maka hal tersebut menunjukkan adanya tantangan long-tail item dalam sistem rekomendasi.

Dengan demikian, error analysis memperkuat pembahasan hasil eksperimen karena menjelaskan pola kelemahan model, bukan hanya melaporkan nilai Hit@K, MRR@K, NDCG@K, dan Coverage@10.

## 32. Sensitivitas `MIN_COURSE_COUNT_TRAIN`

**Penjelasan:** Bagian ini menambahkan analisis sensitivitas terhadap parameter `MIN_COURSE_COUNT_TRAIN`. Parameter ini menentukan batas minimal kemunculan course pada `train_core_raw` agar course tersebut masuk ke vocabulary rekomendasi.

**Tujuan:** Mengecek apakah konfigurasi `MIN_COURSE_COUNT_TRAIN = 20` terlalu ketat atau masih masuk akal. Analisis dilakukan secara ringan tanpa retraining berulang, sehingga aman dijalankan di Colab. Output yang dianalisis meliputi jumlah course valid, jumlah baris train, jumlah target validation/test yang masih dapat dievaluasi, target coverage, dan sanity-check Popularity Hit@10.

**Yang perlu diperhatikan:** Bagian ini bukan pengganti evaluasi model utama, melainkan analisis stabilitas data dan feasibility evaluation. Hasilnya berguna untuk justifikasi metodologis dalam paper.

In [ ]:
# ============================================================
# SENSITIVITAS MIN_COURSE_COUNT_TRAIN
# ============================================================

SENSITIVITY_THRESHOLDS = [1, 2, 5, 10, 15, 20, 30, 50]
SENSITIVITY_K = 10
MIN_HISTORY_AFTER_FILTER = 2


def calculate_popularity_hit_at_k_for_threshold(
    train_df,
    history_df,
    target_df,
    k=10
):
    """
    Menghitung sanity-check Popularity Hit@K untuk satu konfigurasi threshold.

    Popularity baseline merekomendasikan course paling sering muncul di train,
    lalu memfilter course yang sudah pernah diambil user pada history evaluasi.
    Baseline ini ringan dan tidak membutuhkan retraining model.
    """
    if train_df.empty or history_df.empty or target_df.empty:
        return np.nan

    popularity_ranking = (
        train_df["canonical_course_id"]
        .astype(str)
        .value_counts()
        .index
        .astype(str)
        .tolist()
    )

    history_by_user = (
        history_df.groupby("user_id")["canonical_course_id"]
        .apply(lambda values: set(values.astype(str)))
        .to_dict()
    )

    hits = []

    for _, row in target_df.iterrows():
        user_id = row["user_id"]
        target_course = str(row["canonical_course_id"])
        seen_courses = history_by_user.get(user_id, set())

        recommendations = [
            course_id
            for course_id in popularity_ranking
            if course_id not in seen_courses
        ][:k]

        hits.append(float(target_course in recommendations))

    return float(np.mean(hits)) if len(hits) > 0 else np.nan


def build_min_course_count_sensitivity_table(
    thresholds,
    k=10,
    min_history_after_filter=2
):
    """
    Membuat tabel sensitivitas MIN_COURSE_COUNT_TRAIN.

    Semua threshold dihitung dari train_core_raw agar tetap train-only.
    Validation/test tidak dipakai untuk menentukan course vocabulary.
    """
    rows = []
    course_counts_from_train = train_core_raw["canonical_course_id"].astype(str).value_counts()

    raw_val_count = max(len(val_target_raw), 1)
    raw_test_count = max(len(test_target_raw), 1)
    raw_train_course_count = max(train_core_raw["canonical_course_id"].astype(str).nunique(), 1)

    for threshold in thresholds:
        threshold = int(threshold)
        valid_courses = course_counts_from_train[course_counts_from_train >= threshold].index
        valid_course_set = set(valid_courses.astype(str))

        train_filtered = train_core_raw[
            train_core_raw["canonical_course_id"].astype(str).isin(valid_course_set)
        ].copy()

        val_target_filtered = val_target_raw[
            val_target_raw["canonical_course_id"].astype(str).isin(valid_course_set)
        ].copy()

        pretest_history_filtered = pretest_history_raw[
            pretest_history_raw["canonical_course_id"].astype(str).isin(valid_course_set)
        ].copy()

        test_target_filtered = test_target_raw[
            test_target_raw["canonical_course_id"].astype(str).isin(valid_course_set)
        ].copy()

        # Syarat validation: user memiliki minimal history dari train_core setelah filtering.
        train_history_count = train_filtered.groupby("user_id")["canonical_course_id"].count()
        valid_val_users = train_history_count[train_history_count >= min_history_after_filter].index
        val_target_filtered = val_target_filtered[val_target_filtered["user_id"].isin(valid_val_users)].copy()

        # Syarat test default: user memiliki minimal history dari pretest_history setelah filtering.
        pretest_history_count = pretest_history_filtered.groupby("user_id")["canonical_course_id"].count()
        valid_test_users = pretest_history_count[pretest_history_count >= min_history_after_filter].index
        test_target_filtered = test_target_filtered[test_target_filtered["user_id"].isin(valid_test_users)].copy()

        popularity_hit_at_k = calculate_popularity_hit_at_k_for_threshold(
            train_df=train_filtered,
            history_df=pretest_history_filtered,
            target_df=test_target_filtered,
            k=k
        )

        rows.append({
            "MIN_COURSE_COUNT_TRAIN": threshold,
            "is_current_setting": threshold == int(MIN_COURSE_COUNT_TRAIN),
            "valid_course_count": int(len(valid_courses)),
            "valid_course_ratio_percent": float(len(valid_courses) / raw_train_course_count * 100),
            "train_rows_after_filter": int(len(train_filtered)),
            "train_users_after_filter": int(train_filtered["user_id"].nunique()),
            "validation_samples_after_filter": int(len(val_target_filtered)),
            "validation_target_retention_percent": float(len(val_target_filtered) / raw_val_count * 100),
            "test_samples_after_filter": int(len(test_target_filtered)),
            "test_target_retention_percent": float(len(test_target_filtered) / raw_test_count * 100),
            f"popularity_Hit@{k}": popularity_hit_at_k,
            f"popularity_Hit@{k}_percent": popularity_hit_at_k * 100 if not pd.isna(popularity_hit_at_k) else np.nan
        })

    return pd.DataFrame(rows)


min_course_sensitivity_df = build_min_course_count_sensitivity_table(
    thresholds=SENSITIVITY_THRESHOLDS,
    k=SENSITIVITY_K,
    min_history_after_filter=MIN_HISTORY_AFTER_FILTER
)

min_course_sensitivity_display = min_course_sensitivity_df.copy()

for col in [
    "valid_course_ratio_percent",
    "validation_target_retention_percent",
    "test_target_retention_percent",
    f"popularity_Hit@{SENSITIVITY_K}",
    f"popularity_Hit@{SENSITIVITY_K}_percent"
]:
    if col in min_course_sensitivity_display.columns:
        min_course_sensitivity_display[col] = min_course_sensitivity_display[col].round(4 if "Hit@" in col and "percent" not in col else 2)

print("=== Sensitivitas MIN_COURSE_COUNT_TRAIN ===")
display(min_course_sensitivity_display)

current_threshold_sensitivity_row = min_course_sensitivity_df[
    min_course_sensitivity_df["is_current_setting"] == True
].copy()

print("\n=== Setting yang Dipakai pada Model Utama ===")
if current_threshold_sensitivity_row.empty:
    print("Threshold model utama tidak ada di SENSITIVITY_THRESHOLDS. Tambahkan nilainya jika ingin dibandingkan langsung.")
else:
    display(current_threshold_sensitivity_row)


def build_threshold_justification_summary(current_row):
    """
    Membuat ringkasan justifikasi pemilihan MIN_COURSE_COUNT_TRAIN pada model utama.
    Ringkasan ini membantu menjelaskan trade-off antara stabilitas data dan cakupan evaluasi.
    """
    if current_row.empty:
        return pd.DataFrame([{
            "aspect": "Current threshold",
            "value": MIN_COURSE_COUNT_TRAIN,
            "interpretation": "Current threshold is not included in the sensitivity table."
        }])

    row = current_row.iloc[0]

    valid_course_count = int(row["valid_course_count"])
    valid_course_ratio = float(row["valid_course_ratio_percent"])
    test_retention = float(row["test_target_retention_percent"])
    val_retention = float(row["validation_target_retention_percent"])
    train_rows_after_filter = int(row["train_rows_after_filter"])
    test_samples_after_filter = int(row["test_samples_after_filter"])

    return pd.DataFrame([
        {
            "aspect": "Current threshold",
            "value": int(MIN_COURSE_COUNT_TRAIN),
            "interpretation": "Minimum course frequency is computed only from train_core_raw, so validation and test do not determine the course vocabulary."
        },
        {
            "aspect": "Valid course retained",
            "value": f"{valid_course_count} courses ({valid_course_ratio:.2f}% of train courses)",
            "interpretation": "The catalog is intentionally restricted to courses with sufficient historical signal for sequence modeling."
        },
        {
            "aspect": "Training rows retained",
            "value": train_rows_after_filter,
            "interpretation": "The model still has enough interaction rows after filtering to learn next-course patterns."
        },
        {
            "aspect": "Validation target retained",
            "value": f"{val_retention:.2f}%",
            "interpretation": "Validation remains usable after applying the train-only course filter."
        },
        {
            "aspect": "Test target retained",
            "value": f"{test_retention:.2f}% ({test_samples_after_filter} samples)",
            "interpretation": "The selected threshold keeps the evaluation set sufficiently large while reducing sparse-course noise."
        }
    ])


threshold_justification_summary_df = build_threshold_justification_summary(
    current_threshold_sensitivity_row
)

print("\n=== Justifikasi Setting MIN_COURSE_COUNT_TRAIN Utama ===")
display(threshold_justification_summary_df)


# ------------------------------------------------------------
# Visualisasi sensitivitas threshold
# ------------------------------------------------------------
plt.figure(figsize=(9, 5))
plt.plot(
    min_course_sensitivity_df["MIN_COURSE_COUNT_TRAIN"],
    min_course_sensitivity_df["valid_course_count"],
    marker="o"
)
plt.title("Sensitivitas Threshold terhadap Jumlah Course Valid")
plt.xlabel("MIN_COURSE_COUNT_TRAIN")
plt.ylabel("Jumlah Course Valid")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(
    min_course_sensitivity_df["MIN_COURSE_COUNT_TRAIN"],
    min_course_sensitivity_df["test_samples_after_filter"],
    marker="o"
)
plt.title("Sensitivitas Threshold terhadap Jumlah Sampel Test")
plt.xlabel("MIN_COURSE_COUNT_TRAIN")
plt.ylabel("Jumlah Sampel Test Setelah Filtering")
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(
    min_course_sensitivity_df["MIN_COURSE_COUNT_TRAIN"],
    min_course_sensitivity_df[f"popularity_Hit@{SENSITIVITY_K}_percent"],
    marker="o"
)
plt.title(f"Sanity Check Popularity Hit@{SENSITIVITY_K} terhadap Threshold")
plt.xlabel("MIN_COURSE_COUNT_TRAIN")
plt.ylabel(f"Popularity Hit@{SENSITIVITY_K} (%)")
plt.grid(True, alpha=0.3)
plt.show()


### 32.1 Interpretasi Sensitivitas `MIN_COURSE_COUNT_TRAIN`

Analisis sensitivitas dilakukan untuk menilai pengaruh nilai minimum kemunculan course pada data training terhadap kelayakan evaluasi sistem rekomendasi. Nilai `MIN_COURSE_COUNT_TRAIN` yang terlalu rendah akan mempertahankan lebih banyak course, tetapi dapat memasukkan course yang sangat jarang muncul sehingga sinyal pembelajaran menjadi kurang stabil. Sebaliknya, nilai yang terlalu tinggi akan menghasilkan vocabulary course yang lebih bersih, tetapi berisiko menghapus banyak target validation/test sehingga cakupan evaluasi menjadi sempit.

Dalam pipeline ini, seluruh proses seleksi course tetap dilakukan berdasarkan `train_core_raw` saja. Dengan demikian, validation dan test tidak ikut menentukan course vocabulary. Setting utama `MIN_COURSE_COUNT_TRAIN = 20` diposisikan sebagai kompromi metodologis: threshold ini membatasi kandidat rekomendasi pada course yang memiliki cukup sinyal historis di data training, namun tetap mempertahankan jumlah sampel validation dan test yang memadai untuk evaluasi. Ringkasan `threshold_justification_summary_df` digunakan untuk menunjukkan trade-off tersebut secara eksplisit melalui jumlah course valid, jumlah baris training, retensi target validation/test, dan ukuran sampel test setelah filtering.

Justifikasi ini penting karena sistem rekomendasi berbasis sequence sensitif terhadap sparsity. Course yang terlalu jarang muncul sulit dipelajari oleh embedding dan LSTM, sedangkan filtering yang terlalu agresif dapat mempersempit katalog rekomendasi. Oleh karena itu, pemilihan threshold tidak hanya bersifat teknis, tetapi merupakan bagian dari desain eksperimen agar model tetap stabil, leakage-safe, dan evaluasinya masih representatif.

## 33. Strict Train-History Evaluation

**Penjelasan:** Bagian ini membandingkan dua protokol evaluasi test yang berbeda agar hasil model dapat dijelaskan lebih aman dalam konteks paper recommender system.

1. **Default pretest-history evaluation** menggunakan `pretest_history_enc` sebagai input history test. Artinya, model menerima seluruh riwayat user sebelum target test. Skenario ini valid untuk simulasi operasional ketika sistem sudah mengetahui riwayat course terbaru user sebelum merekomendasikan course berikutnya.
2. **Strict train-history evaluation** menggunakan `train_core_enc` sebagai input history test. Artinya, input test hanya berisi riwayat yang tersedia pada data training utama, sehingga validation target tidak ikut digunakan sebagai bagian dari history saat memprediksi target test.

**Tujuan:** Evaluasi strict digunakan sebagai *robustness check*. Jika performa strict masih mendekati performa default, model dapat dikatakan tetap stabil walaupun informasi history yang tersedia lebih terbatas. Jika performa strict turun besar, maka performa default perlu dijelaskan sebagai performa pada skenario operasional dengan history yang lebih lengkap.

**Yang perlu diperhatikan:** Default evaluation dan strict evaluation tidak boleh dicampur sebagai satu jenis hasil. Default evaluation menjawab pertanyaan “seberapa baik sistem ketika riwayat terbaru user tersedia?”, sedangkan strict evaluation menjawab pertanyaan “seberapa kuat model ketika hanya history training yang digunakan sebagai input test?”.

In [ ]:
# ============================================================
# STRICT TRAIN-HISTORY EVALUATION
# ============================================================

(
    X_course_test_strict,
    X_competency_test_strict,
    X_level_test_strict,
    X_course_has_certificate_test_strict,
    X_certificate_count_test_strict,
    y_test_strict,
    test_metadata_strict
) = make_temporal_holdout_dataset(
    history_df=train_core_enc,
    target_df=test_target_enc,
    label_name="strict_train_history_test"
)

print("=== STRICT TRAIN-HISTORY TEST DATASET ===")
print("X_course_test_strict:", X_course_test_strict.shape)
print("y_test_strict:", y_test_strict.shape)
print("test_metadata_strict:", test_metadata_strict.shape)


# ------------------------------------------------------------
# Audit apakah validation target masuk ke history default/strict
# ------------------------------------------------------------
val_target_by_user_encoded = (
    val_target_enc[["user_id", "course_encoded", "course_name"]]
    .drop_duplicates("user_id")
    .rename(columns={
        "course_encoded": "validation_target_encoded",
        "course_name": "validation_target_course"
    })
)


def count_validation_target_inside_history(metadata_df, label_name):
    """Menghitung berapa user yang validation target-nya muncul di history evaluasi."""
    if metadata_df.empty:
        return pd.DataFrame([{
            "evaluation_history": label_name,
            "sample_count": 0,
            "validation_target_in_history_count": 0,
            "validation_target_in_history_percent": 0.0
        }])

    audit_df = metadata_df.merge(
        val_target_by_user_encoded,
        on="user_id",
        how="left"
    )

    audit_df["validation_target_in_history"] = audit_df.apply(
        lambda row: int(row["validation_target_encoded"]) in [int(x) for x in row["history_encoded"]]
        if pd.notna(row["validation_target_encoded"]) else False,
        axis=1
    )

    sample_count = len(audit_df)
    leak_like_count = int(audit_df["validation_target_in_history"].sum())

    return pd.DataFrame([{
        "evaluation_history": label_name,
        "sample_count": sample_count,
        "validation_target_in_history_count": leak_like_count,
        "validation_target_in_history_percent": leak_like_count / sample_count * 100 if sample_count > 0 else 0.0
    }])


strict_history_audit_df = pd.concat([
    count_validation_target_inside_history(test_metadata, "Default test history = pretest_history_enc"),
    count_validation_target_inside_history(test_metadata_strict, "Strict test history = train_core_enc")
], ignore_index=True)

strict_history_audit_display = strict_history_audit_df.copy()
strict_history_audit_display["validation_target_in_history_percent"] = strict_history_audit_display[
    "validation_target_in_history_percent"
].round(2)

print("\n=== Audit Validation Target di History Evaluasi Test ===")
display(strict_history_audit_display)


# ------------------------------------------------------------
# Evaluasi model pada strict history
# ------------------------------------------------------------
if len(y_test_strict) == 0:
    print("Strict evaluation kosong. Artinya tidak ada user test yang masih memiliki minimal history train setelah filtering.")
    strict_softmax_metrics_df = pd.DataFrame()
    strict_similarity_metrics_df = pd.DataFrame()
    strict_metrics_comparison_df = pd.DataFrame()
else:
    strict_test_loss = model.evaluate(
        [
            X_course_test_strict,
            X_competency_test_strict,
            X_level_test_strict,
            X_course_has_certificate_test_strict,
            X_certificate_count_test_strict
        ],
        y_test_strict,
        verbose=0
    )

    strict_softmax_scores_test = model.predict(
        [
            X_course_test_strict,
            X_competency_test_strict,
            X_level_test_strict,
            X_course_has_certificate_test_strict,
            X_certificate_count_test_strict
        ],
        verbose=0
    )

    strict_softmax_metrics, strict_softmax_metrics_df, strict_softmax_top10_indices = evaluate_topk_ranking_masked(
        strict_softmax_scores_test,
        X_course_test_strict,
        y_test_strict,
        k_values=(1, 5, 10)
    )

    strict_user_vectors_test = decoder_vector_model.predict(
        [
            X_course_test_strict,
            X_competency_test_strict,
            X_level_test_strict,
            X_course_has_certificate_test_strict,
            X_certificate_count_test_strict
        ],
        verbose=0
    )

    strict_similarity_scores_test = calculate_similarity(strict_user_vectors_test, course_vectors)

    strict_similarity_metrics, strict_similarity_metrics_df, strict_similarity_top10_indices = evaluate_topk_ranking_masked(
        strict_similarity_scores_test,
        X_course_test_strict,
        y_test_strict,
        k_values=(1, 5, 10)
    )

    print("\nStrict Test Loss:", strict_test_loss)

    print("\n=== Strict Softmax Top-K Ranking Metrics ===")
    display(
        strict_softmax_metrics_df.assign(
            score=strict_softmax_metrics_df["score"].round(4),
            percentage=strict_softmax_metrics_df["percentage"].round(2)
        )
    )

    print("\n=== Strict Similarity Top-K Ranking Metrics ===")
    display(
        strict_similarity_metrics_df.assign(
            score=strict_similarity_metrics_df["score"].round(4),
            percentage=strict_similarity_metrics_df["percentage"].round(2)
        )
    )

    # ------------------------------------------------------------
    # Perbandingan evaluasi utama vs strict
    # ------------------------------------------------------------
    default_softmax_compare = softmax_metrics_df.copy()
    default_softmax_compare["ranking_method"] = "Softmax Ranking"
    default_softmax_compare["evaluation_setting"] = "Default pretest-history"

    strict_softmax_compare = strict_softmax_metrics_df.copy()
    strict_softmax_compare["ranking_method"] = "Softmax Ranking"
    strict_softmax_compare["evaluation_setting"] = "Strict train-history"

    default_similarity_compare = similarity_metrics_df.copy()
    default_similarity_compare["ranking_method"] = "Similarity Ranking"
    default_similarity_compare["evaluation_setting"] = "Default pretest-history"

    strict_similarity_compare = strict_similarity_metrics_df.copy()
    strict_similarity_compare["ranking_method"] = "Similarity Ranking"
    strict_similarity_compare["evaluation_setting"] = "Strict train-history"

    strict_metrics_comparison_df = pd.concat([
        default_softmax_compare,
        strict_softmax_compare,
        default_similarity_compare,
        strict_similarity_compare
    ], ignore_index=True)

    strict_metrics_comparison_display = strict_metrics_comparison_df[[
        "ranking_method",
        "evaluation_setting",
        "metric",
        "score",
        "percentage"
    ]].copy()

    strict_metrics_comparison_display["score"] = strict_metrics_comparison_display["score"].round(4)
    strict_metrics_comparison_display["percentage"] = strict_metrics_comparison_display["percentage"].round(2)

    print("\n=== Perbandingan Default Evaluation vs Strict Train-History Evaluation ===")
    display(strict_metrics_comparison_display)

    strict_pivot_df = strict_metrics_comparison_df.pivot_table(
        index=["ranking_method", "metric"],
        columns="evaluation_setting",
        values="score",
        aggfunc="first"
    ).reset_index()

    if "Default pretest-history" in strict_pivot_df.columns and "Strict train-history" in strict_pivot_df.columns:
        strict_pivot_df["delta_strict_minus_default"] = (
            strict_pivot_df["Strict train-history"]
            - strict_pivot_df["Default pretest-history"]
        )
        strict_pivot_df["delta_percent_point"] = strict_pivot_df["delta_strict_minus_default"] * 100

    strict_delta_display = strict_pivot_df.copy()
    for col in ["Default pretest-history", "Strict train-history", "delta_strict_minus_default"]:
        if col in strict_delta_display.columns:
            strict_delta_display[col] = strict_delta_display[col].round(4)
    if "delta_percent_point" in strict_delta_display.columns:
        strict_delta_display["delta_percent_point"] = strict_delta_display["delta_percent_point"].round(2)

    print("\n=== Delta Strict Evaluation terhadap Default Evaluation ===")
    display(strict_delta_display)

    plt.figure(figsize=(10, 5))
    hit10_compare_df = strict_metrics_comparison_df[
        strict_metrics_comparison_df["metric"] == "Hit@10"
    ].copy()

    x_labels = hit10_compare_df["ranking_method"] + "\n" + hit10_compare_df["evaluation_setting"]
    plt.bar(x_labels, hit10_compare_df["percentage"])
    plt.title("Perbandingan Hit@10: Default vs Strict Train-History")
    plt.xlabel("Metode dan Setting Evaluasi")
    plt.ylabel("Hit@10 (%)")
    plt.xticks(rotation=30, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


### 33.1 Ringkasan Protokol Evaluasi Default dan Strict

Bagian ini merangkum perbedaan antara default pretest-history evaluation dan strict train-history evaluation. Ringkasan ini penting agar pembaca paper tidak menafsirkan kedua hasil evaluasi sebagai skenario yang sama.

Default evaluation digunakan sebagai simulasi rekomendasi operasional karena input test berisi seluruh riwayat sebelum target test. Strict evaluation digunakan sebagai evaluasi konservatif karena hanya memakai riwayat dari train core. Dengan demikian, strict evaluation berperan sebagai *robustness check*, bukan pengganti langsung dari skenario operasional.

In [ ]:
# ============================================================
# EVALUATION PROTOCOL SUMMARY: DEFAULT VS STRICT
# ============================================================

evaluation_protocol_summary_df = pd.DataFrame([
    {
        "evaluation_setting": "Default pretest-history evaluation",
        "history_source_for_test": "pretest_history_enc",
        "test_target": "Last course per user",
        "validation_target_in_test_history": "Can appear, because it occurs before the test target",
        "main_purpose": "Operational next-course recommendation scenario",
        "interpretation": (
            "Measures recommendation quality when the system can use the latest known "
            "course history before the test target."
        )
    },
    {
        "evaluation_setting": "Strict train-history evaluation",
        "history_source_for_test": "train_core_enc",
        "test_target": "Last course per user",
        "validation_target_in_test_history": "Not used",
        "main_purpose": "Conservative robustness check",
        "interpretation": (
            "Measures whether model performance remains stable when the test input "
            "is restricted to training history only."
        )
    }
])

print("=== Ringkasan Protokol Evaluasi Default vs Strict ===")
display(evaluation_protocol_summary_df)


def extract_metric_percentage(comparison_df, ranking_method, evaluation_setting, metric_name):
    """Mengambil nilai percentage untuk metric tertentu dari tabel comparison."""
    if comparison_df is None or comparison_df.empty:
        return np.nan

    subset = comparison_df[
        (comparison_df["ranking_method"] == ranking_method)
        & (comparison_df["evaluation_setting"] == evaluation_setting)
        & (comparison_df["metric"] == metric_name)
    ]

    if subset.empty:
        return np.nan

    return float(subset["percentage"].iloc[0])


def interpret_hit10_delta(delta_percentage_point):
    """Memberi interpretasi ringkas terhadap penurunan Hit@10 pada strict evaluation."""
    if pd.isna(delta_percentage_point):
        return "Not available"

    if delta_percentage_point >= -5:
        return "Small decrease; model remains robust under stricter history input"
    elif delta_percentage_point >= -10:
        return "Moderate decrease; performance is affected but still reasonably stable"
    else:
        return "Large decrease; default performance depends strongly on more complete history"


strict_robustness_rows = []

if "strict_metrics_comparison_df" in globals() and not strict_metrics_comparison_df.empty:
    for ranking_method in ["Softmax Ranking", "Similarity Ranking"]:
        default_hit10_percent = extract_metric_percentage(
            strict_metrics_comparison_df,
            ranking_method,
            "Default pretest-history",
            "Hit@10"
        )

        strict_hit10_percent = extract_metric_percentage(
            strict_metrics_comparison_df,
            ranking_method,
            "Strict train-history",
            "Hit@10"
        )

        delta_percentage_point = strict_hit10_percent - default_hit10_percent

        if pd.notna(default_hit10_percent) and default_hit10_percent != 0:
            strict_retention_percent = strict_hit10_percent / default_hit10_percent * 100
        else:
            strict_retention_percent = np.nan

        strict_robustness_rows.append({
            "ranking_method": ranking_method,
            "default_hit10_percent": default_hit10_percent,
            "strict_hit10_percent": strict_hit10_percent,
            "delta_strict_minus_default_percentage_point": delta_percentage_point,
            "strict_retention_percent": strict_retention_percent,
            "robustness_interpretation": interpret_hit10_delta(delta_percentage_point)
        })

strict_robustness_summary_df = pd.DataFrame(strict_robustness_rows)

if strict_robustness_summary_df.empty:
    print("\nStrict robustness summary belum tersedia karena strict evaluation belum menghasilkan comparison table.")
else:
    strict_robustness_summary_display = strict_robustness_summary_df.copy()
    numeric_columns = [
        "default_hit10_percent",
        "strict_hit10_percent",
        "delta_strict_minus_default_percentage_point",
        "strict_retention_percent"
    ]
    strict_robustness_summary_display[numeric_columns] = strict_robustness_summary_display[numeric_columns].round(2)

    print("\n=== Ringkasan Robustness berdasarkan Hit@10 ===")
    display(strict_robustness_summary_display)


print("\n=== Catatan Interpretasi untuk Paper ===")
print(
    "Default pretest-history evaluation dilaporkan sebagai skenario operasional, "
    "sedangkan strict train-history evaluation dilaporkan sebagai robustness check. "
    "Keduanya perlu dipisahkan agar hasil evaluasi tidak ditafsirkan sebagai protokol yang sama."
)


### 33.2 Interpretasi Strict Train-History Evaluation

Strict train-history evaluation digunakan sebagai evaluasi tambahan yang lebih konservatif. Pada setting default, input test berisi seluruh riwayat user sebelum target test, sehingga validation target dapat berada di dalam history karena secara temporal memang terjadi sebelum test target. Skenario ini sesuai untuk simulasi rekomendasi course berikutnya setelah sistem mengetahui riwayat terbaru user.

Namun, untuk menguji *robustness* model terhadap skenario yang lebih ketat, evaluasi strict hanya menggunakan `train_core_enc` sebagai history ketika memprediksi target test. Dengan demikian, informasi validation target tidak digunakan sebagai bagian dari input test. Perbandingan antara default evaluation dan strict train-history evaluation menunjukkan seberapa besar performa model bergantung pada riwayat tambahan yang muncul setelah train core.

Dalam penulisan paper, default pretest-history evaluation sebaiknya dijelaskan sebagai skenario operasional, sedangkan strict train-history evaluation dijelaskan sebagai *robustness check*. Jika performa strict masih mendekati performa default, maka model dapat dikatakan stabil terhadap pembatasan history input. Jika performa strict turun tajam, maka hasil tersebut perlu dilaporkan sebagai indikasi bahwa kualitas rekomendasi sangat dipengaruhi oleh kelengkapan dan kedekatan temporal riwayat user.

## 34. Visualisasi Distribusi Data dan Hasil

**Penjelasan:** Bagian ini menambahkan visualisasi distribusi data dan distribusi hasil rekomendasi. Visualisasi ini penting untuk paper karena pembaca perlu melihat karakteristik dataset, tingkat ketimpangan popularitas course, distribusi target test, serta pola course yang sering muncul pada hasil rekomendasi model.

**Tujuan:**

1. Menunjukkan distribusi jumlah interaksi per user pada data training.
2. Menunjukkan distribusi popularitas course pada data training.
3. Menampilkan ringkasan split train/validation/test secara eksplisit.
4. Menampilkan distribusi ground-truth course pada test set.
5. Menampilkan distribusi course yang paling sering direkomendasikan oleh model.

**Catatan leakage-safe:** Seluruh visualisasi yang bersifat statistik training dihitung dari `train_core_enc`. Distribusi target test hanya dipakai untuk analisis/evaluasi, bukan untuk membentuk fitur model atau melakukan fitting encoder/scaler.

In [ ]:
# ============================================================
# VISUALISASI DISTRIBUSI DATA DAN HASIL
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path

PAPER_FIGURE_DIR = Path("paper_ready_figures")
PAPER_FIGURE_DIR.mkdir(exist_ok=True)


def save_and_show_plot(filename):
    """Menyimpan plot agar bisa dipakai ulang untuk laporan/paper."""
    path = PAPER_FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return str(path)


# ------------------------------------------------------------
# 1. Ringkasan split dataset
# ------------------------------------------------------------
def summarize_split(split_name, df):
    return {
        "split": split_name,
        "rows": int(len(df)),
        "unique_users": int(df["user_id"].nunique()) if "user_id" in df.columns else 0,
        "unique_courses": int(df["canonical_course_id"].nunique()) if "canonical_course_id" in df.columns else 0
    }


data_split_summary_df = pd.DataFrame([
    summarize_split("Train core", train_core_enc),
    summarize_split("Validation target", val_target_enc),
    summarize_split("Pretest history", pretest_history_enc),
    summarize_split("Test target", test_target_enc)
])

print("=== Ringkasan Split Dataset ===")
display(data_split_summary_df)

plt.figure(figsize=(8, 4))
plt.bar(data_split_summary_df["split"], data_split_summary_df["rows"])
plt.title("Jumlah Baris per Split Dataset")
plt.xlabel("Split Dataset")
plt.ylabel("Jumlah Baris")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)
split_rows_plot_path = save_and_show_plot("01_split_rows_distribution.png")


# ------------------------------------------------------------
# 2. Distribusi jumlah interaksi per user pada train_core
# ------------------------------------------------------------
interaction_per_user_df = (
    train_core_enc
    .groupby("user_id")
    .size()
    .reset_index(name="train_interaction_count")
)

interaction_per_user_summary_df = (
    interaction_per_user_df["train_interaction_count"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .to_frame()
    .T
)
interaction_per_user_summary_df.index = ["train_interaction_count"]

print("=== Statistik Jumlah Interaksi per User pada Train Core ===")
display(interaction_per_user_summary_df.round(2))

plt.figure(figsize=(8, 4))
plt.hist(interaction_per_user_df["train_interaction_count"], bins=30)
plt.title("Distribusi Jumlah Interaksi per User pada Train Core")
plt.xlabel("Jumlah Interaksi per User")
plt.ylabel("Jumlah User")
plt.grid(axis="y", alpha=0.3)
interaction_per_user_plot_path = save_and_show_plot("02_train_interactions_per_user_distribution.png")


# ------------------------------------------------------------
# 3. Distribusi popularitas course pada train_core
# ------------------------------------------------------------
course_popularity_train_df = (
    train_core_enc
    .groupby(["course_encoded", "canonical_course_id", "course_name"])
    .size()
    .reset_index(name="train_interaction_count")
    .sort_values("train_interaction_count", ascending=False)
    .reset_index(drop=True)
)
course_popularity_train_df["popularity_rank"] = np.arange(1, len(course_popularity_train_df) + 1)

course_popularity_summary_df = (
    course_popularity_train_df["train_interaction_count"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .to_frame()
    .T
)
course_popularity_summary_df.index = ["course_train_interaction_count"]

print("=== Statistik Popularitas Course pada Train Core ===")
display(course_popularity_summary_df.round(2))

print("=== Top-20 Course Terpopuler pada Train Core ===")
display(course_popularity_train_df.head(20))

top_course_popularity_plot_df = course_popularity_train_df.head(20).copy()
top_course_popularity_plot_df["course_label"] = (
    top_course_popularity_plot_df["course_name"].astype(str).str.slice(0, 45)
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_course_popularity_plot_df["course_label"][::-1],
    top_course_popularity_plot_df["train_interaction_count"][::-1]
)
plt.title("Top-20 Course Terpopuler pada Train Core")
plt.xlabel("Jumlah Interaksi Train")
plt.ylabel("Course")
plt.grid(axis="x", alpha=0.3)
train_course_popularity_plot_path = save_and_show_plot("03_top20_train_course_popularity.png")


# ------------------------------------------------------------
# 4. Distribusi ground-truth course pada test set
# ------------------------------------------------------------
if "test_target_distribution" in globals():
    test_target_visual_df = test_target_distribution.copy()
else:
    test_target_visual_df = (
        test_metadata
        .groupby(["ground_truth_encoded", "ground_truth_course"])
        .size()
        .reset_index(name="test_target_count")
        .sort_values("test_target_count", ascending=False)
        .reset_index(drop=True)
    )

print("=== Top-20 Ground-Truth Course pada Test Set ===")
display(test_target_visual_df.head(20))

top_test_target_plot_df = test_target_visual_df.head(20).copy()
top_test_target_plot_df["course_label"] = (
    top_test_target_plot_df["ground_truth_course"].astype(str).str.slice(0, 45)
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_test_target_plot_df["course_label"][::-1],
    top_test_target_plot_df["test_target_count"][::-1]
)
plt.title("Top-20 Ground-Truth Course pada Test Set")
plt.xlabel("Jumlah Kemunculan sebagai Target Test")
plt.ylabel("Course")
plt.grid(axis="x", alpha=0.3)
test_target_distribution_plot_path = save_and_show_plot("04_top20_test_target_distribution.png")


# ------------------------------------------------------------
# 5. Distribusi course yang sering direkomendasikan model
# ------------------------------------------------------------
def build_recommendation_frequency(top_indices, ranking_method, catalog_df, k=10):
    """Menghitung frekuensi course muncul pada daftar Top-K rekomendasi."""
    rows = []

    for sample_idx, recommendation_row in enumerate(top_indices):
        for rank, course_encoded in enumerate(recommendation_row[:k], start=1):
            course_encoded = int(course_encoded)
            if course_encoded == 0:
                continue
            rows.append({
                "sample_index": sample_idx,
                "rank": rank,
                "course_encoded": course_encoded
            })

    if len(rows) == 0:
        return pd.DataFrame(columns=[
            "ranking_method",
            "course_encoded",
            "recommendation_count",
            "average_rank",
            "canonical_course_id",
            "course_name",
            "kompetensi",
            "level"
        ])

    raw_df = pd.DataFrame(rows)

    frequency_df = (
        raw_df
        .groupby("course_encoded")
        .agg(
            recommendation_count=("course_encoded", "size"),
            average_rank=("rank", "mean")
        )
        .reset_index()
        .sort_values(["recommendation_count", "average_rank"], ascending=[False, True])
    )

    catalog_columns = [
        "course_encoded",
        "canonical_course_id",
        "course_name",
        "kompetensi",
        "level"
    ]
    available_catalog_columns = [col for col in catalog_columns if col in catalog_df.columns]

    frequency_df = frequency_df.merge(
        catalog_df[available_catalog_columns].drop_duplicates("course_encoded"),
        on="course_encoded",
        how="left"
    )

    frequency_df.insert(0, "ranking_method", ranking_method)
    frequency_df["recommendation_share_percent"] = (
        frequency_df["recommendation_count"] / max(len(top_indices) * k, 1) * 100
    )
    frequency_df["average_rank"] = frequency_df["average_rank"].round(2)
    frequency_df["recommendation_share_percent"] = frequency_df["recommendation_share_percent"].round(2)

    return frequency_df.reset_index(drop=True)


softmax_recommendation_frequency_df = build_recommendation_frequency(
    softmax_top10_indices,
    ranking_method="Softmax Ranking",
    catalog_df=course_catalog,
    k=10
)

similarity_recommendation_frequency_df = build_recommendation_frequency(
    similarity_top10_indices,
    ranking_method="Similarity Ranking",
    catalog_df=course_catalog,
    k=10
)

model_recommendation_frequency_df = pd.concat(
    [softmax_recommendation_frequency_df, similarity_recommendation_frequency_df],
    ignore_index=True
)

print("=== Top-20 Course yang Paling Sering Direkomendasikan — Similarity Ranking ===")
display(similarity_recommendation_frequency_df.head(20))

top_similarity_rec_plot_df = similarity_recommendation_frequency_df.head(20).copy()
top_similarity_rec_plot_df["course_label"] = (
    top_similarity_rec_plot_df["course_name"].astype(str).str.slice(0, 45)
)

plt.figure(figsize=(10, 6))
plt.barh(
    top_similarity_rec_plot_df["course_label"][::-1],
    top_similarity_rec_plot_df["recommendation_count"][::-1]
)
plt.title("Top-20 Course yang Paling Sering Direkomendasikan — Similarity Ranking")
plt.xlabel("Jumlah Kemunculan pada Top-10")
plt.ylabel("Course")
plt.grid(axis="x", alpha=0.3)
similarity_recommendation_frequency_plot_path = save_and_show_plot("05_top20_similarity_recommendation_frequency.png")


# ------------------------------------------------------------
# 6. Distribusi ranking ground truth pada hasil Top-10
# ------------------------------------------------------------
if "similarity_ground_truth_result" in globals():
    hit_rank_df = similarity_ground_truth_result.dropna(subset=["rank_ground_truth"]).copy()
    hit_rank_df["rank_ground_truth"] = hit_rank_df["rank_ground_truth"].astype(int)

    if not hit_rank_df.empty:
        rank_distribution_df = (
            hit_rank_df["rank_ground_truth"]
            .value_counts()
            .sort_index()
            .reset_index()
        )
        rank_distribution_df.columns = ["rank_ground_truth", "hit_count"]

        print("=== Distribusi Rank Ground Truth pada Hit@10 — Similarity Ranking ===")
        display(rank_distribution_df)

        plt.figure(figsize=(8, 4))
        plt.bar(rank_distribution_df["rank_ground_truth"], rank_distribution_df["hit_count"])
        plt.title("Distribusi Rank Ground Truth pada Hit@10 — Similarity Ranking")
        plt.xlabel("Rank Ground Truth")
        plt.ylabel("Jumlah Hit")
        plt.xticks(range(1, 11))
        plt.grid(axis="y", alpha=0.3)
        ground_truth_rank_plot_path = save_and_show_plot("06_similarity_ground_truth_rank_distribution.png")
    else:
        rank_distribution_df = pd.DataFrame(columns=["rank_ground_truth", "hit_count"])
        ground_truth_rank_plot_path = None
        print("Tidak ada Hit@10 pada Similarity Ranking, sehingga distribusi rank ground truth kosong.")
else:
    rank_distribution_df = pd.DataFrame(columns=["rank_ground_truth", "hit_count"])
    ground_truth_rank_plot_path = None
    print("similarity_ground_truth_result belum tersedia.")


visualization_summary_df = pd.DataFrame({
    "visualization": [
        "Split rows distribution",
        "Train interactions per user",
        "Top-20 train course popularity",
        "Top-20 test target distribution",
        "Top-20 similarity recommendation frequency",
        "Similarity ground-truth rank distribution"
    ],
    "artifact_path": [
        split_rows_plot_path,
        interaction_per_user_plot_path,
        train_course_popularity_plot_path,
        test_target_distribution_plot_path,
        similarity_recommendation_frequency_plot_path,
        ground_truth_rank_plot_path
    ]
})

print("=== Ringkasan File Visualisasi ===")
display(visualization_summary_df)


### Interpretasi Visualisasi Evaluasi

Visualisasi distribusi data digunakan untuk memahami karakteristik dataset sebelum hasil model diinterpretasikan. Distribusi jumlah interaksi per user menunjukkan apakah sebagian besar user memiliki riwayat course pendek atau panjang. Informasi ini penting karena sistem rekomendasi berbasis sequence seperti LSTM biasanya memperoleh sinyal yang lebih kuat pada user dengan riwayat yang lebih panjang.

Distribusi popularitas course pada data training digunakan untuk melihat apakah dataset bersifat imbalanced. Jika hanya sedikit course yang sangat dominan, maka model berpotensi terdorong untuk merekomendasikan course populer secara berulang. Oleh karena itu, visualisasi popularitas training perlu dibandingkan dengan distribusi hasil rekomendasi model.

Distribusi ground-truth target pada test set membantu menjelaskan tingkat kesulitan evaluasi. Jika target test tersebar pada banyak course, maka task rekomendasi menjadi lebih menantang dibandingkan jika target test terkonsentrasi pada sedikit course populer. Sementara itu, distribusi course yang paling sering direkomendasikan oleh model digunakan untuk melihat apakah hasil rekomendasi cukup beragam atau terlalu terkonsentrasi pada beberapa course tertentu.

Visualisasi rank ground truth pada hasil Top-10 memberikan gambaran lebih detail dibanding Hit@10 saja. Jika ground truth sering muncul pada rank awal, maka rekomendasi model lebih kuat secara praktis karena course yang benar berada dekat dengan posisi teratas daftar rekomendasi.

## 35. Popularity Sanity Check

**Penjelasan:** Bagian ini menambahkan baseline sederhana berbasis popularitas course. Baseline ini merekomendasikan course paling populer pada `train_core_enc`, kemudian course yang sudah pernah diambil user di-mask agar tidak direkomendasikan ulang.

**Tujuan:** Mengecek apakah model benar-benar menghasilkan rekomendasi yang lebih personal daripada sekadar mengikuti course populer. Jika performa model hanya setara atau lebih rendah dari baseline popularity, maka klaim personalisasi harus dibuat lebih hati-hati. Sebaliknya, jika model lebih baik dan overlap rekomendasinya tidak terlalu tinggi dengan baseline popularity, maka model memiliki bukti tambahan bahwa ranking-nya menangkap pola sequence/user history.

**Catatan leakage-safe:** Popularity baseline hanya dihitung dari data training (`train_core_enc`). Data validation/test tidak dipakai untuk menentukan course populer.

In [ ]:
# ============================================================
# POPULARITY SANITY CHECK
# ============================================================


def build_popularity_score_matrix(train_df, n_samples, num_courses):
    """
    Membuat score matrix popularity baseline dari train_core_enc.

    Setiap sample mendapat skor popularity yang sama, lalu masking course yang
    sudah pernah diambil user dilakukan oleh evaluate_topk_ranking_masked().
    """
    popularity_counts = (
        train_df["course_encoded"]
        .astype(int)
        .value_counts()
        .to_dict()
    )

    popularity_scores = np.zeros(num_courses, dtype="float32")

    for course_encoded, count in popularity_counts.items():
        if 0 <= int(course_encoded) < num_courses:
            popularity_scores[int(course_encoded)] = float(count)

    # PAD bukan kandidat rekomendasi.
    popularity_scores[0] = -np.inf

    popularity_score_matrix = np.tile(popularity_scores, (n_samples, 1))
    return popularity_score_matrix


popularity_scores_test = build_popularity_score_matrix(
    train_df=train_core_enc,
    n_samples=len(y_test),
    num_courses=num_courses
)

popularity_metrics, popularity_metrics_df, popularity_top10_indices = evaluate_topk_ranking_masked(
    popularity_scores_test,
    X_course_test,
    y_test,
    k_values=(1, 5, 10)
)

print("=== Popularity Baseline Top-K Ranking Metrics ===")
display(
    popularity_metrics_df.assign(
        score=popularity_metrics_df["score"].round(4),
        percentage=popularity_metrics_df["percentage"].round(2)
    )
)


# ------------------------------------------------------------
# Perbandingan metrik model vs popularity baseline
# ------------------------------------------------------------
def add_method_label(metrics_df, method_name):
    df = metrics_df.copy()
    df.insert(0, "method", method_name)
    return df


popularity_metrics_comparison_df = pd.concat([
    add_method_label(popularity_metrics_df, "Popularity Baseline"),
    add_method_label(softmax_metrics_df, "LSTM + Attention Softmax"),
    add_method_label(similarity_metrics_df, "LSTM + Attention Similarity")
], ignore_index=True)

popularity_metrics_comparison_display = popularity_metrics_comparison_df.copy()
popularity_metrics_comparison_display["score"] = popularity_metrics_comparison_display["score"].round(4)
popularity_metrics_comparison_display["percentage"] = popularity_metrics_comparison_display["percentage"].round(2)

print("=== Perbandingan Metrik Model vs Popularity Baseline ===")
display(popularity_metrics_comparison_display)

metric_order = ["Hit@1", "Hit@5", "Hit@10", "MRR@10", "NDCG@10"]
comparison_plot_df = popularity_metrics_comparison_df[
    popularity_metrics_comparison_df["metric"].isin(metric_order)
].copy()
comparison_plot_df["metric"] = pd.Categorical(
    comparison_plot_df["metric"],
    categories=metric_order,
    ordered=True
)
comparison_plot_df = comparison_plot_df.sort_values(["metric", "method"])

plt.figure(figsize=(10, 5))
plot_pivot = comparison_plot_df.pivot_table(
    index="metric",
    columns="method",
    values="percentage",
    aggfunc="first"
)
plot_pivot.plot(kind="bar", figsize=(10, 5))
plt.title("Model vs Popularity Baseline")
plt.xlabel("Metrik")
plt.ylabel("Score (%)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
popularity_metric_plot_path = save_and_show_plot("07_model_vs_popularity_baseline_metrics.png")


# ------------------------------------------------------------
# Overlap rekomendasi model dengan popularity baseline
# ------------------------------------------------------------
def calculate_topk_overlap(reference_topk, model_topk, k=10, model_name="Model"):
    rows = []

    for i in range(len(model_topk)):
        reference_set = set(int(x) for x in reference_topk[i, :k] if int(x) != 0)
        model_set = set(int(x) for x in model_topk[i, :k] if int(x) != 0)
        intersection_count = len(reference_set.intersection(model_set))
        union_count = len(reference_set.union(model_set))

        rows.append({
            "sample_index": i,
            "model": model_name,
            "overlap_count_at_10": intersection_count,
            "overlap_percent_at_10": intersection_count / max(k, 1) * 100,
            "jaccard_at_10": intersection_count / union_count if union_count > 0 else 0.0
        })

    return pd.DataFrame(rows)


softmax_popularity_overlap_df = calculate_topk_overlap(
    popularity_top10_indices,
    softmax_top10_indices,
    k=10,
    model_name="LSTM + Attention Softmax"
)

similarity_popularity_overlap_df = calculate_topk_overlap(
    popularity_top10_indices,
    similarity_top10_indices,
    k=10,
    model_name="LSTM + Attention Similarity"
)

popularity_overlap_df = pd.concat(
    [softmax_popularity_overlap_df, similarity_popularity_overlap_df],
    ignore_index=True
)

popularity_overlap_summary_df = (
    popularity_overlap_df
    .groupby("model")
    .agg(
        mean_overlap_count_at_10=("overlap_count_at_10", "mean"),
        median_overlap_count_at_10=("overlap_count_at_10", "median"),
        mean_overlap_percent_at_10=("overlap_percent_at_10", "mean"),
        mean_jaccard_at_10=("jaccard_at_10", "mean")
    )
    .reset_index()
)

popularity_overlap_summary_display = popularity_overlap_summary_df.copy()
for col in [
    "mean_overlap_count_at_10",
    "median_overlap_count_at_10",
    "mean_overlap_percent_at_10",
    "mean_jaccard_at_10"
]:
    popularity_overlap_summary_display[col] = popularity_overlap_summary_display[col].round(4)

print("=== Overlap Top-10 Model dengan Popularity Baseline ===")
display(popularity_overlap_summary_display)

plt.figure(figsize=(8, 4))
plt.hist(
    similarity_popularity_overlap_df["overlap_count_at_10"],
    bins=np.arange(-0.5, 11.5, 1)
)
plt.title("Distribusi Overlap Top-10 Similarity Ranking vs Popularity Baseline")
plt.xlabel("Jumlah Course yang Sama dalam Top-10")
plt.ylabel("Jumlah Sample Test")
plt.xticks(range(0, 11))
plt.grid(axis="y", alpha=0.3)
popularity_overlap_plot_path = save_and_show_plot("08_similarity_vs_popularity_overlap_distribution.png")


# ------------------------------------------------------------
# Course yang sering direkomendasikan oleh popularity baseline
# ------------------------------------------------------------
popularity_recommendation_frequency_df = build_recommendation_frequency(
    popularity_top10_indices,
    ranking_method="Popularity Baseline",
    catalog_df=course_catalog,
    k=10
)

print("=== Top-20 Course yang Paling Sering Direkomendasikan — Popularity Baseline ===")
display(popularity_recommendation_frequency_df.head(20))


# ------------------------------------------------------------
# Delta model terhadap popularity baseline
# ------------------------------------------------------------
popularity_baseline_scores = popularity_metrics_df.set_index("metric")["score"]
similarity_scores = similarity_metrics_df.set_index("metric")["score"]
softmax_scores = softmax_metrics_df.set_index("metric")["score"]

popularity_delta_rows = []
for metric in metric_order:
    if metric in popularity_baseline_scores.index:
        popularity_delta_rows.append({
            "metric": metric,
            "popularity_baseline": popularity_baseline_scores.get(metric, np.nan),
            "softmax_model": softmax_scores.get(metric, np.nan),
            "similarity_model": similarity_scores.get(metric, np.nan),
            "softmax_minus_popularity": softmax_scores.get(metric, np.nan) - popularity_baseline_scores.get(metric, np.nan),
            "similarity_minus_popularity": similarity_scores.get(metric, np.nan) - popularity_baseline_scores.get(metric, np.nan)
        })

popularity_delta_df = pd.DataFrame(popularity_delta_rows)
popularity_delta_display = popularity_delta_df.copy()
for col in popularity_delta_display.columns:
    if col != "metric":
        popularity_delta_display[col] = popularity_delta_display[col].round(4)

popularity_delta_display["softmax_minus_popularity_pp"] = (
    popularity_delta_df["softmax_minus_popularity"] * 100
).round(2)
popularity_delta_display["similarity_minus_popularity_pp"] = (
    popularity_delta_df["similarity_minus_popularity"] * 100
).round(2)

print("=== Delta Model terhadap Popularity Baseline ===")
display(popularity_delta_display)


### Interpretasi Popularity Baseline

Popularity sanity check digunakan untuk memastikan bahwa sistem rekomendasi tidak hanya meniru course yang paling populer di data training. Baseline popularity dihitung dengan cara mengurutkan course berdasarkan frekuensi kemunculan pada `train_core_enc`, lalu menghapus course yang sudah pernah diambil user dari kandidat rekomendasi.

Perbandingan metrik antara model LSTM + Attention dan popularity baseline menunjukkan apakah model sequence memberikan nilai tambah terhadap baseline sederhana. Jika Hit@10, MRR@10, atau NDCG@10 model lebih tinggi daripada popularity baseline, maka model memiliki indikasi bahwa pola riwayat user membantu proses ranking. Jika nilainya mirip, interpretasi hasil harus lebih hati-hati karena sebagian performa mungkin berasal dari bias popularitas course.

Analisis overlap Top-10 digunakan untuk melihat kemiripan daftar rekomendasi model dengan daftar rekomendasi populer. Overlap yang sangat tinggi menunjukkan bahwa model cenderung merekomendasikan item populer, sedangkan overlap yang lebih moderat menunjukkan adanya diferensiasi ranking berdasarkan riwayat user. Dengan demikian, sanity check ini memperkuat transparansi evaluasi dan membantu membedakan performa personalisasi dari efek popularitas dataset.

## 36. Repeated Run Stability Check

**Penjelasan:** Bagian ini menjalankan model dengan konfigurasi utama pada beberapa seed berbeda. Tujuannya adalah melihat kestabilan metrik ranking, bukan mengganti hasil utama yang sudah dilaporkan sebelumnya.

**Mengapa penting:** Model deep learning dapat menghasilkan nilai metrik yang sedikit berbeda ketika inisialisasi bobot berubah. Dengan repeated runs, laporan/paper dapat menunjukkan apakah performa model relatif stabil atau sangat bergantung pada satu kali training saja.

**Catatan komputasi:** Bagian ini melatih beberapa model tambahan. Jika runtime Colab terbatas, jumlah seed atau jumlah epoch dapat dikurangi tanpa mengubah pipeline utama.

In [ ]:
# ============================================================
# REPEATED RUN STABILITY CHECK
# ============================================================

import gc

REPEATED_RUN_SEEDS = [42, 101, 202]
ADDITIONAL_EXPERIMENT_EPOCHS = 25
ADDITIONAL_EXPERIMENT_BATCH_SIZE = 64
ADDITIONAL_EXPERIMENT_PATIENCE = 3


def get_full_feature_inputs(split_name):
    """Mengambil input full-feature sesuai split data."""
    if split_name == "train":
        return [
            X_course_train,
            X_competency_train,
            X_level_train,
            X_course_has_certificate_train,
            X_certificate_count_train
        ]

    if split_name == "val":
        return [
            X_course_val,
            X_competency_val,
            X_level_val,
            X_course_has_certificate_val,
            X_certificate_count_val
        ]

    if split_name == "test":
        return [
            X_course_test,
            X_competency_test,
            X_level_test,
            X_course_has_certificate_test,
            X_certificate_count_test
        ]

    raise ValueError("split_name harus bernilai 'train', 'val', atau 'test'.")


def get_history_and_target_for_split(split_name):
    """Mengambil history course dan target sesuai split evaluasi."""
    if split_name == "val":
        return X_course_val, y_val

    if split_name == "test":
        return X_course_test, y_test

    raise ValueError("split_name evaluasi harus bernilai 'val' atau 'test'.")


def build_configurable_lstm_attention_model(experiment_id, config):
    """Membangun model Encoder-Decoder LSTM + attention dengan konfigurasi eksplisit."""
    if "make_safe_layer_name" in globals():
        prefix = make_safe_layer_name(experiment_id)
    else:
        prefix = "".join(ch.lower() if ch.isalnum() else "_" for ch in experiment_id)
        prefix = "_".join([part for part in prefix.split("_") if part])

    local_course_embedding_dim = int(config.get("course_embedding_dim", course_embedding_dim))
    local_competency_embedding_dim = int(config.get("competency_embedding_dim", competency_embedding_dim))
    local_level_embedding_dim = int(config.get("level_embedding_dim", level_embedding_dim))
    local_latent_dim = int(config.get("latent_dim", latent_dim))
    local_dropout_rate = float(config.get("dropout_rate", 0.3))
    local_learning_rate = float(config.get("learning_rate", 0.001))

    local_course_input = Input(shape=(max_sequence_length,), name=f"{prefix}_course_input")
    local_course_embedding = Embedding(
        input_dim=num_courses,
        output_dim=local_course_embedding_dim,
        name=f"{prefix}_course_embedding"
    )(local_course_input)

    local_competency_input = Input(shape=(max_sequence_length,), name=f"{prefix}_competency_input")
    local_competency_embedding = Embedding(
        input_dim=num_competencies,
        output_dim=local_competency_embedding_dim,
        name=f"{prefix}_competency_embedding"
    )(local_competency_input)

    local_level_input = Input(shape=(max_sequence_length,), name=f"{prefix}_level_input")
    local_level_embedding = Embedding(
        input_dim=num_levels,
        output_dim=local_level_embedding_dim,
        name=f"{prefix}_level_embedding"
    )(local_level_input)

    local_certificate_flag_input = Input(
        shape=(max_sequence_length, 1),
        name=f"{prefix}_course_has_certificate_input"
    )

    local_certificate_count_input = Input(
        shape=(max_sequence_length, 1),
        name=f"{prefix}_certificate_count_input"
    )

    local_combined_input = Concatenate(name=f"{prefix}_combined_encoder_input")([
        local_course_embedding,
        local_competency_embedding,
        local_level_embedding,
        local_certificate_flag_input,
        local_certificate_count_input
    ])

    local_encoder_sequence_output, local_state_h, local_state_c = LSTM(
        local_latent_dim,
        return_sequences=True,
        return_state=True,
        name=f"{prefix}_encoder_lstm"
    )(local_combined_input)

    local_attention_context, _ = TemporalAttention(name=f"{prefix}_temporal_attention")([
        local_encoder_sequence_output,
        local_course_input
    ])

    local_attention_enhanced_state = Concatenate(name=f"{prefix}_attention_state_concatenate")([
        local_state_h,
        local_attention_context
    ])

    local_attention_enhanced_state = Dense(
        local_latent_dim,
        activation="tanh",
        name=f"{prefix}_attention_enhanced_state"
    )(local_attention_enhanced_state)

    local_decoder_input = RepeatVector(1, name=f"{prefix}_decoder_repeat")(
        local_attention_enhanced_state
    )

    local_decoder_output = LSTM(
        local_latent_dim,
        return_sequences=False,
        name=f"{prefix}_decoder_lstm"
    )(
        local_decoder_input,
        initial_state=[local_attention_enhanced_state, local_state_c]
    )

    local_decoder_representation = Dropout(
        local_dropout_rate,
        name=f"{prefix}_decoder_representation"
    )(local_decoder_output)

    local_course_prediction = Dense(
        num_courses,
        activation="softmax",
        name=f"{prefix}_course_prediction"
    )(local_decoder_representation)

    local_model = Model(
        inputs=[
            local_course_input,
            local_competency_input,
            local_level_input,
            local_certificate_flag_input,
            local_certificate_count_input
        ],
        outputs=local_course_prediction,
        name=f"{prefix}_model"
    )

    local_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=local_learning_rate),
        loss="sparse_categorical_crossentropy"
    )

    return local_model, prefix


def train_and_evaluate_configurable_model(experiment_id, config, seed, evaluation_split="test"):
    """Melatih model tambahan dan mengevaluasi metrik ranking pada split tertentu."""
    tf.keras.utils.set_random_seed(int(seed))

    experiment_model, experiment_prefix = build_configurable_lstm_attention_model(
        experiment_id=experiment_id,
        config=config
    )

    train_inputs = get_full_feature_inputs("train")
    val_inputs = get_full_feature_inputs("val")

    experiment_early_stop = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=ADDITIONAL_EXPERIMENT_PATIENCE,
        restore_best_weights=True
    )

    experiment_reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5
    )

    experiment_history = experiment_model.fit(
        train_inputs,
        y_train,
        validation_data=(val_inputs, y_val),
        epochs=int(config.get("epochs", ADDITIONAL_EXPERIMENT_EPOCHS)),
        batch_size=int(config.get("batch_size", ADDITIONAL_EXPERIMENT_BATCH_SIZE)),
        callbacks=[experiment_early_stop, experiment_reduce_lr],
        shuffle=True,
        verbose=0
    )

    eval_inputs = get_full_feature_inputs(evaluation_split)
    eval_history_course, eval_target = get_history_and_target_for_split(evaluation_split)

    eval_scores = experiment_model.predict(eval_inputs, verbose=0)

    eval_metrics, _, eval_top10_indices = evaluate_topk_ranking_masked(
        eval_scores,
        eval_history_course,
        eval_target,
        k_values=(1, 5, 10)
    )

    eval_coverage, _, _ = calculate_coverage_at_k(
        eval_top10_indices,
        course_catalog,
        k=10,
        model_name=f"{experiment_id} - Softmax Ranking"
    )

    history_dict = experiment_history.history
    val_loss_values = history_dict.get("val_loss", [])
    train_loss_values = history_dict.get("loss", [])

    result_row = {
        "experiment": experiment_id,
        "seed": int(seed),
        "evaluation_split": evaluation_split,
        "epochs_trained": len(val_loss_values),
        "final_train_loss": float(train_loss_values[-1]) if len(train_loss_values) > 0 else np.nan,
        "best_val_loss": float(np.min(val_loss_values)) if len(val_loss_values) > 0 else np.nan,
        "Hit@1": eval_metrics["Hit@1"],
        "Hit@5": eval_metrics["Hit@5"],
        "Hit@10": eval_metrics["Hit@10"],
        "Recall@1": eval_metrics["Recall@1"],
        "Recall@5": eval_metrics["Recall@5"],
        "Recall@10": eval_metrics["Recall@10"],
        "MRR@10": eval_metrics["MRR@10"],
        "NDCG@10": eval_metrics["NDCG@10"],
        "Coverage@10": eval_coverage["coverage_at_k"]
    }

    del experiment_model
    gc.collect()

    return result_row, history_dict


main_experiment_config = {
    "course_embedding_dim": course_embedding_dim,
    "competency_embedding_dim": competency_embedding_dim,
    "level_embedding_dim": level_embedding_dim,
    "latent_dim": latent_dim,
    "dropout_rate": 0.3,
    "learning_rate": 0.001,
    "batch_size": ADDITIONAL_EXPERIMENT_BATCH_SIZE,
    "epochs": ADDITIONAL_EXPERIMENT_EPOCHS
}

repeated_run_rows = []
repeated_run_histories = {}

for run_seed in REPEATED_RUN_SEEDS:
    experiment_name = f"Main configuration seed {run_seed}"
    print("Menjalankan repeated run:", experiment_name)

    run_row, run_history = train_and_evaluate_configurable_model(
        experiment_id=experiment_name,
        config=main_experiment_config,
        seed=run_seed,
        evaluation_split="test"
    )

    repeated_run_rows.append(run_row)
    repeated_run_histories[experiment_name] = run_history

repeated_run_summary_df = pd.DataFrame(repeated_run_rows)

repeated_run_metric_columns = [
    "Hit@1",
    "Hit@5",
    "Hit@10",
    "Recall@1",
    "Recall@5",
    "Recall@10",
    "MRR@10",
    "NDCG@10",
    "Coverage@10"
]

repeated_run_display_df = repeated_run_summary_df.copy()
for col in repeated_run_metric_columns:
    repeated_run_display_df[col] = (repeated_run_display_df[col] * 100).round(2)

for col in ["final_train_loss", "best_val_loss"]:
    repeated_run_display_df[col] = repeated_run_display_df[col].round(4)

repeated_run_aggregate_df = (
    repeated_run_summary_df[repeated_run_metric_columns]
    .agg(["mean", "std", "min", "max"])
    .reset_index()
    .rename(columns={"index": "statistic"})
)

repeated_run_aggregate_display_df = repeated_run_aggregate_df.copy()
for col in repeated_run_metric_columns:
    repeated_run_aggregate_display_df[col] = (repeated_run_aggregate_display_df[col] * 100).round(2)

print("=== Repeated Run Summary - Softmax Ranking on Test Split ===")
display(repeated_run_display_df)

print("=== Repeated Run Aggregate ===")
display(repeated_run_aggregate_display_df)


### Analisis Stabilitas Model

Repeated run stability check digunakan untuk melihat konsistensi performa model ketika training dijalankan dengan seed berbeda. Jika nilai Hit@10, MRR@10, dan NDCG@10 memiliki standar deviasi yang kecil, maka performa model dapat dianggap relatif stabil dan tidak hanya bergantung pada satu inisialisasi bobot.

Dalam laporan/paper, hasil utama tetap dapat menggunakan model utama, sedangkan tabel repeated runs digunakan sebagai bukti tambahan bahwa konfigurasi model memberikan performa yang konsisten pada beberapa percobaan. Jika standar deviasi relatif besar, bagian diskusi perlu menyebutkan bahwa model sensitif terhadap random initialization dan membutuhkan evaluasi lanjutan dengan jumlah run yang lebih banyak.

## 37. Hyperparameter Comparison

**Penjelasan:** Bagian ini membandingkan beberapa konfigurasi hyperparameter sederhana menggunakan validation split. Tujuannya adalah memberikan dasar pemilihan konfigurasi model, bukan melakukan tuning besar-besaran.

**Prinsip evaluasi:** Pemilihan konfigurasi dilakukan berdasarkan validation split agar test split tetap berperan sebagai evaluasi akhir. Dengan demikian, test set tidak dipakai untuk memilih hyperparameter.

**Catatan komputasi:** Bagian ini melatih beberapa model tambahan. Jumlah konfigurasi dibuat terbatas agar masih realistis dijalankan di Colab.

In [ ]:
# ============================================================
# HYPERPARAMETER COMPARISON
# ============================================================

hyperparameter_configs = [
    {
        "experiment_id": "H1 - Compact capacity",
        "course_embedding_dim": 64,
        "competency_embedding_dim": 16,
        "level_embedding_dim": 8,
        "latent_dim": 128,
        "dropout_rate": 0.3,
        "learning_rate": 0.001,
        "batch_size": 64,
        "epochs": ADDITIONAL_EXPERIMENT_EPOCHS
    },
    {
        "experiment_id": "H2 - Main capacity",
        "course_embedding_dim": 96,
        "competency_embedding_dim": 32,
        "level_embedding_dim": 8,
        "latent_dim": 192,
        "dropout_rate": 0.3,
        "learning_rate": 0.001,
        "batch_size": 64,
        "epochs": ADDITIONAL_EXPERIMENT_EPOCHS
    },
    {
        "experiment_id": "H3 - Higher regularization",
        "course_embedding_dim": 96,
        "competency_embedding_dim": 32,
        "level_embedding_dim": 8,
        "latent_dim": 192,
        "dropout_rate": 0.4,
        "learning_rate": 0.0007,
        "batch_size": 64,
        "epochs": ADDITIONAL_EXPERIMENT_EPOCHS
    }
]

hyperparameter_rows = []
hyperparameter_histories = {}

for config_index, config in enumerate(hyperparameter_configs, start=1):
    experiment_name = config["experiment_id"]
    print("Menjalankan hyperparameter comparison:", experiment_name)

    config_row, config_history = train_and_evaluate_configurable_model(
        experiment_id=experiment_name,
        config=config,
        seed=42,
        evaluation_split="val"
    )

    config_row.update({
        "course_embedding_dim": config["course_embedding_dim"],
        "competency_embedding_dim": config["competency_embedding_dim"],
        "level_embedding_dim": config["level_embedding_dim"],
        "latent_dim": config["latent_dim"],
        "dropout_rate": config["dropout_rate"],
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"]
    })

    hyperparameter_rows.append(config_row)
    hyperparameter_histories[experiment_name] = config_history

hyperparameter_comparison_df = pd.DataFrame(hyperparameter_rows)

hyperparameter_comparison_df = hyperparameter_comparison_df.sort_values(
    by=["Hit@10", "MRR@10", "NDCG@10", "best_val_loss"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

hyperparameter_comparison_display_df = hyperparameter_comparison_df.copy()
for col in repeated_run_metric_columns:
    hyperparameter_comparison_display_df[col] = (hyperparameter_comparison_display_df[col] * 100).round(2)

for col in ["final_train_loss", "best_val_loss", "dropout_rate", "learning_rate"]:
    hyperparameter_comparison_display_df[col] = hyperparameter_comparison_display_df[col].round(4)

hyperparameter_best_config_df = hyperparameter_comparison_df.head(1).copy()
hyperparameter_best_config_display_df = hyperparameter_comparison_display_df.head(1).copy()

print("=== Hyperparameter Comparison - Softmax Ranking on Validation Split ===")
display(hyperparameter_comparison_display_df)

print("=== Selected Configuration Based on Validation Split ===")
display(hyperparameter_best_config_display_df)


### Analisis Perbandingan Hyperparameter

Hyperparameter comparison digunakan untuk menunjukkan bahwa konfigurasi model tidak dipilih secara arbitrer. Perbandingan dilakukan pada validation split dengan beberapa konfigurasi kapasitas model dan regularisasi. Konfigurasi terbaik dipilih berdasarkan metrik ranking pada validation split, terutama Hit@10, MRR@10, dan NDCG@10.

Dalam laporan/paper, bagian ini dapat dijelaskan sebagai tuning sederhana atau sensitivity check terhadap kapasitas model. Karena jumlah konfigurasi dibuat terbatas, klaim yang aman adalah bahwa konfigurasi utama telah dibandingkan dengan beberapa alternatif dasar, bukan bahwa pencarian hyperparameter sudah bersifat menyeluruh.

## 38. Reproducibility, Save Artifact, dan Persistensi Output
**Penjelasan:** Bagian ini menambahkan konfigurasi reproducibility dan penyimpanan artifact penting dari eksperimen. Artifact yang disimpan meliputi model, konfigurasi eksperimen, encoder/mapping, scaler, metrik evaluasi, hasil visualisasi, dan tabel analisis pendukung.

**Tujuan:** Membuat eksperimen lebih mudah diulang, diverifikasi, dan dilaporkan. Untuk kebutuhan paper, bagian ini menunjukkan bahwa hasil eksperimen tidak hanya muncul di output notebook, tetapi juga disimpan sebagai artifact yang bisa diperiksa ulang.

**Catatan Colab:** Folder artifact yang dibuat di runtime Colab akan hilang ketika runtime di-reset. Karena itu, notebook ini juga menyediakan pembuatan file ZIP artifact dan opsi penyimpanan ke Google Drive agar hasil eksperimen lebih aman.

In [ ]:
# ============================================================
# REPRODUCIBILITY, BACKEND ARTIFACT EXPORT, AND MANIFEST
# ============================================================

import json
import platform
import shutil
import sys
from pathlib import Path

ARTIFACT_DIR = Path("course_recommender_artifacts")
TABLE_ARTIFACT_DIR = ARTIFACT_DIR / "tables"
MODEL_ARTIFACT_DIR = ARTIFACT_DIR / "model"
CONFIG_ARTIFACT_DIR = ARTIFACT_DIR / "config"

for directory in [ARTIFACT_DIR, TABLE_ARTIFACT_DIR, MODEL_ARTIFACT_DIR, CONFIG_ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def json_safe(value):
    """Convert numpy/pandas values into JSON-safe Python values."""
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def write_json(path, payload):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False, default=json_safe)


def require_runtime_object(name):
    if name not in globals():
        raise RuntimeError(
            f"Required object {name} is not available. Run the notebook training cells before exporting artifacts."
        )
    return globals()[name]


# ------------------------------------------------------------
# 1. Reproducible experiment and backend runtime configuration
# ------------------------------------------------------------
model = require_runtime_object("model")
course_catalog = require_runtime_object("course_catalog")
course_to_idx = require_runtime_object("course_to_idx")
idx_to_course = require_runtime_object("idx_to_course")
competency_to_idx = require_runtime_object("competency_to_idx")
level_to_idx = require_runtime_object("level_to_idx")
scaler = require_runtime_object("scaler")

if "decoder_vector_model" not in globals():
    decoder_vector_model = Model(
        inputs=model.inputs,
        outputs=model.get_layer("decoder_representation").output,
        name="decoder_vector_model"
    )

if "course_vectors" not in globals():
    course_output_weights = model.get_layer("course_prediction").get_weights()[0]
    course_vectors = course_output_weights.T

MIN_HISTORY_FOR_MODEL_INFERENCE = int(globals().get("MIN_HISTORY_FOR_MODEL_INFERENCE", 2))

experiment_config = {
    "random_seed_numpy": 42,
    "random_seed_tensorflow": 42,
    "max_sequence_length": int(max_sequence_length),
    "min_course_count_train": int(MIN_COURSE_COUNT_TRAIN) if "MIN_COURSE_COUNT_TRAIN" in globals() else None,
    "num_courses": int(num_courses),
    "num_courses_in_train_vocab": int(num_courses - 1),
    "num_competencies": int(num_competencies),
    "num_competencies_in_train_vocab": int(num_competencies - 1),
    "num_levels": int(num_levels),
    "num_levels_in_train_vocab": int(num_levels - 1),
    "course_embedding_dim": int(course_embedding_dim),
    "competency_embedding_dim": int(competency_embedding_dim),
    "level_embedding_dim": int(level_embedding_dim),
    "latent_dim": int(latent_dim),
    "ranking_output_primary": "similarity_ranking",
    "evaluation_protocol": "temporal_holdout_train_only_encoder_scaler_seen_course_masking",
    "python_version": sys.version,
    "platform": platform.platform(),
    "tensorflow_version": tf.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "custom_objects_for_loading": ["TemporalAttention"],
    "repeated_run_seeds": REPEATED_RUN_SEEDS if "REPEATED_RUN_SEEDS" in globals() else None,
    "additional_experiment_epochs": int(ADDITIONAL_EXPERIMENT_EPOCHS) if "ADDITIONAL_EXPERIMENT_EPOCHS" in globals() else None,
    "additional_experiment_batch_size": int(ADDITIONAL_EXPERIMENT_BATCH_SIZE) if "ADDITIONAL_EXPERIMENT_BATCH_SIZE" in globals() else None,
    "hyperparameter_config_count": len(hyperparameter_configs) if "hyperparameter_configs" in globals() else None,
}

production_config = {
    "artifact_version": "1.0.0",
    "model_family": "encoder_decoder_lstm_attention",
    "ranking_strategy": "decoder_vector_cosine_similarity_primary_softmax_secondary",
    "max_sequence_length": int(max_sequence_length),
    "min_history_for_model_inference": MIN_HISTORY_FOR_MODEL_INFERENCE,
    "num_courses": int(num_courses),
    "num_competencies": int(num_competencies),
    "num_levels": int(num_levels),
    "pad_token_index": 0,
    "unknown_token": UNKNOWN_TOKEN,
    "required_history_columns": ["canonical_course_id"],
    "recommended_history_columns": ["user_id", "sort_timestamp", "sort_interaction_id", "sort_row_order"],
    "optional_history_columns": [
        "course_encoded",
        "kompetensi",
        "level",
        "certificate_count_train",
        "course_has_certificate_train"
    ],
    "response_contract": {
        "recommendations": [
            "rank",
            "course_id",
            "recommended_course",
            "competency",
            "level",
            "certificate_status",
            "certificate_count_train",
            "similarity_score_0_1",
            "similarity_score_percent",
            "softmax_score",
            "softmax_score_percent",
            "train_popularity_count"
        ],
        "metadata": [
            "fallback_reason",
            "known_history_count",
            "unknown_history_count",
            "total_history_count",
            "history_status",
            "catalog_check_status",
            "catalog_check_message",
            "catalog_check_issues",
            "inference_error"
        ]
    }
}

experiment_config_path = CONFIG_ARTIFACT_DIR / "experiment_config.json"
production_config_path = CONFIG_ARTIFACT_DIR / "production_config.json"
write_json(experiment_config_path, experiment_config)
write_json(production_config_path, production_config)


# ------------------------------------------------------------
# 2. Save model, decoder-vector model, and similarity vectors
# ------------------------------------------------------------
model_path = MODEL_ARTIFACT_DIR / "encoder_decoder_lstm_attention.keras"
decoder_vector_model_path = MODEL_ARTIFACT_DIR / "decoder_vector_model.keras"
course_vectors_path = MODEL_ARTIFACT_DIR / "course_vectors.npy"

model_save_status = "saved"
model_save_error = None
decoder_model_save_status = "saved"
decoder_model_save_error = None
course_vectors_save_status = "saved"
course_vectors_save_error = None

try:
    model.save(model_path)
except Exception as e:
    model_save_status = "failed"
    model_save_error = str(e)

try:
    decoder_vector_model.save(decoder_vector_model_path)
except Exception as e:
    decoder_model_save_status = "failed"
    decoder_model_save_error = str(e)

try:
    np.save(course_vectors_path, np.asarray(course_vectors))
except Exception as e:
    course_vectors_save_status = "failed"
    course_vectors_save_error = str(e)


# ------------------------------------------------------------
# 3. Save JSON-safe encoder mappings and scaler parameters
# ------------------------------------------------------------
encoder_artifact = {
    "course_to_idx": {str(key): int(value) for key, value in course_to_idx.items()},
    "idx_to_course": {str(key): str(value) for key, value in idx_to_course.items()},
    "competency_to_idx": {str(key): int(value) for key, value in competency_to_idx.items()},
    "level_to_idx": {str(key): int(value) for key, value in level_to_idx.items()},
    "unknown_token": UNKNOWN_TOKEN,
    "num_courses": int(num_courses),
    "num_competencies": int(num_competencies),
    "num_levels": int(num_levels),
}

encoder_path = CONFIG_ARTIFACT_DIR / "encoder_mappings.json"
write_json(encoder_path, encoder_artifact)

scaler_artifact = {
    "feature": "certificate_count_train",
    "type": "min_max_scaler",
    "feature_range": [float(scaler.feature_range[0]), float(scaler.feature_range[1])],
    "data_min": float(scaler.data_min_[0]),
    "data_max": float(scaler.data_max_[0]),
    "data_range": float(scaler.data_range_[0]),
    "scale": float(scaler.scale_[0]),
    "min": float(scaler.min_[0]),
    "clip": True,
}

scaler_path = CONFIG_ARTIFACT_DIR / "certificate_count_scaler.json"
write_json(scaler_path, scaler_artifact)


# ------------------------------------------------------------
# 4. Save backend runtime tables and analysis tables
# ------------------------------------------------------------
course_catalog_export = course_catalog.copy()
course_catalog_export.to_csv(TABLE_ARTIFACT_DIR / "course_catalog.csv", index=False)

course_popularity_train_df = (
    train_core_enc
    .groupby("course_encoded")
    .size()
    .reset_index(name="train_popularity_count")
    .sort_values(["train_popularity_count", "course_encoded"], ascending=[False, True])
    .reset_index(drop=True)
)
course_popularity_train_df.to_csv(TABLE_ARTIFACT_DIR / "course_popularity_train.csv", index=False)

artifact_table_names = [
    "course_catalog",
    "course_popularity_train_df",
    "softmax_metrics_df",
    "similarity_metrics_df",
    "topk_performance_curve_df",
    "coverage_summary_df",
    "ablation_summary_df",
    "ablation_delta_display",
    "error_analysis_summary_display",
    "error_type_summary_display",
    "ranking_case_comparison_summary_display",
    "min_course_sensitivity_df",
    "min_course_sensitivity_display",
    "strict_metrics_comparison_df",
    "strict_metrics_comparison_display",
    "strict_delta_display",
    "data_split_summary_df",
    "interaction_per_user_summary_df",
    "course_popularity_summary_df",
    "test_target_distribution",
    "model_recommendation_frequency_df",
    "similarity_recommendation_frequency_df",
    "popularity_metrics_comparison_df",
    "popularity_metrics_comparison_display",
    "popularity_overlap_summary_df",
    "popularity_overlap_summary_display",
    "popularity_recommendation_frequency_df",
    "popularity_delta_df",
    "popularity_delta_display",
    "visualization_summary_df",
    "course_noise_summary_df",
    "course_noise_removed_df",
    "course_manual_review_df",
    "target_distribution_summary_df",
    "similarity_ground_truth_result",
    "softmax_ground_truth_result",
    "repeated_run_summary_df",
    "repeated_run_display_df",
    "repeated_run_aggregate_df",
    "repeated_run_aggregate_display_df",
    "hyperparameter_comparison_df",
    "hyperparameter_comparison_display_df",
    "hyperparameter_best_config_df",
    "hyperparameter_best_config_display_df",
    "production_readiness_summary_df",
    "inference_test_summary_df",
    "deployment_flow_df",
    "deployment_flow_summary_df",
]

saved_table_rows = []
seen_table_names = set()

for table_name in artifact_table_names:
    if table_name in seen_table_names:
        continue
    seen_table_names.add(table_name)

    obj = globals().get(table_name)
    if isinstance(obj, pd.DataFrame):
        table_path = TABLE_ARTIFACT_DIR / f"{table_name}.csv"
        obj.to_csv(table_path, index=False)
        saved_table_rows.append({
            "artifact_name": table_name,
            "artifact_type": "table_csv",
            "path": str(table_path),
            "status": "saved",
            "rows": int(len(obj)),
            "columns": int(len(obj.columns))
        })


# ------------------------------------------------------------
# 5. Manifest and ZIP bundle
# ------------------------------------------------------------
artifact_manifest_rows = [
    {
        "artifact_name": "experiment_config",
        "artifact_type": "json_config",
        "path": str(experiment_config_path),
        "status": "saved",
        "rows": None,
        "columns": None
    },
    {
        "artifact_name": "production_config",
        "artifact_type": "json_config",
        "path": str(production_config_path),
        "status": "saved",
        "rows": None,
        "columns": None
    },
    {
        "artifact_name": "encoder_mappings",
        "artifact_type": "json_mapping",
        "path": str(encoder_path),
        "status": "saved",
        "rows": None,
        "columns": None
    },
    {
        "artifact_name": "certificate_count_scaler",
        "artifact_type": "json_scaler",
        "path": str(scaler_path),
        "status": "saved",
        "rows": None,
        "columns": None
    },
    {
        "artifact_name": "encoder_decoder_lstm_attention",
        "artifact_type": "keras_model",
        "path": str(model_path),
        "status": model_save_status,
        "rows": None,
        "columns": None
    },
    {
        "artifact_name": "decoder_vector_model",
        "artifact_type": "keras_model",
        "path": str(decoder_vector_model_path),
        "status": decoder_model_save_status,
        "rows": None,
        "columns": None
    },
    {
        "artifact_name": "course_vectors",
        "artifact_type": "numpy_array",
        "path": str(course_vectors_path),
        "status": course_vectors_save_status,
        "rows": int(np.asarray(course_vectors).shape[0]) if course_vectors_save_status == "saved" else None,
        "columns": int(np.asarray(course_vectors).shape[1]) if course_vectors_save_status == "saved" and np.asarray(course_vectors).ndim == 2 else None
    }
]
artifact_manifest_rows.extend(saved_table_rows)

artifact_manifest_df = pd.DataFrame(artifact_manifest_rows)
artifact_manifest_path = ARTIFACT_DIR / "artifact_manifest.csv"
artifact_manifest_df.to_csv(artifact_manifest_path, index=False)

artifact_zip_status = "saved"
artifact_zip_error = None
artifact_zip_path = None

try:
    artifact_zip_path = Path(
        shutil.make_archive(
            base_name=str(ARTIFACT_DIR),
            format="zip",
            root_dir=str(ARTIFACT_DIR)
        )
    )
except Exception as e:
    artifact_zip_status = "failed"
    artifact_zip_error = str(e)

artifact_manifest_df.loc[len(artifact_manifest_df)] = {
    "artifact_name": "artifact_zip",
    "artifact_type": "zip_archive",
    "path": str(artifact_zip_path) if artifact_zip_path is not None else "",
    "status": artifact_zip_status,
    "rows": None,
    "columns": None
}
artifact_manifest_df.to_csv(artifact_manifest_path, index=False)

print("=== Backend Artifact Manifest ===")
display(artifact_manifest_df)

for label, error in [
    ("model save", model_save_error),
    ("decoder vector model save", decoder_model_save_error),
    ("course vectors save", course_vectors_save_error),
    ("artifact zip", artifact_zip_error),
]:
    if error is not None:
        print(f"{label} error: {error}")

if artifact_zip_path is not None:
    print("Artifact ZIP:", artifact_zip_path)

print("Backend integration module: course_recommender.CourseRecommenderService")
print("Backend integration contract: BACKEND_INTEGRATION.md")


### Reproducibility dan Penyimpanan Artifact
Reproducibility pada penelitian ini dijaga dengan menetapkan seed `numpy` dan `tensorflow`, menyimpan konfigurasi eksperimen, serta menyimpan mapping encoder dan scaler yang digunakan selama training. Hal ini penting karena sistem rekomendasi menggunakan transformasi train-only, sehingga mapping course, kompetensi, level, dan scaler harus konsisten ketika model dievaluasi ulang atau digunakan untuk inference.

Artifact yang disimpan meliputi model utama, konfigurasi eksperimen, mapping encoder, scaler, metrik evaluasi, hasil error analysis, hasil ablation, hasil strict evaluation, visualisasi, dan sanity check popularity. Dengan adanya artifact manifest, setiap output penting dari pipeline dapat dilacak kembali. Bagian ini memperkuat kesiapan notebook untuk kebutuhan pelaporan akademik maupun pengembangan lanjutan ke arah deployment.

## 39. Fungsi Menampilkan Riwayat User
**Penjelasan:** Bagian ini membuat fungsi `show_user_history()` untuk melihat riwayat course user berdasarkan `user_id`. Riwayat diurutkan berdasarkan timestamp, interaction ID, dan urutan baris.

**Yang perlu diperhatikan:** Fungsi ini dipakai pada interface manual agar sebelum melihat rekomendasi, pengguna bisa melihat dulu course apa saja yang sudah pernah diambil user tersebut.

In [ ]:
def show_user_history(user_id, source_df=pretest_history_enc):
    user_id = str(user_id).strip().replace(".0", "")

    history_df = source_df[source_df["user_id"] == user_id].copy()

    if history_df.empty:
        return "User ID tidak ditemukan pada source_df."

    history_df = history_df.sort_values(
        ["sort_timestamp", "sort_interaction_id", "sort_row_order"]
    )

    return history_df[[
        "user_id",
        "canonical_course_id",
        "course_name",
        "kompetensi",
        "level",
        "course_has_certificate_train",
        "certificate_count_train",
        "first_event_timestamp"
    ]]

## 40. Pipeline Prediksi Top-N Recommendation
**Penjelasan:** Bagian ini menjelaskan alur prediksi yang digunakan untuk menghasilkan rekomendasi course Top-N. Pipeline ini bukan klasifikasi satu label tunggal, melainkan **ranking-based recommendation**, yaitu model memberi skor pada seluruh kandidat course, lalu sistem memilih `N` course dengan skor tertinggi setelah course yang sudah pernah diambil user dihapus dari kandidat.

**Tujuan pipeline:** memprediksi course berikutnya yang paling relevan untuk seorang karyawan berdasarkan riwayat course yang sudah pernah diambil sebelumnya.

| Tahap | Proses | Output |
|---|---|---|
| 1 | User memasukkan `user_id` / `employee_id` secara manual | ID user yang akan diberi rekomendasi |
| 2 | Sistem mengambil riwayat course user dari `pretest_history_enc` dan mengurutkannya secara temporal | Sequence riwayat course user |
| 3 | Riwayat course diubah menjadi lima input model: `course_encoded`, `competency_encoded`, `level_encoded`, `course_has_certificate_train`, dan `certificate_count_scaled` | Tensor input model |
| 4 | Sequence dipotong/dipadatkan dengan `pad_sequences` sesuai `max_sequence_length` | Input berdimensi tetap |
| 5 | Model LSTM dengan attention layer menghasilkan skor softmax untuk seluruh course kandidat | `softmax_score` per course |
| 6 | Representasi user attention-enhanced dari layer `decoder_representation` dibandingkan dengan vektor course menggunakan cosine similarity | `similarity_score_0_1` per course |
| 7 | Course yang sudah pernah diambil user dan indeks PAD `0` di-mask agar tidak direkomendasikan ulang | Daftar kandidat bersih |
| 8 | Kandidat course diurutkan berdasarkan skor tertinggi, lalu diambil sebanyak `top_n` | Top-N rekomendasi course |
| 9 | Course encoded dikembalikan ke informasi asli dari `course_catalog` | Tabel rekomendasi akhir |

**Skema inference yang digunakan pada fungsi `recommend_courses()`:**

`user_id` → ambil riwayat user → sorting temporal → encoding fitur → padding sequence → prediksi skor model → hitung similarity user-course → masking course yang sudah diambil → ranking kandidat → ambil Top-N → tampilkan tabel rekomendasi.

**Catatan penting:** Pada notebook ini, output rekomendasi interaktif diurutkan menggunakan `similarity_score_0_1` sebagai skor ranking utama. Nilai `softmax_score` tetap ditampilkan sebagai skor pendukung dari output probabilitas model. Dengan demikian, tabel rekomendasi menunjukkan dua perspektif skor: kemiripan representasi user-course dan skor prediksi softmax dari model.

## 41. Prepare Input User untuk Inference
**Penjelasan:** Bagian ini menjalankan tahap awal pipeline prediksi Top-N. Fungsi `prepare_user_input()` mengambil riwayat course seorang user dari data history, mengurutkannya berdasarkan waktu, lalu mengubahnya menjadi format input yang sesuai dengan arsitektur model.

Fitur yang dimasukkan ke model terdiri dari:

1. `course_encoded`: urutan course yang sudah pernah diambil user;
2. `competency_encoded`: urutan kompetensi dari course tersebut;
3. `level_encoded`: urutan level course;
4. `course_has_certificate_train`: indikator apakah course memiliki sertifikat berdasarkan data training;
5. `certificate_count_scaled`: jumlah sertifikat course yang sudah di-scale berdasarkan statistik training.

Setiap sequence diproses dengan `pad_sequences` agar panjang input sama dengan `max_sequence_length`. Fungsi ini juga mengembalikan `taken_courses`, yaitu daftar course yang sudah pernah diambil user. Daftar ini diperlukan pada tahap masking agar sistem tidak merekomendasikan course yang sama kepada user.

**Output utama fungsi:** `model_input` untuk prediksi model dan `taken_courses` untuk proses filtering kandidat rekomendasi.

In [ ]:
def prepare_user_input(user_id, source_df=pretest_history_enc):
    user_id = str(user_id).strip().replace(".0", "")

    user_df = source_df[source_df["user_id"] == user_id].copy()

    if user_df.empty:
        return None, None

    user_df = user_df.sort_values(
        ["sort_timestamp", "sort_interaction_id", "sort_row_order"]
    )

    course_seq = user_df["course_encoded"].tolist()
    competency_seq = user_df["competency_encoded"].tolist()
    level_seq = user_df["level_encoded"].tolist()
    course_cert_seq = user_df["course_has_certificate_train"].tolist()
    cert_count_seq = user_df["certificate_count_scaled"].tolist()

    input_course = pad_sequences([course_seq], maxlen=max_sequence_length, padding="pre")
    input_competency = pad_sequences([competency_seq], maxlen=max_sequence_length, padding="pre")
    input_level = pad_sequences([level_seq], maxlen=max_sequence_length, padding="pre")

    input_course_cert = pad_sequences(
        [course_cert_seq],
        maxlen=max_sequence_length,
        padding="pre",
        dtype="float32"
    )

    input_cert_count = pad_sequences(
        [cert_count_seq],
        maxlen=max_sequence_length,
        padding="pre",
        dtype="float32"
    )

    input_course_cert = np.expand_dims(input_course_cert, axis=-1)
    input_cert_count = np.expand_dims(input_cert_count, axis=-1)

    model_input = [
        input_course,
        input_competency,
        input_level,
        input_course_cert,
        input_cert_count
    ]

    taken_courses = set(course_seq)

    return model_input, taken_courses

## 42. Fungsi Rekomendasi Course Berdasarkan User ID — Output Tabel Rapi
**Penjelasan:** Bagian ini menjalankan inti pipeline prediksi Top-N melalui fungsi `recommend_courses()`. Output rekomendasi dibuat dalam bentuk tabel yang rapi, sedangkan makna kolom `certificate_count_train` dijelaskan agar interpretasinya tepat.

Perbaikan output rekomendasi meliputi:

1. Menambahkan kolom `rank` agar urutan rekomendasi terlihat jelas.
2. Menampilkan identitas course secara eksplisit melalui `course_id` dan `recommended_course`.
3. Menampilkan metadata course: `competency`, `level`, `certificate_status`, dan `certificate_count_train`.
4. Menampilkan skor ranking utama `similarity_score_0_1` dan skor pendukung `softmax_score`.
5. Menambahkan versi persentase dari skor agar lebih mudah diinterpretasikan.
6. Mengurutkan tabel berdasarkan ranking utama, yaitu `similarity_score_0_1`.

**Interpretasi kolom `certificate_count_train` pada tabel rekomendasi:** kolom ini menunjukkan frekuensi course tersebut muncul sebagai course bersertifikat pada data training. Kolom ini **bukan** jumlah sertifikat milik user dan **bukan** hasil prediksi model. Nilai ini ditampilkan sebagai informasi tambahan untuk menjelaskan karakteristik course yang direkomendasikan.

**Output akhir:** tabel Top-N rekomendasi yang lebih rapi, konsisten, dan dapat digunakan untuk analisis hasil rekomendasi.

In [ ]:
def recommend_courses(user_id, top_n=10, source_df=pretest_history_enc):
    """
    Menghasilkan tabel Top-N rekomendasi course untuk user tertentu.

    Ranking utama menggunakan similarity_score_0_1, sedangkan softmax_score
    ditampilkan sebagai skor pendukung dari output probabilitas model.

    Catatan:
    certificate_count_train pada output adalah statistik course dari train_core,
    bukan jumlah sertifikat milik user dan bukan hasil prediksi model.
    """
    model_input, taken_courses = prepare_user_input(user_id, source_df=source_df)

    if model_input is None:
        return "User ID tidak ditemukan."

    softmax_score = model.predict(model_input, verbose=0)[0].copy()

    user_vector = decoder_vector_model.predict(model_input, verbose=0)
    similarity_score = calculate_similarity(user_vector, course_vectors)[0].copy()

    # Masking course yang sudah pernah diambil agar tidak direkomendasikan ulang.
    for taken in taken_courses:
        if taken < len(similarity_score):
            similarity_score[taken] = -np.inf
            softmax_score[taken] = -np.inf

    # Masking indeks 0 karena indeks tersebut adalah PAD, bukan course valid.
    similarity_score[0] = -np.inf
    softmax_score[0] = -np.inf

    ranking = np.argsort(similarity_score)[::-1]

    recommendations = []

    for idx in ranking:
        if idx == 0 or not np.isfinite(similarity_score[idx]):
            continue

        info = course_catalog[course_catalog["course_encoded"] == idx]

        if info.empty:
            continue

        row = info.iloc[0]

        has_certificate = int(row["course_has_certificate_train"])
        certificate_status = "Bersertifikat" if has_certificate == 1 else "Tidak Bersertifikat"

        similarity_value = float(similarity_score[idx])
        softmax_value = float(softmax_score[idx])

        recommendations.append({
            "rank": len(recommendations) + 1,
            "course_id": row["canonical_course_id"],
            "recommended_course": row["course_name"],
            "competency": row["kompetensi"],
            "level": row["level"],
            "certificate_status": certificate_status,
            "certificate_count_train": int(row["certificate_count_train"]),
            "similarity_score_0_1": round(similarity_value, 4),
            "similarity_score_percent": round(similarity_value * 100, 2),
            "softmax_score": round(softmax_value, 6),
            "softmax_score_percent": round(softmax_value * 100, 4)
        })

        if len(recommendations) >= top_n:
            break

    result = pd.DataFrame(recommendations)

    if result.empty:
        return "Rekomendasi tidak tersedia."

    ordered_columns = [
        "rank",
        "course_id",
        "recommended_course",
        "competency",
        "level",
        "certificate_status",
        "certificate_count_train",
        "similarity_score_0_1",
        "similarity_score_percent",
        "softmax_score",
        "softmax_score_percent"
    ]

    return result[ordered_columns]


def display_recommendation_table(recommendation_result):
    """
    Menampilkan tabel rekomendasi dengan format yang lebih rapi di notebook/Colab.
    Fungsi ini mendukung output model utama dan output fallback production-ready.
    """
    if isinstance(recommendation_result, str):
        print(recommendation_result)
        return

    if recommendation_result is None or len(recommendation_result) == 0:
        print("Rekomendasi tidak tersedia.")
        return

    display_df = recommendation_result.copy().rename(columns={
        "rank": "Rank",
        "course_id": "Course ID",
        "recommended_course": "Recommended Course",
        "competency": "Competency",
        "level": "Level",
        "certificate_status": "Certificate Status",
        "certificate_count_train": "Certificate Count from Train",
        "similarity_score_0_1": "Similarity Score (0-1)",
        "similarity_score_percent": "Similarity Score (%)",
        "softmax_score": "Softmax Score",
        "softmax_score_percent": "Softmax Score (%)",
        "train_popularity_count": "Train Popularity Count",
        "fallback_reason": "Fallback Reason",
        "known_history_count": "Known History Count",
        "unknown_history_count": "Unknown History Count",
        "total_history_count": "Total History Count",
        "catalog_check_status": "Catalog Check Status",
        "catalog_check_message": "Catalog Check Message"
    })

    format_rules = {}
    if "Similarity Score (0-1)" in display_df.columns:
        format_rules["Similarity Score (0-1)"] = "{:.4f}"
    if "Similarity Score (%)" in display_df.columns:
        format_rules["Similarity Score (%)"] = "{:.2f}%"
    if "Softmax Score" in display_df.columns:
        format_rules["Softmax Score"] = "{:.6f}"
    if "Softmax Score (%)" in display_df.columns:
        format_rules["Softmax Score (%)"] = "{:.4f}%"

    styler = display_df.style

    try:
        styler = styler.hide(axis="index")
    except Exception:
        try:
            styler = styler.hide_index()
        except Exception:
            pass

    styled_table = (
        styler
        .format(format_rules, na_rep="-")
        .set_properties(**{
            "text-align": "left",
            "white-space": "normal"
        })
        .set_table_styles([
            {"selector": "th", "props": [("text-align", "left")]},
            {"selector": "caption", "props": [("caption-side", "top"), ("font-weight", "bold")]}
        ])
        .set_caption("Top-N Course Recommendation Result")
    )

    display(styled_table)




### 42.1. Penjelasan
Dalam penelitian ini, `certificate_count_train` digunakan sebagai fitur tambahan berbasis course. Nilai tersebut dihitung sebagai jumlah kemunculan course pada data training (`train_core`) yang memiliki informasi sertifikat tersedia (`certificate_available_raw = 1`). Fitur ini tidak merepresentasikan jumlah sertifikat yang dimiliki oleh seorang karyawan, melainkan frekuensi historis ketersediaan sertifikat pada level course berdasarkan data training. Untuk menjaga validitas evaluasi dan mencegah data leakage, perhitungan `certificate_count_train` hanya menggunakan data training, sedangkan data validation dan test hanya menerima hasil transformasi dari statistik training tersebut.

Pada tahap pemodelan, `certificate_count_train` dinormalisasi menjadi `certificate_count_scaled` menggunakan `MinMaxScaler` yang di-fit pada data training saja. Fitur ini kemudian digunakan sebagai salah satu input sekuensial model LSTM bersama fitur course, kompetensi, level, dan indikator sertifikat. Pada tabel rekomendasi akhir, nilai `certificate_count_train` ditampilkan sebagai informasi pendukung agar karakteristik course yang direkomendasikan lebih mudah diinterpretasikan.

## 43. Deployment Fallback Handling
**Penjelasan:** Bagian ini menambahkan mekanisme fallback agar sistem rekomendasi tetap aman digunakan pada skenario deployment. Fallback dibutuhkan karena input di dunia nyata tidak selalu ideal. Sistem perlu tetap menghasilkan rekomendasi ketika menghadapi karyawan baru, riwayat course yang terlalu sedikit, course history yang tidak dikenali encoder, rekomendasi kosong, atau ketidaksesuaian antara model, encoder, dan katalog course.

Fallback utama yang digunakan adalah **popularity-based recommendation** dari `train_core_enc`, sehingga tetap leakage-safe karena hanya menggunakan informasi dari data training. Pada kondisi normal, sistem tetap menggunakan model LSTM dengan attention.

In [ ]:
# ============================================================
# PRODUCTION-READY FALLBACK HANDLING
# ============================================================

MIN_HISTORY_FOR_MODEL_INFERENCE = 2


def normalize_user_id_value(value):
    """Normalisasi User ID agar input manual seperti 123 dan 123.0 tetap terbaca sama."""
    return str(value).strip().replace(".0", "")


def sort_history_for_inference(history_df):
    """Mengurutkan history user dengan kolom waktu/urutan yang tersedia."""
    history_df = history_df.copy()

    sort_columns = [
        col
        for col in ["sort_timestamp", "sort_interaction_id", "sort_row_order"]
        if col in history_df.columns
    ]

    if len(sort_columns) > 0:
        return history_df.sort_values(sort_columns).reset_index(drop=True)

    if "first_event_timestamp" in history_df.columns:
        return history_df.sort_values("first_event_timestamp").reset_index(drop=True)

    return history_df.reset_index(drop=True)


def validate_catalog_consistency():
    """
    Mengecek kesesuaian antara model output, encoder, course vectors, dan course catalog.
    Fungsi ini penting untuk deployment karena artifact training dan katalog produksi bisa tidak sinkron.
    """
    issues = []
    critical_issues = []

    expected_num_courses = int(num_courses) if "num_courses" in globals() else None

    if "model" not in globals():
        critical_issues.append("Model belum tersedia di runtime.")
    elif expected_num_courses is not None:
        try:
            model_output_dim = int(model.output_shape[-1])
            if model_output_dim != expected_num_courses:
                critical_issues.append(
                    f"Model output dimension ({model_output_dim}) tidak sama dengan num_courses ({expected_num_courses})."
                )
        except Exception as e:
            issues.append(f"Model output dimension tidak dapat dicek: {e}")

    if "course_vectors" in globals() and expected_num_courses is not None:
        try:
            course_vector_rows = int(course_vectors.shape[0])
            if course_vector_rows != expected_num_courses:
                critical_issues.append(
                    f"Jumlah baris course_vectors ({course_vector_rows}) tidak sama dengan num_courses ({expected_num_courses})."
                )
        except Exception as e:
            issues.append(f"course_vectors tidak dapat dicek: {e}")

    if "course_catalog" not in globals():
        critical_issues.append("course_catalog belum tersedia di runtime.")
    else:
        if "course_encoded" not in course_catalog.columns:
            critical_issues.append("course_catalog tidak memiliki kolom course_encoded.")
        elif expected_num_courses is not None:
            catalog_encoded = pd.to_numeric(course_catalog["course_encoded"], errors="coerce").dropna().astype(int)
            catalog_encoded_set = set(catalog_encoded.tolist())
            expected_encoded_set = set(range(1, expected_num_courses))

            invalid_catalog_ids = sorted([
                int(x) for x in catalog_encoded_set
                if x <= 0 or x >= expected_num_courses
            ])

            missing_catalog_ids = sorted(list(expected_encoded_set - catalog_encoded_set))

            if len(invalid_catalog_ids) > 0:
                critical_issues.append(
                    f"Ada {len(invalid_catalog_ids)} course_encoded di course_catalog yang berada di luar range encoder."
                )

            if len(missing_catalog_ids) > 0:
                issues.append(
                    f"Ada {len(missing_catalog_ids)} course dari encoder yang tidak ditemukan di course_catalog."
                )

    all_issues = critical_issues + issues

    if len(critical_issues) > 0:
        status = "CRITICAL"
        message = "Ditemukan mismatch kritis. Sistem sebaiknya memakai fallback sampai artifact diperbaiki."
    elif len(issues) > 0:
        status = "WARNING"
        message = "Ditemukan potensi catalog mismatch non-kritis. Sistem masih dapat berjalan dengan catatan."
    else:
        status = "OK"
        message = "Model, encoder, course vectors, dan course catalog konsisten."

    return {
        "status": status,
        "message": message,
        "issues": all_issues,
        "critical": len(critical_issues) > 0
    }


def attach_inference_metadata(recommendation_df, audit_info, catalog_check, reason):
    """Menambahkan metadata fallback/audit pada output rekomendasi."""
    if isinstance(recommendation_df, str):
        return recommendation_df

    result = recommendation_df.copy()
    result["fallback_reason"] = reason
    result["known_history_count"] = int(audit_info.get("known_history_count", 0))
    result["unknown_history_count"] = int(audit_info.get("unknown_history_count", 0))
    result["total_history_count"] = int(audit_info.get("total_history_count", 0))
    result["catalog_check_status"] = catalog_check.get("status", "UNKNOWN")
    result["catalog_check_message"] = catalog_check.get("message", "")
    return result


def build_popularity_fallback_table(top_n=10, exclude_encoded=None):
    """
    Membuat fallback rekomendasi berbasis popularitas course dari train_core_enc.
    Course yang sudah pernah diambil user dapat dikeluarkan melalui exclude_encoded.
    """
    if exclude_encoded is None:
        exclude_encoded = set()

    exclude_encoded = set([
        int(x)
        for x in exclude_encoded
        if pd.notna(x)
    ])
    exclude_encoded.add(0)  # indeks 0 adalah PAD, bukan course valid

    popularity_df = (
        train_core_enc
        .groupby("course_encoded")
        .size()
        .reset_index(name="train_popularity_count")
        .sort_values("train_popularity_count", ascending=False)
    )

    popularity_df["course_encoded"] = pd.to_numeric(
        popularity_df["course_encoded"],
        errors="coerce"
    ).dropna().astype(int)

    popularity_df = popularity_df[
        ~popularity_df["course_encoded"].isin(exclude_encoded)
    ].copy()

    catalog_columns = [
        "course_encoded",
        "canonical_course_id",
        "course_name",
        "kompetensi",
        "level",
        "course_has_certificate_train",
        "certificate_count_train"
    ]

    fallback_df = popularity_df.merge(
        course_catalog[catalog_columns].drop_duplicates("course_encoded"),
        on="course_encoded",
        how="inner"
    )

    recommendations = []

    for _, row in fallback_df.head(top_n).iterrows():
        has_certificate = int(row["course_has_certificate_train"])
        certificate_status = "Bersertifikat" if has_certificate == 1 else "Tidak Bersertifikat"

        recommendations.append({
            "rank": len(recommendations) + 1,
            "course_id": row["canonical_course_id"],
            "recommended_course": row["course_name"],
            "competency": row["kompetensi"],
            "level": row["level"],
            "certificate_status": certificate_status,
            "certificate_count_train": int(row["certificate_count_train"]),
            "similarity_score_0_1": np.nan,
            "similarity_score_percent": np.nan,
            "softmax_score": np.nan,
            "softmax_score_percent": np.nan,
            "train_popularity_count": int(row["train_popularity_count"])
        })

    if len(recommendations) == 0:
        return "Fallback rekomendasi tidak tersedia karena course catalog kosong atau semua kandidat sudah pernah diambil."

    return pd.DataFrame(recommendations)


def prepare_user_input_production_ready(user_id, source_df=pretest_history):
    """
    Menyiapkan input inference dari data history mentah maupun encoded.

    Fungsi ini menangani:
    - user baru/tidak ditemukan,
    - course history yang tidak masuk encoder training,
    - sebagian course history yang dikenali dan sebagian unknown,
    - fitur kompetensi/level/certificate yang perlu di-encode saat inference.
    """
    user_id = normalize_user_id_value(user_id)

    audit_info = {
        "user_id": user_id,
        "total_history_count": 0,
        "known_history_count": 0,
        "unknown_history_count": 0,
        "status": "not_found"
    }

    if source_df is None or "user_id" not in source_df.columns:
        audit_info["status"] = "invalid_source_df"
        return None, set(), audit_info

    source_df = source_df.copy()
    user_mask = source_df["user_id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True) == user_id
    user_df = source_df[user_mask].copy()

    if user_df.empty:
        return None, set(), audit_info

    user_df = sort_history_for_inference(user_df)
    audit_info["total_history_count"] = int(len(user_df))
    audit_info["status"] = "found"

    # Encode course jika source_df masih berupa data mentah.
    if "course_encoded" not in user_df.columns:
        if "canonical_course_id" not in user_df.columns:
            audit_info["status"] = "missing_course_identifier"
            return None, set(), audit_info

        user_df["canonical_course_id"] = user_df["canonical_course_id"].astype(str)
        user_df["course_encoded"] = user_df["canonical_course_id"].map(course_to_idx)
    else:
        user_df["course_encoded"] = pd.to_numeric(user_df["course_encoded"], errors="coerce")

    # Lengkapi fitur course certificate jika belum tersedia.
    if "certificate_count_train" not in user_df.columns or "course_has_certificate_train" not in user_df.columns:
        cert_lookup = course_catalog[[
            "course_encoded",
            "certificate_count_train",
            "course_has_certificate_train"
        ]].drop_duplicates("course_encoded")

        user_df = user_df.drop(
            columns=["certificate_count_train", "course_has_certificate_train"],
            errors="ignore"
        ).merge(
            cert_lookup,
            on="course_encoded",
            how="left"
        )

    user_df["certificate_count_train"] = pd.to_numeric(
        user_df.get("certificate_count_train", 0),
        errors="coerce"
    ).fillna(0)

    user_df["course_has_certificate_train"] = pd.to_numeric(
        user_df.get("course_has_certificate_train", 0),
        errors="coerce"
    ).fillna(0).astype(int)

    # Encode kompetensi jika belum encoded.
    if "competency_encoded" not in user_df.columns:
        if "kompetensi" in user_df.columns:
            user_df["kompetensi"] = user_df["kompetensi"].astype(str)
            user_df["competency_clean"] = user_df["kompetensi"].where(
                user_df["kompetensi"].isin(competency_to_idx.keys()),
                UNKNOWN_TOKEN
            )
        else:
            user_df["competency_clean"] = UNKNOWN_TOKEN

        user_df["competency_encoded"] = user_df["competency_clean"].map(competency_to_idx)
    else:
        user_df["competency_encoded"] = pd.to_numeric(user_df["competency_encoded"], errors="coerce")

    # Encode level jika belum encoded.
    if "level_encoded" not in user_df.columns:
        if "level" in user_df.columns:
            user_df["level"] = user_df["level"].astype(str)
            user_df["level_clean"] = user_df["level"].where(
                user_df["level"].isin(level_to_idx.keys()),
                UNKNOWN_TOKEN
            )
        else:
            user_df["level_clean"] = UNKNOWN_TOKEN

        user_df["level_encoded"] = user_df["level_clean"].map(level_to_idx)
    else:
        user_df["level_encoded"] = pd.to_numeric(user_df["level_encoded"], errors="coerce")

    # Scaling certificate_count_train memakai scaler dari train_core saja.
    if "certificate_count_scaled" not in user_df.columns:
        user_df[["certificate_count_scaled"]] = scaler.transform(
            user_df[["certificate_count_train"]]
        )
    else:
        user_df["certificate_count_scaled"] = pd.to_numeric(
            user_df["certificate_count_scaled"],
            errors="coerce"
        ).fillna(0)

    known_history_df = user_df[
        user_df["course_encoded"].notna()
        & user_df["competency_encoded"].notna()
        & user_df["level_encoded"].notna()
        & user_df["course_encoded"].astype(float).between(1, num_courses - 1)
    ].copy()

    audit_info["known_history_count"] = int(len(known_history_df))
    audit_info["unknown_history_count"] = int(len(user_df) - len(known_history_df))

    if known_history_df.empty:
        audit_info["status"] = "all_history_unknown_to_encoder"
        return None, set(), audit_info

    known_history_df["course_encoded"] = known_history_df["course_encoded"].astype(int)
    known_history_df["competency_encoded"] = known_history_df["competency_encoded"].astype(int)
    known_history_df["level_encoded"] = known_history_df["level_encoded"].astype(int)

    course_seq = known_history_df["course_encoded"].tolist()
    competency_seq = known_history_df["competency_encoded"].tolist()
    level_seq = known_history_df["level_encoded"].tolist()
    course_cert_seq = known_history_df["course_has_certificate_train"].astype(float).tolist()
    cert_count_seq = known_history_df["certificate_count_scaled"].astype(float).tolist()

    input_course = pad_sequences([course_seq], maxlen=max_sequence_length, padding="pre")
    input_competency = pad_sequences([competency_seq], maxlen=max_sequence_length, padding="pre")
    input_level = pad_sequences([level_seq], maxlen=max_sequence_length, padding="pre")

    input_course_cert = pad_sequences(
        [course_cert_seq],
        maxlen=max_sequence_length,
        padding="pre",
        dtype="float32"
    )

    input_cert_count = pad_sequences(
        [cert_count_seq],
        maxlen=max_sequence_length,
        padding="pre",
        dtype="float32"
    )

    input_course_cert = np.expand_dims(input_course_cert, axis=-1)
    input_cert_count = np.expand_dims(input_cert_count, axis=-1)

    model_input = [
        input_course,
        input_competency,
        input_level,
        input_course_cert,
        input_cert_count
    ]

    taken_courses = set(course_seq)
    audit_info["status"] = "ready_for_model"

    return model_input, taken_courses, audit_info


def recommend_courses_production_ready(user_id, top_n=10, source_df=pretest_history):
    """
    Versi production-ready dari fungsi rekomendasi.

    Fungsi ini menangani:
    1. new employee / user tidak ditemukan,
    2. employee dengan history terlalu sedikit,
    3. employee whose history courses are not in encoder,
    4. employee with no generated recommendations,
    5. course catalog mismatch.
    """
    catalog_check = validate_catalog_consistency()

    model_input, taken_courses, audit_info = prepare_user_input_production_ready(
        user_id=user_id,
        source_df=source_df
    )

    if catalog_check.get("critical", False):
        fallback_result = build_popularity_fallback_table(top_n=top_n, exclude_encoded=taken_courses)
        return attach_inference_metadata(
            fallback_result,
            audit_info,
            catalog_check,
            "Catalog/model mismatch detected; using popularity fallback"
        )

    if audit_info["status"] == "not_found":
        fallback_result = build_popularity_fallback_table(top_n=top_n)
        return attach_inference_metadata(
            fallback_result,
            audit_info,
            catalog_check,
            "New employee or user history not found; using popularity fallback"
        )

    if audit_info["status"] == "all_history_unknown_to_encoder":
        fallback_result = build_popularity_fallback_table(top_n=top_n)
        return attach_inference_metadata(
            fallback_result,
            audit_info,
            catalog_check,
            "All user history courses are unknown to encoder; using popularity fallback"
        )

    if audit_info["known_history_count"] < MIN_HISTORY_FOR_MODEL_INFERENCE:
        fallback_result = build_popularity_fallback_table(top_n=top_n, exclude_encoded=taken_courses)
        return attach_inference_metadata(
            fallback_result,
            audit_info,
            catalog_check,
            "User history is too short for stable model inference; using popularity fallback"
        )

    try:
        softmax_score = model.predict(model_input, verbose=0)[0].copy()
        user_vector = decoder_vector_model.predict(model_input, verbose=0)
        similarity_score = calculate_similarity(user_vector, course_vectors)[0].copy()
    except Exception as e:
        fallback_result = build_popularity_fallback_table(top_n=top_n, exclude_encoded=taken_courses)
        return attach_inference_metadata(
            fallback_result,
            audit_info,
            catalog_check,
            f"Model inference failed ({e}); using popularity fallback"
        )

    # Masking course yang sudah pernah diambil agar tidak direkomendasikan ulang.
    for taken in taken_courses:
        if taken < len(similarity_score):
            similarity_score[taken] = -np.inf
        if taken < len(softmax_score):
            softmax_score[taken] = -np.inf

    # Masking indeks 0 karena indeks tersebut adalah PAD, bukan course valid.
    similarity_score[0] = -np.inf
    softmax_score[0] = -np.inf

    ranking = np.argsort(similarity_score)[::-1]
    recommendations = []

    for idx in ranking:
        if idx == 0 or not np.isfinite(similarity_score[idx]):
            continue

        info = course_catalog[course_catalog["course_encoded"] == idx]

        if info.empty:
            continue

        row = info.iloc[0]

        has_certificate = int(row["course_has_certificate_train"])
        certificate_status = "Bersertifikat" if has_certificate == 1 else "Tidak Bersertifikat"

        similarity_value = float(similarity_score[idx])
        softmax_value = float(softmax_score[idx])

        recommendations.append({
            "rank": len(recommendations) + 1,
            "course_id": row["canonical_course_id"],
            "recommended_course": row["course_name"],
            "competency": row["kompetensi"],
            "level": row["level"],
            "certificate_status": certificate_status,
            "certificate_count_train": int(row["certificate_count_train"]),
            "similarity_score_0_1": round(similarity_value, 4),
            "similarity_score_percent": round(similarity_value * 100, 2),
            "softmax_score": round(softmax_value, 6),
            "softmax_score_percent": round(softmax_value * 100, 4),
            "train_popularity_count": np.nan
        })

        if len(recommendations) >= top_n:
            break

    result = pd.DataFrame(recommendations)

    if result.empty:
        fallback_result = build_popularity_fallback_table(top_n=top_n, exclude_encoded=taken_courses)
        return attach_inference_metadata(
            fallback_result,
            audit_info,
            catalog_check,
            "Model generated no valid recommendations; using popularity fallback"
        )

    if audit_info["unknown_history_count"] > 0:
        reason = "Model recommendation; unknown history courses ignored during inference"
    else:
        reason = "Model recommendation"

    return attach_inference_metadata(
        result,
        audit_info,
        catalog_check,
        reason
    )


catalog_consistency_check = validate_catalog_consistency()

catalog_consistency_display = pd.DataFrame([{
    "status": catalog_consistency_check["status"],
    "message": catalog_consistency_check["message"],
    "issue_count": len(catalog_consistency_check["issues"]),
    "critical": catalog_consistency_check["critical"]
}])

print("=== Catalog Consistency Check ===")
display(catalog_consistency_display)

if len(catalog_consistency_check["issues"]) > 0:
    print("Detail issues:")
    for issue in catalog_consistency_check["issues"]:
        print("-", issue)
else:
    print("Tidak ada catalog mismatch yang terdeteksi.")


### 43.1 Catatan Implementasi Sistem
Bagian ini menjelaskan bagaimana sistem menangani kondisi input yang tidak ideal saat deployment. Dalam kondisi normal, rekomendasi tetap dihasilkan oleh model LSTM dengan attention. Jika input tidak memenuhi syarat inference, sistem menggunakan fallback berbasis popularitas course dari data training.

| Skenario deployment | Perlakuan sistem |
|---|---|
| New employee / user belum punya history | Menggunakan popularity fallback dari `train_core_enc` |
| History terlalu sedikit | Menggunakan popularity fallback dan mengecualikan course yang sudah diketahui pernah diambil |
| Semua history course tidak ada di encoder | Menggunakan popularity fallback |
| Sebagian history course tidak ada di encoder | Mengabaikan course unknown, memakai history yang valid, dan mencatat jumlah unknown |
| Model tidak menghasilkan rekomendasi valid | Menggunakan popularity fallback |
| Catalog/model mismatch | Menjalankan catalog consistency check dan memakai fallback jika mismatch kritis |

Catatan untuk laporan/paper: mekanisme fallback ini tidak digunakan untuk mengganti evaluasi utama Hit@K, MRR@10, dan NDCG@10. Fallback ini ditambahkan untuk mendukung kesiapan implementasi sistem rekomendasi pada skenario produksi.

In [ ]:
production_readiness_summary_df = pd.DataFrame([
    {
        "scenario": "New employee / cold-start user",
        "risk": "Model tidak memiliki riwayat course untuk membentuk sequence input.",
        "handling": "Popularity fallback dari train_core_enc."
    },
    {
        "scenario": "Employee with too little history",
        "risk": "Sequence terlalu pendek sehingga prediksi LSTM kurang stabil.",
        "handling": "Popularity fallback, dengan masking course yang sudah diketahui pernah diambil."
    },
    {
        "scenario": "History courses are not in encoder",
        "risk": "Course baru/asing tidak dapat diproses oleh embedding model.",
        "handling": "Course unknown diabaikan; jika tidak ada history valid, sistem memakai popularity fallback."
    },
    {
        "scenario": "No generated recommendations",
        "risk": "Semua kandidat termasking atau tidak ditemukan pada course_catalog.",
        "handling": "Popularity fallback dari kandidat course valid."
    },
    {
        "scenario": "Course catalog mismatch",
        "risk": "Dimensi model, encoder, course_vectors, dan course_catalog tidak sinkron.",
        "handling": "Catalog consistency check; jika mismatch kritis, sistem memakai fallback."
    }
])

display(production_readiness_summary_df)


## 44. Inference Test Cases untuk Skenario Deployment
**Penjelasan:** Bagian ini menguji fungsi rekomendasi Deployment pada beberapa skenario input yang umum muncul saat sistem digunakan di lingkungan nyata. Pengujian ini tidak menggantikan evaluasi utama model, tetapi memastikan pipeline inference tetap memberikan output yang aman untuk user normal, user baru, user dengan history terlalu pendek, history course yang tidak dikenali encoder, kondisi rekomendasi kosong, dan pengecekan konsistensi katalog.

Skenario uji ini penting untuk memastikan mekanisme fallback benar-benar aktif sesuai kebutuhan deployment.

In [ ]:
# ============================================================
# INFERENCE TEST CASES UNTUK SKENARIO DEPLOYMENT
# ============================================================

SYNTHETIC_USER_PREFIX = "__SYNTHETIC_INFERENCE_TEST__"


def summarize_inference_test_result(scenario_name, result):
    """Meringkas output fungsi rekomendasi production-ready untuk setiap skenario uji."""
    if isinstance(result, pd.DataFrame):
        fallback_reason = (
            result["fallback_reason"].dropna().astype(str).iloc[0]
            if "fallback_reason" in result.columns and result["fallback_reason"].notna().any()
            else "Not available"
        )

        catalog_status = (
            result["catalog_check_status"].dropna().astype(str).iloc[0]
            if "catalog_check_status" in result.columns and result["catalog_check_status"].notna().any()
            else "Not available"
        )

        known_history_count = (
            int(result["known_history_count"].dropna().iloc[0])
            if "known_history_count" in result.columns and result["known_history_count"].notna().any()
            else np.nan
        )

        unknown_history_count = (
            int(result["unknown_history_count"].dropna().iloc[0])
            if "unknown_history_count" in result.columns and result["unknown_history_count"].notna().any()
            else np.nan
        )

        total_history_count = (
            int(result["total_history_count"].dropna().iloc[0])
            if "total_history_count" in result.columns and result["total_history_count"].notna().any()
            else np.nan
        )

        return {
            "test_scenario": scenario_name,
            "output_type": "table",
            "generated_rows": int(len(result)),
            "fallback_reason": fallback_reason,
            "known_history_count": known_history_count,
            "unknown_history_count": unknown_history_count,
            "total_history_count": total_history_count,
            "catalog_check_status": catalog_status,
            "test_status": "PASS"
        }

    return {
        "test_scenario": scenario_name,
        "output_type": "message",
        "generated_rows": 0,
        "fallback_reason": str(result),
        "known_history_count": np.nan,
        "unknown_history_count": np.nan,
        "total_history_count": np.nan,
        "catalog_check_status": catalog_consistency_check.get("status", "UNKNOWN") if "catalog_consistency_check" in globals() else "UNKNOWN",
        "test_status": "PASS" if isinstance(result, str) and len(result) > 0 else "CHECK"
    }


def run_inference_test_case(scenario_name, user_id, source_df, top_n=10):
    """Menjalankan satu skenario uji inference dengan penanganan error agar notebook tetap berjalan."""
    try:
        result = recommend_courses_production_ready(
            user_id=user_id,
            top_n=top_n,
            source_df=source_df
        )
        summary = summarize_inference_test_result(scenario_name, result)
        return summary, result
    except Exception as e:
        summary = {
            "test_scenario": scenario_name,
            "output_type": "error",
            "generated_rows": 0,
            "fallback_reason": str(e),
            "known_history_count": np.nan,
            "unknown_history_count": np.nan,
            "total_history_count": np.nan,
            "catalog_check_status": "ERROR",
            "test_status": "CHECK"
        }
        return summary, str(e)


def find_existing_user_for_inference_test(source_df=pretest_history, max_candidates=100):
    """Mencari user existing yang memiliki history valid untuk model inference."""
    if source_df is None or source_df.empty or "user_id" not in source_df.columns:
        return None

    candidate_counts = (
        source_df["user_id"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .value_counts()
    )

    candidate_user_ids = candidate_counts[
        candidate_counts >= MIN_HISTORY_FOR_MODEL_INFERENCE
    ].index.tolist()

    for candidate_user_id in candidate_user_ids[:max_candidates]:
        _, _, audit_info = prepare_user_input_production_ready(
            user_id=candidate_user_id,
            source_df=source_df
        )

        if audit_info.get("known_history_count", 0) >= MIN_HISTORY_FOR_MODEL_INFERENCE:
            return candidate_user_id

    return candidate_user_ids[0] if len(candidate_user_ids) > 0 else None


inference_test_summaries = []
inference_test_outputs = {}

# 1. Existing employee dengan history cukup untuk model inference.
existing_test_user_id = find_existing_user_for_inference_test(pretest_history)

if existing_test_user_id is not None:
    summary, output = run_inference_test_case(
        scenario_name="Existing employee with sufficient history",
        user_id=existing_test_user_id,
        source_df=pretest_history,
        top_n=10
    )
    inference_test_summaries.append(summary)
    inference_test_outputs["Existing employee with sufficient history"] = output
else:
    inference_test_summaries.append({
        "test_scenario": "Existing employee with sufficient history",
        "output_type": "skipped",
        "generated_rows": 0,
        "fallback_reason": "Tidak ada user dengan history valid yang cukup pada pretest_history.",
        "known_history_count": np.nan,
        "unknown_history_count": np.nan,
        "total_history_count": np.nan,
        "catalog_check_status": catalog_consistency_check.get("status", "UNKNOWN") if "catalog_consistency_check" in globals() else "UNKNOWN",
        "test_status": "CHECK"
    })

# 2. New employee / cold-start user.
summary, output = run_inference_test_case(
    scenario_name="New employee / cold-start user",
    user_id=f"{SYNTHETIC_USER_PREFIX}_NEW_EMPLOYEE",
    source_df=pretest_history,
    top_n=10
)
inference_test_summaries.append(summary)
inference_test_outputs["New employee / cold-start user"] = output

# 3. Employee dengan history terlalu sedikit.
if pretest_history is not None and len(pretest_history) > 0:
    short_history_df = pretest_history.head(1).copy()
    short_history_df["user_id"] = f"{SYNTHETIC_USER_PREFIX}_SHORT_HISTORY"

    summary, output = run_inference_test_case(
        scenario_name="Employee with too little history",
        user_id=f"{SYNTHETIC_USER_PREFIX}_SHORT_HISTORY",
        source_df=short_history_df,
        top_n=10
    )
    inference_test_summaries.append(summary)
    inference_test_outputs["Employee with too little history"] = output

# 4. Employee whose history courses are not in encoder.
if pretest_history is not None and len(pretest_history) > 0:
    unknown_history_df = pretest_history.head(1).copy()
    unknown_history_df["user_id"] = f"{SYNTHETIC_USER_PREFIX}_UNKNOWN_HISTORY"

    if "canonical_course_id" in unknown_history_df.columns:
        unknown_history_df["canonical_course_id"] = "__UNKNOWN_COURSE_FOR_ENCODER_TEST__"

    if "course_encoded" in unknown_history_df.columns:
        unknown_history_df["course_encoded"] = np.nan

    summary, output = run_inference_test_case(
        scenario_name="History courses are not in encoder",
        user_id=f"{SYNTHETIC_USER_PREFIX}_UNKNOWN_HISTORY",
        source_df=unknown_history_df,
        top_n=10
    )
    inference_test_summaries.append(summary)
    inference_test_outputs["History courses are not in encoder"] = output

# 5. Employee with no remaining candidate recommendations.
if "course_catalog" in globals() and not course_catalog.empty:
    all_candidate_history_df = course_catalog.copy()
    all_candidate_history_df["user_id"] = f"{SYNTHETIC_USER_PREFIX}_ALL_COURSES_TAKEN"
    all_candidate_history_df["sort_row_order"] = np.arange(len(all_candidate_history_df))

    summary, output = run_inference_test_case(
        scenario_name="Employee with no remaining candidate recommendations",
        user_id=f"{SYNTHETIC_USER_PREFIX}_ALL_COURSES_TAKEN",
        source_df=all_candidate_history_df,
        top_n=10
    )
    inference_test_summaries.append(summary)
    inference_test_outputs["Employee with no remaining candidate recommendations"] = output

# 6. Course catalog consistency check.
catalog_check_for_test = validate_catalog_consistency()
inference_test_summaries.append({
    "test_scenario": "Course catalog consistency check",
    "output_type": "diagnostic",
    "generated_rows": np.nan,
    "fallback_reason": catalog_check_for_test.get("message", ""),
    "known_history_count": np.nan,
    "unknown_history_count": np.nan,
    "total_history_count": np.nan,
    "catalog_check_status": catalog_check_for_test.get("status", "UNKNOWN"),
    "test_status": "PASS" if catalog_check_for_test.get("status") in ["OK", "WARNING"] else "CHECK"
})

inference_test_summary_df = pd.DataFrame(inference_test_summaries)

display(inference_test_summary_df)

print("=== Contoh Output Tiap Skenario ===")
for scenario_name, scenario_output in inference_test_outputs.items():
    print(f"\n--- {scenario_name} ---")
    if isinstance(scenario_output, pd.DataFrame):
        display(scenario_output.head(3))
    else:
        print(scenario_output)


## 45. Interface Input User ID Manual dengan Output Rekomendasi Deployment
**Penjelasan:** Bagian ini menjadi interface sederhana di Colab. Pengguna memasukkan `User ID` atau `Employee ID` secara manual, lalu notebook menampilkan riwayat course dan Top-10 rekomendasi. Fungsi yang digunakan adalah `recommend_courses_production_ready()`, sehingga output tetap tersedia untuk user baru, history terlalu pendek, history yang tidak masuk encoder, rekomendasi kosong, atau catalog mismatch.

**Catatan penting:** output utama tetap berasal dari model LSTM dengan attention jika input valid. Fallback hanya aktif ketika kondisi input tidak memenuhi syarat inference produksi.

In [ ]:
# Non-blocking manual inference example.
# Set EXAMPLE_USER_ID_FOR_INFERENCE to a real ID, then run this cell.
EXAMPLE_USER_ID_FOR_INFERENCE = None

if EXAMPLE_USER_ID_FOR_INFERENCE is None:
    print("Set EXAMPLE_USER_ID_FOR_INFERENCE to run a notebook inference example.")
else:
    user_id_input = str(EXAMPLE_USER_ID_FOR_INFERENCE).strip()

    print("\n=== RIWAYAT USER SEBELUM TEST TARGET ===")
    display(show_user_history(user_id_input, source_df=pretest_history))

    print("\n=== TOP-10 REKOMENDASI COURSE - PRODUCTION READY ===")
    recommendation_result = recommend_courses_production_ready(
        user_id=user_id_input,
        top_n=10,
        source_df=pretest_history
    )

    display_recommendation_table(recommendation_result)


## 46. Panduan Interpretasi Metrik Recommender System
**Penjelasan:** Bagian ini menegaskan aturan pelaporan hasil agar interpretasi model tidak keliru. Karena sistem yang dibangun adalah **Top-N recommender system**, hasil eksperimen dilaporkan menggunakan metrik ranking seperti **Hit@K / Recall@K, MRR@10, NDCG@10, dan Coverage@10**.

Nilai **Hit@10 89,89%** berarti target course test muncul di daftar **Top-10 rekomendasi** pada 89,89% sampel evaluasi default. Nilai tersebut tidak boleh ditafsirkan sebagai skor klasifikasi tunggal. Dengan demikian, penulisan hasil pada laporan/paper harus selalu menyebut nama metriknya secara eksplisit, misalnya **Hit@10**, **MRR@10**, atau **NDCG@10**.

In [ ]:
metric_reporting_guideline_df = pd.DataFrame([
    {
        "reported_metric": "Hit@10",
        "correct_interpretation": "Persentase sampel ketika target course muncul di daftar Top-10 rekomendasi.",
        "recommended_wording": "Model memperoleh Hit@10 sebesar 89,89% pada skenario evaluasi default."
    },
    {
        "reported_metric": "Hit@5",
        "correct_interpretation": "Persentase sampel ketika target course muncul di daftar Top-5 rekomendasi.",
        "recommended_wording": "Model memperoleh Hit@5 sebesar 80,16% pada skenario evaluasi default."
    },
    {
        "reported_metric": "MRR@10",
        "correct_interpretation": "Rata-rata reciprocal rank target course dalam daftar Top-10.",
        "recommended_wording": "Model memperoleh MRR@10 sebesar 66,82%, menunjukkan target cenderung muncul pada ranking atas."
    },
    {
        "reported_metric": "NDCG@10",
        "correct_interpretation": "Kualitas ranking Top-10 dengan penalti ketika target muncul di posisi lebih bawah.",
        "recommended_wording": "Model memperoleh NDCG@10 sebesar 72,31% pada skenario evaluasi default."
    },
    {
        "reported_metric": "Coverage@10",
        "correct_interpretation": "Proporsi course kandidat yang pernah muncul dalam hasil rekomendasi Top-10.",
        "recommended_wording": "Coverage@10 digunakan untuk mengevaluasi keragaman course yang direkomendasikan."
    }
])

display(metric_reporting_guideline_df)


## 47. Deployment Flowchart dan Alur Inference Sistem
**Penjelasan:** Bagian ini merangkum alur penggunaan sistem rekomendasi pada skenario deployment. Flowchart memperlihatkan proses dari input `Employee ID`, validasi riwayat course, pengecekan konsistensi katalog, inference menggunakan model LSTM dengan attention, masking course yang sudah pernah diambil, fallback jika input tidak memenuhi syarat, hingga keluaran Top-N recommendation.

**Tujuan:** Memberikan gambaran implementasi sistem secara lebih operasional. Bagian ini berguna untuk laporan, paper, maupun presentasi karena menjelaskan bahwa sistem tidak hanya berhenti pada evaluasi model, tetapi juga memiliki alur inference yang aman untuk kondisi nyata seperti karyawan baru, riwayat terlalu sedikit, course unknown, rekomendasi kosong, dan catalog mismatch.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


deployment_flow_df = pd.DataFrame([
    {
        "step": 1,
        "stage": "Input employee",
        "process": "Sistem menerima Employee ID atau User ID dari pengguna.",
        "normal_output": "Employee ID siap diproses.",
        "fallback_or_control": "Jika input kosong atau tidak valid, sistem meminta input ulang."
    },
    {
        "step": 2,
        "stage": "Load employee history",
        "process": "Sistem mengambil riwayat course employee dari data history yang tersedia.",
        "normal_output": "Riwayat course employee ditemukan.",
        "fallback_or_control": "Jika user baru atau history tidak ditemukan, sistem memakai popularity fallback."
    },
    {
        "step": 3,
        "stage": "History validation",
        "process": "Sistem mengecek apakah jumlah history cukup untuk inference model.",
        "normal_output": "History cukup untuk membentuk sequence input.",
        "fallback_or_control": "Jika history terlalu sedikit, sistem memakai popularity fallback dengan seen-course masking."
    },
    {
        "step": 4,
        "stage": "Encoder validation",
        "process": "Sistem mengecek apakah course dalam history dikenal oleh encoder hasil training.",
        "normal_output": "Course history valid dan dapat diubah ke encoded sequence.",
        "fallback_or_control": "Course unknown diabaikan; jika semua course unknown, sistem memakai popularity fallback."
    },
    {
        "step": 5,
        "stage": "Catalog consistency check",
        "process": "Sistem mengecek konsistensi model output, encoder, course vectors, dan course catalog.",
        "normal_output": "Artifact model dan catalog konsisten.",
        "fallback_or_control": "Jika mismatch kritis ditemukan, sistem memakai fallback yang aman."
    },
    {
        "step": 6,
        "stage": "Preprocess sequence",
        "process": "History yang valid diproses menjadi input sequence sesuai format training.",
        "normal_output": "Input tensor siap untuk model.",
        "fallback_or_control": "Jika sequence gagal dibuat, sistem memakai popularity fallback."
    },
    {
        "step": 7,
        "stage": "Model inference",
        "process": "Model Encoder–Decoder LSTM dengan attention menghasilkan skor kandidat course.",
        "normal_output": "Skor rekomendasi untuk setiap kandidat course.",
        "fallback_or_control": "Jika inference gagal, sistem memakai popularity fallback."
    },
    {
        "step": 8,
        "stage": "Seen-course masking",
        "process": "Course yang sudah pernah diambil employee dikeluarkan dari kandidat rekomendasi.",
        "normal_output": "Kandidat course baru yang belum pernah diambil.",
        "fallback_or_control": "Jika semua kandidat termasking, sistem memakai popularity fallback."
    },
    {
        "step": 9,
        "stage": "Recommendation availability check",
        "process": "Sistem mengecek apakah daftar rekomendasi final tersedia.",
        "normal_output": "Top-N recommendation valid.",
        "fallback_or_control": "Jika tidak ada rekomendasi valid, sistem memakai popularity fallback."
    },
    {
        "step": 10,
        "stage": "Output recommendation",
        "process": "Sistem menampilkan Top-N course recommendation beserta atribut course.",
        "normal_output": "Tabel rekomendasi course untuk employee.",
        "fallback_or_control": "Output menyertakan alasan fallback jika fallback digunakan."
    },
    {
        "step": 11,
        "stage": "Monitoring and retraining",
        "process": "Hasil rekomendasi dan perubahan catalog dipantau untuk menjaga kualitas sistem.",
        "normal_output": "Log performa dan kebutuhan retraining.",
        "fallback_or_control": "Retraining dilakukan jika data, catalog, atau performa berubah signifikan."
    }
])

display(deployment_flow_df)


# Visual flowchart sederhana untuk laporan/paper/presentasi.
flow_nodes = [
    "Input Employee ID",
    "Load Course History",
    "History Valid?",
    "Course Known by Encoder?",
    "Catalog Consistent?",
    "Preprocess Sequence",
    "LSTM + Attention Inference",
    "Mask Seen Courses",
    "Recommendation Available?",
    "Output Top-N Recommendation"
]

fallback_box_text = "Popularity Fallback\n- New employee\n- Too little history\n- Unknown courses\n- Empty recommendation\n- Critical catalog mismatch"

fig, ax = plt.subplots(figsize=(12, 15))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

node_width = 0.42
node_height = 0.055
x_center = 0.34
y_start = 0.93
y_gap = 0.085

node_positions = []

for idx, label in enumerate(flow_nodes):
    y = y_start - idx * y_gap
    x = x_center - node_width / 2
    box = FancyBboxPatch(
        (x, y - node_height / 2),
        node_width,
        node_height,
        boxstyle="round,pad=0.012",
        linewidth=1.2,
        fill=False
    )
    ax.add_patch(box)
    ax.text(x_center, y, label, ha="center", va="center", fontsize=10)
    node_positions.append((x_center, y))

for idx in range(len(node_positions) - 1):
    start_x, start_y = node_positions[idx]
    end_x, end_y = node_positions[idx + 1]
    arrow = FancyArrowPatch(
        (start_x, start_y - node_height / 2),
        (end_x, end_y + node_height / 2),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=1.0
    )
    ax.add_patch(arrow)

fallback_x = 0.78
fallback_y = 0.54
fallback_width = 0.35
fallback_height = 0.22
fallback_box = FancyBboxPatch(
    (fallback_x - fallback_width / 2, fallback_y - fallback_height / 2),
    fallback_width,
    fallback_height,
    boxstyle="round,pad=0.018",
    linewidth=1.2,
    fill=False
)
ax.add_patch(fallback_box)
ax.text(fallback_x, fallback_y, fallback_box_text, ha="center", va="center", fontsize=9)

# Panah kontrol dari decision/check nodes menuju fallback.
decision_indices = [2, 3, 4, 8]
for decision_idx in decision_indices:
    start_x, start_y = node_positions[decision_idx]
    arrow = FancyArrowPatch(
        (start_x + node_width / 2, start_y),
        (fallback_x - fallback_width / 2, fallback_y),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=1.0,
        connectionstyle="arc3,rad=0.15"
    )
    ax.add_patch(arrow)
    ax.text(
        (start_x + fallback_x) / 2 + 0.02,
        (start_y + fallback_y) / 2 + 0.02,
        "fallback if needed",
        fontsize=8,
        ha="center",
        va="center"
    )

# Panah fallback menuju output rekomendasi.
output_x, output_y = node_positions[-1]
arrow = FancyArrowPatch(
    (fallback_x, fallback_y - fallback_height / 2),
    (output_x + node_width / 2, output_y),
    arrowstyle="->",
    mutation_scale=12,
    linewidth=1.0,
    connectionstyle="arc3,rad=-0.2"
)
ax.add_patch(arrow)

ax.set_title("Deployment Flowchart Sistem Rekomendasi Course", fontsize=14)
plt.show()


deployment_flow_summary_df = pd.DataFrame([
    {
        "component": "Primary inference path",
        "description": "Employee history valid diproses oleh model Encoder–Decoder LSTM dengan attention untuk menghasilkan Top-N course recommendation."
    },
    {
        "component": "Fallback path",
        "description": "Popularity fallback digunakan ketika input tidak cukup aman untuk inference model atau rekomendasi model tidak tersedia."
    },
    {
        "component": "Safety control",
        "description": "Seen-course masking, encoder validation, dan catalog consistency check menjaga output tetap valid untuk deployment."
    },
    {
        "component": "Paper contribution",
        "description": "Flowchart menunjukkan bahwa sistem memiliki alur deployment yang jelas, bukan hanya eksperimen model di notebook."
    }
])

display(deployment_flow_summary_df)

## 48. Ringkasan Evaluasi Final
**Penjelasan:** Bagian ini menampilkan ringkasan akhir pipeline, meliputi jumlah training sequence, jumlah sample validation dan test, ringkasan distribusi target test, **Coverage@10**, metrik ranking softmax, metrik ranking similarity, **Top-K Performance Curve**, **Error Analysis**, **ablation study fitur**, hasil leakage audit, inference test cases deployment, deployment flowchart, Panduan Interpretasi Metrik Recommender System, perbandingan protokol evaluasi default vs strict, serta ringkasan *robustness* strict train-history evaluation.

**Yang perlu diperhatikan:** Ringkasan tidak lagi memakai istilah evaluasi klasifikasi umum. Untuk paper recommender system, hasil utama dilaporkan sebagai **Hit@K / Recall@K, MRR@10, NDCG@10, dan Coverage@10**. Default pretest-history evaluation dilaporkan sebagai skenario operasional, sedangkan strict train-history evaluation dilaporkan sebagai *robustness check* agar interpretasi hasil evaluasi lebih aman dan transparan.

In [ ]:
print("=== RINGKASAN EVALUASI FINAL LEAKAGE-SAFE ===")

print("Training sequences:", len(y_train))
print("Validation temporal samples:", len(y_val))
print("Test isolated samples:", len(y_test))

print()
print("=== Panduan Pelaporan Metrik Recommender System ===")
if "metric_reporting_guideline_df" in globals():
    display(metric_reporting_guideline_df)
else:
    print("Panduan pelaporan metrik belum dibuat. Jalankan section Panduan Pelaporan Metrik Recommender System terlebih dahulu.")

print()
print("=== Course Noise Cleaning ===")
if "course_noise_summary_df" in globals():
    display(course_noise_summary_df)
    if "course_noise_removed_df" in globals():
        print("Course noise yang dihapus:", len(course_noise_removed_df))
else:
    print("Course noise cleaning belum dibuat. Jalankan section Course Noise Cleaning terlebih dahulu.")

if "course_manual_review_df" in globals():
    print()
    print("=== Manual Review Course Ambigu ===")
    if len(course_manual_review_df) > 0:
        display(course_manual_review_df)
    else:
        print("Tidak ada course ambigu yang cocok dengan pola manual review saat ini.")

if "course_senin_decision_df" in globals():
    print()
    print("=== Keputusan Course senin ===")
    display(course_senin_decision_df)

print()
print("=== Distribusi Target Test ===")
if "target_distribution_summary_df" in globals():
    display(target_distribution_summary_df)
else:
    print("Target distribution summary belum dibuat. Jalankan section distribusi target test terlebih dahulu.")

print()
print("=== Coverage@10 Summary ===")
if "coverage_summary_df" in globals():
    coverage_summary_display = coverage_summary_df.copy()
    percentage_columns = [
        "coverage_at_k_percent",
        "top_1_recommendation_share_percent",
        "top_5_recommendation_share_percent",
        "top_10_recommendation_share_percent"
    ]
    existing_percentage_columns = [col for col in percentage_columns if col in coverage_summary_display.columns]
    coverage_summary_display[existing_percentage_columns] = coverage_summary_display[existing_percentage_columns].round(2)
    if "coverage_at_k" in coverage_summary_display.columns:
        coverage_summary_display["coverage_at_k"] = coverage_summary_display["coverage_at_k"].round(4)
    display(coverage_summary_display)
else:
    print("Coverage@10 belum dibuat. Jalankan section Coverage@10 terlebih dahulu.")

print()
print("=== Softmax Ranking Metrics ===")
display(
    softmax_metrics_df.assign(
        score=softmax_metrics_df["score"].round(4),
        percentage=softmax_metrics_df["percentage"].round(2)
    )
)

print()
print("=== Similarity Ranking Metrics ===")
display(
    similarity_metrics_df.assign(
        score=similarity_metrics_df["score"].round(4),
        percentage=similarity_metrics_df["percentage"].round(2)
    )
)

print()
print("=== Top-K Performance Curve ===")
if "topk_curve_display" in globals():
    display(topk_curve_display)
else:
    print("Top-K Performance Curve belum dibuat. Jalankan section Top-K Performance Curve terlebih dahulu.")

print()
print("=== Error Analysis Summary ===")
if "error_analysis_summary_display" in globals():
    display(error_analysis_summary_display)
else:
    print("Error Analysis belum dibuat. Jalankan section Error Analysis Rekomendasi Top-10 terlebih dahulu.")

print()
print("=== Error Type Distribution ===")
if "error_type_summary_display" in globals():
    display(error_type_summary_display)
else:
    print("Distribusi error type belum dibuat. Jalankan section Error Analysis Rekomendasi Top-10 terlebih dahulu.")

print()
print("=== Softmax vs Similarity Case Comparison ===")
if "ranking_case_comparison_summary_display" in globals():
    display(ranking_case_comparison_summary_display)
else:
    print("Perbandingan kasus Softmax vs Similarity belum dibuat. Jalankan section Error Analysis Rekomendasi Top-10 terlebih dahulu.")

print()
print("=== Ablation Study Feature Summary ===")
if "ablation_summary_display" in globals():
    display(ablation_summary_display)
else:
    print("Ablation study belum dijalankan. Jalankan section Ablation Study Fitur terlebih dahulu.")

print()
print("=== Delta Ablation terhadap Baseline Course-Only ===")
if "ablation_delta_display" in globals():
    display(ablation_delta_display)
else:
    print("Delta ablation belum dibuat. Jalankan section Analisis Delta terhadap Baseline terlebih dahulu.")

print()
print("=== Sensitivitas MIN_COURSE_COUNT_TRAIN ===")
if "min_course_sensitivity_display" in globals():
    display(min_course_sensitivity_display)
else:
    print("Sensitivitas MIN_COURSE_COUNT_TRAIN belum dibuat. Jalankan section Sensitivitas MIN_COURSE_COUNT_TRAIN terlebih dahulu.")

if "threshold_justification_summary_df" in globals():
    print()
    print("=== Justifikasi MIN_COURSE_COUNT_TRAIN Utama ===")
    display(threshold_justification_summary_df)

print()
print("=== Strict Train-History Evaluation ===")
if "evaluation_protocol_summary_df" in globals():
    print("Ringkasan protokol evaluasi:")
    display(evaluation_protocol_summary_df)
else:
    print("Ringkasan protokol evaluasi belum dibuat. Jalankan section Strict Train-History Evaluation terlebih dahulu.")

if "strict_history_audit_display" in globals():
    print("Audit validation target di history evaluasi:")
    display(strict_history_audit_display)
else:
    print("Audit strict train-history belum dibuat. Jalankan section Strict Train-History Evaluation terlebih dahulu.")

if "strict_metrics_comparison_display" in globals():
    print("Perbandingan metrik default vs strict:")
    display(strict_metrics_comparison_display)
else:
    print("Perbandingan metrik strict evaluation belum tersedia atau strict dataset kosong.")

if "strict_robustness_summary_display" in globals():
    print("Ringkasan robustness berdasarkan Hit@10:")
    display(strict_robustness_summary_display)
else:
    print("Ringkasan robustness belum tersedia atau strict dataset kosong.")

print()
print("=== Visualisasi Distribusi Data dan Hasil ===")
if "visualization_summary_df" in globals():
    display(visualization_summary_df)
else:
    print("Visualisasi distribusi data/hasil belum dibuat. Jalankan section visualisasi distribusi data dan hasil terlebih dahulu.")

if "data_split_summary_df" in globals():
    display(data_split_summary_df)

if "similarity_recommendation_frequency_df" in globals():
    print("Top-10 course yang paling sering direkomendasikan oleh Similarity Ranking:")
    display(similarity_recommendation_frequency_df.head(10))

print()
print("=== Popularity Sanity Check ===")
if "popularity_metrics_comparison_display" in globals():
    display(popularity_metrics_comparison_display)
else:
    print("Popularity sanity check belum dibuat. Jalankan section popularity sanity check terlebih dahulu.")

if "popularity_overlap_summary_display" in globals():
    display(popularity_overlap_summary_display)

if "popularity_delta_display" in globals():
    display(popularity_delta_display)

print()
print("=== Repeated Run Stability Check ===")
if "repeated_run_display_df" in globals():
    display(repeated_run_display_df)
else:
    print("Repeated run stability check belum dijalankan. Jalankan section Repeated Run Stability Check terlebih dahulu.")

if "repeated_run_aggregate_display_df" in globals():
    display(repeated_run_aggregate_display_df)

print()
print("=== Hyperparameter Comparison ===")
if "hyperparameter_comparison_display_df" in globals():
    display(hyperparameter_comparison_display_df)
else:
    print("Hyperparameter comparison belum dijalankan. Jalankan section Hyperparameter Comparison terlebih dahulu.")

if "hyperparameter_best_config_display_df" in globals():
    print("Konfigurasi terbaik berdasarkan validation split:")
    display(hyperparameter_best_config_display_df)

print()
print("=== Production-Ready Fallback Handling ===")
if "production_readiness_summary_df" in globals():
    display(production_readiness_summary_df)
else:
    print("Production-ready fallback summary belum dibuat. Jalankan section Production-Ready Fallback Handling terlebih dahulu.")

if "catalog_consistency_check" in globals():
    catalog_consistency_display_final = pd.DataFrame([{
        "status": catalog_consistency_check["status"],
        "message": catalog_consistency_check["message"],
        "issue_count": len(catalog_consistency_check["issues"]),
        "critical": catalog_consistency_check["critical"]
    }])
    display(catalog_consistency_display_final)
else:
    print("Catalog consistency check belum tersedia.")

print()
print("=== Inference Test Cases Deployment ===")
if "inference_test_summary_df" in globals():
    display(inference_test_summary_df)
else:
    print("Inference test cases belum dijalankan. Jalankan section Inference Test Cases untuk Skenario Deployment terlebih dahulu.")

print()
print("=== Deployment Flowchart dan Alur Inference Sistem ===")
if "deployment_flow_df" in globals():
    display(deployment_flow_df)
else:
    print("Deployment flowchart belum dibuat. Jalankan section Deployment Flowchart dan Alur Inference Sistem terlebih dahulu.")

if "deployment_flow_summary_df" in globals():
    display(deployment_flow_summary_df)

print()
print("=== Reproducibility, Save Artifact, dan Persistensi Output ===")
if "artifact_manifest_df" in globals():
    display(artifact_manifest_df)
else:
    print("Artifact belum disimpan. Jalankan section reproducibility dan save artifact terlebih dahulu.")

if "artifact_zip_path" in globals() and artifact_zip_path is not None:
    print("Artifact ZIP:", artifact_zip_path)

if "google_drive_copy_status" in globals():
    print("Status Google Drive copy:", google_drive_copy_status)

print()
print("=== Attention Layer Check ===")
if "attention_weight_model" in globals():
    print("Attention layer aktif: temporal_attention tersedia untuk interpretasi bobot riwayat course.")
    if "attention_example_df" in globals():
        display(attention_example_df)
else:
    print("Attention weight model belum tersedia. Jalankan section Model Encoder–Decoder LSTM dengan Attention Layer terlebih dahulu.")

print()
print("Leakage audit:")
print("Validation target bocor ke train history:", len(val_leak_df))
print("Test target bocor ke pretest history:", len(test_leak_df))
print("Overlap user-course train-test:", len(overlap_train_test))
print("Overlap user-course train-validation:", len(overlap_train_val))

if (
    len(val_leak_df) == 0
    and len(test_leak_df) == 0
    and len(overlap_train_test) == 0
    and len(overlap_train_val) == 0
):
    print()
    print("KESIMPULAN: PASS. Pipeline sudah temporal, train-only transformed, leakage-safe, dilengkapi course dummy/test/noise cleaning, keputusan eksplisit untuk course ambigu, attention layer, analisis distribusi target test, Top-K Performance Curve, Error Analysis, Coverage@10, ablation study fitur, sensitivitas dan justifikasi MIN_COURSE_COUNT_TRAIN, strict train-history evaluation, visualisasi distribusi data/hasil, popularity sanity check, repeated run stability check, hyperparameter comparison, production-ready fallback handling, inference test cases deployment, deployment flowchart, catalog consistency check, serta reproducibility, save artifact, ZIP artifact, dan opsi Google Drive untuk skenario next-course recommendation.")
else:
    print()
    print("KESIMPULAN: FAIL. Masih ada leakage yang perlu dicek.")
